In [ ]:
# ============================================================
# MAMBA EXPERIMENT — KAGGLE ENVIRONMENT SETUP
# ============================================================

import os
import sys
import subprocess
import torch

print("=" * 70)
print("MAMBA EXPERIMENT — ENVIRONMENT SETUP")
print("=" * 70)

# ============================================================
# KAGGLE DATASET ROOTS
# ============================================================

TRAINING_PREP_ROOT = (
    "/kaggle/input/datasets/oipranesh/cs781-phase4-training-prep"
)

FINAL_RECOVERY_ROOT = (
    "/kaggle/input/datasets/oipranesh/cs781-phase4-final-recovery"
)

print("Training-prep dataset:")
print(TRAINING_PREP_ROOT)

print()
print("Final-recovery dataset:")
print(FINAL_RECOVERY_ROOT)

# ============================================================
# VERIFY DATASETS ARE ATTACHED
# ============================================================

if not os.path.isdir(TRAINING_PREP_ROOT):
    raise FileNotFoundError(
        f"""
Training-prep dataset was not found:

{TRAINING_PREP_ROOT}

Make sure the Kaggle dataset
'oipranesh/cs781-phase4-training-prep'
is added to this notebook.
"""
    )

if not os.path.isdir(FINAL_RECOVERY_ROOT):
    raise FileNotFoundError(
        f"""
Final-recovery dataset was not found:

{FINAL_RECOVERY_ROOT}

Make sure the Kaggle dataset
'oipranesh/cs781-phase4-final-recovery'
is added to this notebook.
"""
    )

print("✓ Training-prep dataset mounted")
print("✓ Final-recovery dataset mounted")

# ============================================================
# PYTORCH / GPU
# ============================================================

print()
print("Python :", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA   :", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU is required for the Mamba experiment."
    )

DEVICE = torch.device("cuda:0")

print("GPU:", torch.cuda.get_device_name(DEVICE))

gpu_memory = (
    torch.cuda.get_device_properties(DEVICE).total_memory
    / 1024**3
)

print(
    "GPU memory:",
    f"{gpu_memory:.2f} GB"
)

print()
print("=" * 70)
print("✓ BASE ENVIRONMENT VERIFIED")
print("=" * 70)

In [ ]:
# ============================================================
# MAMBA — DATASET ARTIFACT DISCOVERY
# ============================================================

import os

def show_tree(root, max_depth=3):

    root = os.path.abspath(root)

    print()
    print("=" * 70)
    print(root)
    print("=" * 70)

    root_depth = root.rstrip(os.sep).count(os.sep)

    for current_root, dirs, files in os.walk(root):

        depth = (
            current_root.rstrip(os.sep).count(os.sep)
            - root_depth
        )

        if depth > max_depth:
            dirs[:] = []
            continue

        indent = "  " * depth

        print(f"{indent}{os.path.basename(current_root)}/")

        for filename in sorted(files):
            path = os.path.join(current_root, filename)
            size_gb = os.path.getsize(path) / 1024**3

            print(
                f"{indent}  {filename}"
                f"  ({size_gb:.3f} GB)"
            )


show_tree(
    TRAINING_PREP_ROOT,
    max_depth=3,
)

show_tree(
    FINAL_RECOVERY_ROOT,
    max_depth=3,
)

In [ ]:
# ============================================================
# MAMBA — RESOLVE CANONICAL ARTIFACT PATHS
# ============================================================

def find_file(root, filename):

    matches = []

    for current_root, dirs, files in os.walk(root):

        if filename in files:
            matches.append(
                os.path.join(
                    current_root,
                    filename
                )
            )

    return matches


# ------------------------------------------------------------
# TRAINING MANIFEST
# ------------------------------------------------------------

train_manifest_matches = find_file(
    TRAINING_PREP_ROOT,
    "train_manifest.jsonl",
)

if len(train_manifest_matches) != 1:

    raise RuntimeError(
        "Expected exactly one train_manifest.jsonl, "
        f"found {len(train_manifest_matches)}:\n"
        + "\n".join(train_manifest_matches)
    )

TRAIN_MANIFEST = train_manifest_matches[0]


# ------------------------------------------------------------
# TOKENIZER
# ------------------------------------------------------------

tokenizer_matches = find_file(
    TRAINING_PREP_ROOT,
    "legal_bpe_32k.json",
)

if len(tokenizer_matches) != 1:

    raise RuntimeError(
        "Expected exactly one legal_bpe_32k.json, "
        f"found {len(tokenizer_matches)}:\n"
        + "\n".join(tokenizer_matches)
    )

TOKENIZER_PATH = tokenizer_matches[0]


# ------------------------------------------------------------
# SOURCE INDEX
# ------------------------------------------------------------

index_matches = find_file(
    FINAL_RECOVERY_ROOT,
    "source_index.jsonl",
)

if len(index_matches) == 0:

    print(
        "⚠ source_index.jsonl was not found "
        "inside final-recovery."
    )

    INDEX_PATH = None

elif len(index_matches) == 1:

    INDEX_PATH = index_matches[0]

else:

    raise RuntimeError(
        "Multiple source_index.jsonl files found:\n"
        + "\n".join(index_matches)
    )


# ============================================================
# REPORT
# ============================================================

print("=" * 70)
print("CANONICAL MAMBA ARTIFACT PATHS")
print("=" * 70)

print()
print("TRAIN_MANIFEST")
print(TRAIN_MANIFEST)

print()
print("TOKENIZER_PATH")
print(TOKENIZER_PATH)

print()
print("INDEX_PATH")
print(INDEX_PATH)

print()
print("=" * 70)
print("✓ ARTIFACT RESOLUTION COMPLETE")
print("=" * 70)

In [ ]:
TRAIN_MANIFEST = (
    "/kaggle/input/datasets/oipranesh/"
    "cs781-phase4-training-prep/"
    "manifests/train_manifest.jsonl"
)

VAL_MANIFEST = (
    "/kaggle/input/datasets/oipranesh/"
    "cs781-phase4-training-prep/"
    "manifests/validation_manifest.jsonl"
)

TEST_MANIFEST = (
    "/kaggle/input/datasets/oipranesh/"
    "cs781-phase4-training-prep/"
    "manifests/test_manifest.jsonl"
)

TOKENIZER_PATH = (
    "/kaggle/input/datasets/oipranesh/"
    "cs781-phase4-training-prep/"
    "tokenizer/legal_bpe_32k.json"
)

INDEX_PATH = (
    "/kaggle/input/datasets/oipranesh/"
    "cs781-phase4-final-recovery/"
    "indexes/source_index.jsonl"
)

In [ ]:
# ============================================================
# MAMBA — DATASET CONSISTENCY AUDIT
# READ-ONLY
# ============================================================

import os
import json
import hashlib

print("=" * 70)
print("MAMBA — DATASET CONSISTENCY AUDIT")
print("READ-ONLY — NOTHING WILL BE MODIFIED")
print("=" * 70)


def sha256_file(path, chunk_size=8 * 1024 * 1024):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        while True:

            chunk = f.read(chunk_size)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


# ------------------------------------------------------------
# VERIFY FILES
# ------------------------------------------------------------

required_files = {
    "train_manifest": TRAIN_MANIFEST,
    "validation_manifest": VAL_MANIFEST,
    "test_manifest": TEST_MANIFEST,
    "tokenizer": TOKENIZER_PATH,
    "source_index": INDEX_PATH,
}

for name, path in required_files.items():

    print(f"{name:22s}: {path}")

    if not os.path.isfile(path):

        raise FileNotFoundError(
            f"\nMissing required artifact:\n{path}"
        )

    print(
        f"{'':22s}  ✓ exists "
        f"({os.path.getsize(path) / 1024**2:.2f} MB)"
    )


# ------------------------------------------------------------
# MANIFEST RECORD COUNTS
# ------------------------------------------------------------

print()
print("=" * 70)
print("MANIFEST COUNTS")
print("=" * 70)


def count_jsonl(path):

    count = 0

    with open(path, "r") as f:

        for line in f:

            if line.strip():
                count += 1

    return count


train_count = count_jsonl(TRAIN_MANIFEST)
val_count = count_jsonl(VAL_MANIFEST)
test_count = count_jsonl(TEST_MANIFEST)

print("Training examples   :", train_count)
print("Validation examples :", val_count)
print("Test examples       :", test_count)


# ------------------------------------------------------------
# FIRST RECORDS
# ------------------------------------------------------------

print()
print("=" * 70)
print("FIRST TRAIN MANIFEST RECORD")
print("=" * 70)

with open(TRAIN_MANIFEST, "r") as f:

    first_record = json.loads(
        f.readline()
    )

print(
    json.dumps(
        first_record,
        indent=2,
    )
)


# ------------------------------------------------------------
# CHECKSUMS
# ------------------------------------------------------------

print()
print("=" * 70)
print("ARTIFACT CHECKSUMS")
print("=" * 70)

for name, path in required_files.items():

    print(
        f"{name:22s}: "
        f"{sha256_file(path)}"
    )


print()
print("=" * 70)
print("✓ DATASET CONSISTENCY AUDIT COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================
# MAMBA — STREAMING CONFIGURATION AUDIT
# READ-ONLY
# ============================================================

import json

STREAMING_CONFIG_PATH = os.path.join(
    TRAINING_PREP_ROOT,
    "streaming_config.json",
)

TRAINING_EXAMPLE_CONFIG_PATH = os.path.join(
    TRAINING_PREP_ROOT,
    "training_example_config.json",
)

print("=" * 70)
print("MAMBA — STREAMING CONFIGURATION")
print("=" * 70)


for name, path in {
    "streaming_config": STREAMING_CONFIG_PATH,
    "training_example_config":
        TRAINING_EXAMPLE_CONFIG_PATH,
}.items():

    print()
    print(name)
    print(path)

    if not os.path.isfile(path):

        print("⚠ NOT FOUND")
        continue

    with open(path, "r") as f:

        config = json.load(f)

    print(
        json.dumps(
            config,
            indent=2,
        )
    )


print()
print("=" * 70)
print("✓ STREAMING CONFIGURATION AUDIT COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================
# MAMBA — RECOVERY CONFIGURATION AUDIT
# READ-ONLY
# ============================================================

RECOVERY_MANIFEST = os.path.join(
    FINAL_RECOVERY_ROOT,
    "RECOVERY_MANIFEST.json",
)

RECOVERY_CONFIG = os.path.join(
    FINAL_RECOVERY_ROOT,
    "training",
    "phase4_recovery_config.json",
)

TRAINING_STATE = os.path.join(
    FINAL_RECOVERY_ROOT,
    "training_state.json",
)

MODEL_ARCHITECTURE = os.path.join(
    FINAL_RECOVERY_ROOT,
    "model_architecture.json",
)

print("=" * 70)
print("MAMBA — EXISTING RECOVERY METADATA")
print("=" * 70)


for name, path in {
    "RECOVERY_MANIFEST": RECOVERY_MANIFEST,
    "RECOVERY_CONFIG": RECOVERY_CONFIG,
    "TRAINING_STATE": TRAINING_STATE,
    "MODEL_ARCHITECTURE": MODEL_ARCHITECTURE,
}.items():

    print()
    print("-" * 70)
    print(name)
    print(path)

    if not os.path.isfile(path):

        print("⚠ NOT FOUND")
        continue

    try:

        with open(path, "r") as f:

            data = json.load(f)

        print(
            json.dumps(
                data,
                indent=2,
            )
        )

    except Exception as e:

        print(
            "⚠ Could not parse as JSON:",
            repr(e)
        )


print()
print("=" * 70)
print("✓ RECOVERY METADATA AUDIT COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================
# MAMBA — TOKENIZER AUDIT
# READ-ONLY
# ============================================================

import json
import os

print("=" * 70)
print("MAMBA — TOKENIZER AUDIT")
print("=" * 70)

with open(TOKENIZER_PATH, "r") as f:

    tokenizer_json = json.load(f)

print("Tokenizer file:", TOKENIZER_PATH)

print()
print("Top-level keys:")
print(
    list(tokenizer_json.keys())
)


# ------------------------------------------------------------
# VOCABULARY
# ------------------------------------------------------------

vocab = (
    tokenizer_json
    .get("model", {})
    .get("vocab", {})
)

if not vocab:

    raise RuntimeError(
        "Could not locate tokenizer vocabulary."
    )

VOCAB_SIZE = len(vocab)

print()
print("Vocabulary size:", VOCAB_SIZE)


# ------------------------------------------------------------
# SPECIAL TOKENS
# ------------------------------------------------------------

added_tokens = tokenizer_json.get(
    "added_tokens",
    [],
)

print()
print("Added/special tokens:")

for token in added_tokens:

    print(token)


print()
print("=" * 70)
print("✓ TOKENIZER AUDIT COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================
# MAMBA EXPERIMENT — DEPENDENCY / ENVIRONMENT AUDIT
# READ-ONLY — NOTHING WILL BE MODIFIED
# ============================================================

import sys
import os
import importlib
import subprocess
import torch

print("=" * 70)
print("MAMBA — DEPENDENCY / ENVIRONMENT AUDIT")
print("READ-ONLY")
print("=" * 70)

# ============================================================
# PYTHON / PYTORCH / CUDA
# ============================================================

print()
print("=" * 70)
print("PYTHON / PYTORCH / CUDA")
print("=" * 70)

print("Python :", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA   :", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU    :", torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print(
        "GPU memory:",
        f"{props.total_memory / 1024**3:.2f} GB"
    )

assert torch.cuda.is_available(), \
    "CUDA is not available."

# ============================================================
# MAMBA PACKAGE DISCOVERY
# ============================================================

print()
print("=" * 70)
print("MAMBA PACKAGE DISCOVERY")
print("=" * 70)

candidate_packages = [
    "mamba_ssm",
    "causal_conv1d",
]

package_status = {}

for package_name in candidate_packages:

    try:
        module = importlib.import_module(package_name)

        package_status[package_name] = True

        print(
            f"✓ {package_name:20s} "
            f"importable"
        )

        version = getattr(
            module,
            "__version__",
            None
        )

        if version is not None:
            print(
                f"  version: {version}"
            )

        module_file = getattr(
            module,
            "__file__",
            None
        )

        if module_file:
            print(
                f"  path   : {module_file}"
            )

    except Exception as e:

        package_status[package_name] = False

        print(
            f"✗ {package_name:20s} "
            f"not importable"
        )

        print(
            f"  reason: {type(e).__name__}: {e}"
        )

# ============================================================
# PYTORCH 2.x FEATURES NEEDED BY MODERN MAMBA IMPLEMENTATIONS
# ============================================================

print()
print("=" * 70)
print("PYTORCH COMPATIBILITY")
print("=" * 70)

torch_version = tuple(
    int(x)
    for x in torch.__version__.split("+")[0].split(".")[:2]
)

print(
    "PyTorch major/minor:",
    torch_version
)

assert torch_version >= (2, 0), (
    "Mamba experiment requires PyTorch 2.x."
)

print("✓ PyTorch 2.x detected")

# ============================================================
# BASIC CUDA COMPUTATION
# ============================================================

print()
print("=" * 70)
print("CUDA COMPUTATION TEST")
print("=" * 70)

device = torch.device("cuda:0")

x = torch.randn(
    2,
    16,
    768,
    device=device,
    dtype=torch.float32,
)

y = torch.randn(
    2,
    16,
    768,
    device=device,
    dtype=torch.float32,
)

z = torch.matmul(
    x,
    y.transpose(-1, -2),
)

assert z.device == device
assert torch.isfinite(z).all()

print("✓ CUDA tensor allocation")
print("✓ Float32 computation")
print("✓ Matrix multiplication")
print("✓ Finite output")

del x
del y
del z

torch.cuda.empty_cache()

# ============================================================
# OPTIONAL MAMBA API INSPECTION
# ============================================================

print()
print("=" * 70)
print("MAMBA API INSPECTION")
print("=" * 70)

if package_status.get("mamba_ssm", False):

    import mamba_ssm

    print(
        "mamba_ssm location:",
        mamba_ssm.__file__
    )

    try:

        from mamba_ssm import Mamba

        print("✓ from mamba_ssm import Mamba")

        print()
        print("Mamba object:")
        print(Mamba)

    except Exception as e:

        print(
            "✗ Standard Mamba import failed:"
        )

        print(
            type(e).__name__,
            ":",
            e
        )

else:

    print(
        "Mamba package is not currently installed."
    )

# ============================================================
# CAUSAL-CONV API INSPECTION
# ============================================================

print()
print("=" * 70)
print("CAUSAL CONVOLUTION API")
print("=" * 70)

if package_status.get("causal_conv1d", False):

    try:

        import causal_conv1d

        print(
            "✓ causal_conv1d import successful"
        )

        print(
            "Location:",
            causal_conv1d.__file__
        )

    except Exception as e:

        print(
            "✗ causal_conv1d import failed:",
            e
        )

else:

    print(
        "causal_conv1d is not currently installed."
    )

# ============================================================
# MEMORY BASELINE
# ============================================================

print()
print("=" * 70)
print("GPU MEMORY BASELINE")
print("=" * 70)

allocated = (
    torch.cuda.memory_allocated(device)
    / 1024**3
)

reserved = (
    torch.cuda.memory_reserved(device)
    / 1024**3
)

print(
    f"Allocated : {allocated:.3f} GB"
)

print(
    f"Reserved  : {reserved:.3f} GB"
)

# ============================================================
# FINAL VERDICT
# ============================================================

print()
print("=" * 70)
print("ENVIRONMENT AUDIT VERDICT")
print("=" * 70)

if package_status.get("mamba_ssm", False):

    print(
        "✓ mamba_ssm is already available."
    )

    print(
        "✓ Environment is ready for Mamba API inspection."
    )

else:

    print(
        "⚠ mamba_ssm is NOT installed."
    )

    print(
        "No installation was attempted."
    )

    print(
        "Do NOT start Mamba training yet."
    )

print()
print(
    "✓ READ-ONLY AUDIT COMPLETE"
)

print(
    "✓ NO CHECKPOINTS MODIFIED"
)

print(
    "✓ NO DATASETS MODIFIED"
)

print(
    "✓ NO TRAINING STARTED"
)

print("=" * 70)

In [ ]:
# ============================================================
# MAMBA EXPERIMENT — DEPENDENCY INSTALLATION
# INSTALL ONLY — NO DATA / CHECKPOINT MODIFICATION
# ============================================================

import sys
import subprocess

print("=" * 70)
print("MAMBA — INSTALLATION")
print("=" * 70)

print("Python:", sys.version)

# Install causal-conv1d first.
# --no-build-isolation is intentional: keep the existing
# Kaggle CUDA-enabled PyTorch environment.
subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "--no-build-isolation",
    "causal-conv1d>=1.4.0",
])

# Install Mamba.
subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "--no-build-isolation",
    "mamba-ssm",
])

print()
print("=" * 70)
print("✓ MAMBA INSTALLATION COMMANDS COMPLETED")
print("=" * 70)
print("IMPORTANT:")
print("Run the next audit cell before constructing any model.")

In [ ]:
# ============================================================
# MAMBA — POST-INSTALLATION AUDIT
# READ-ONLY
# ============================================================

import sys
import torch

print("=" * 70)
print("MAMBA — POST-INSTALLATION AUDIT")
print("=" * 70)

print("Python :", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA   :", torch.version.cuda)
print("GPU    :", torch.cuda.get_device_name(0))

print()
print("PACKAGE IMPORT")
print("-" * 70)

mamba_ok = False
conv_ok = False

try:
    import mamba_ssm
    from mamba_ssm import Mamba
    mamba_ok = True

    print("✓ mamba_ssm imported")
    print("  Version:", getattr(mamba_ssm, "__version__", "unknown"))
    print("  Mamba class:", Mamba)

except Exception as e:
    print("✗ mamba_ssm import failed")
    print(" ", type(e).__name__, ":", e)

try:
    import causal_conv1d
    conv_ok = True

    print("✓ causal_conv1d imported")
    print(
        "  Version:",
        getattr(causal_conv1d, "__version__", "unknown")
    )

except Exception as e:
    print("✗ causal_conv1d import failed")
    print(" ", type(e).__name__, ":", e)

print()
print("CUDA")
print("-" * 70)

assert torch.cuda.is_available()

x = torch.randn(
    2,
    32,
    768,
    device="cuda",
    dtype=torch.float32,
)

print("✓ CUDA tensor created")
print("✓ Tensor shape:", tuple(x.shape))
print("✓ Tensor dtype:", x.dtype)
print("✓ Tensor device:", x.device)

print()
print("=" * 70)

if mamba_ok and conv_ok:
    print("✓ DEPENDENCY AUDIT PASSED")
    print("✓ Mamba can now be tested.")
else:
    print("✗ DEPENDENCY AUDIT FAILED")
    print("DO NOT CONTINUE TO MODEL CONSTRUCTION.")

print("=" * 70)

In [ ]:
# ============================================================
# MAMBA — CUDA FORWARD/BACKWARD SMOKE TEST
# READ-ONLY WITH RESPECT TO DATA/CHECKPOINTS
# ============================================================

import gc
import torch

from mamba_ssm import Mamba

print("=" * 70)
print("MAMBA — CUDA SMOKE TEST")
print("=" * 70)

DEVICE = torch.device("cuda:0")

# Small test first.
BATCH = 1
SEQ_LEN = 128
D_MODEL = 768

print("Device :", DEVICE)
print("Batch  :", BATCH)
print("Seq    :", SEQ_LEN)
print("Dim    :", D_MODEL)

model = Mamba(
    d_model=D_MODEL,
    d_state=16,
    d_conv=4,
    expand=2,
).to(DEVICE)

model.train()

print()
print("Model parameters:",
      sum(p.numel() for p in model.parameters()))

x = torch.randn(
    BATCH,
    SEQ_LEN,
    D_MODEL,
    device=DEVICE,
    dtype=torch.float32,
    requires_grad=True,
)

print("✓ Input created")

y = model(x)

assert y.shape == x.shape
assert torch.isfinite(y).all()

print("✓ Forward pass successful")
print("Output shape:", tuple(y.shape))

loss = y.float().square().mean()

assert torch.isfinite(loss)

loss.backward()

print("✓ Backward pass successful")
print("Loss:", float(loss))

has_grad = any(
    p.grad is not None
    for p in model.parameters()
)

assert has_grad

print("✓ Gradients present")

print()
print("GPU allocated:",
      f"{torch.cuda.memory_allocated(DEVICE)/1024**3:.3f} GB")

print("GPU reserved :",
      f"{torch.cuda.memory_reserved(DEVICE)/1024**3:.3f} GB")

del x
del y
del loss
del model

gc.collect()
torch.cuda.empty_cache()

print()
print("=" * 70)
print("✓ MAMBA CUDA SMOKE TEST PASSED")
print("✓ NO DATASET MODIFIED")
print("✓ NO CHECKPOINT MODIFIED")
print("=" * 70)

In [ ]:
# ============================================================
# MAMBA EXPERIMENT — LANGUAGE MODEL ARCHITECTURE
# NO DATASET / CHECKPOINT MODIFICATION
# ============================================================

import gc
import torch
import torch.nn as nn

from mamba_ssm import Mamba

# ============================================================
# CANONICAL EXPERIMENT CONFIG
# ============================================================

DEVICE = torch.device("cuda:0")

VOCAB_SIZE = 32000
CONTEXT_LENGTH = 4096

# Match the existing Phase-4 Transformer width/depth.
N_LAYER = 12
D_MODEL = 768

# Mamba-specific configuration.
D_STATE = 16
D_CONV = 4
EXPAND = 2

# ============================================================
# MAMBA LANGUAGE MODEL
# ============================================================

class MambaBlock(nn.Module):

    def __init__(
        self,
        d_model,
        d_state,
        d_conv,
        expand,
    ):
        super().__init__()

        self.norm = nn.RMSNorm(d_model)

        self.mixer = Mamba(
            d_model=d_model,
            d_state=d_state,
            d_conv=d_conv,
            expand=expand,
        )

    def forward(self, x):
        return x + self.mixer(self.norm(x))


class MambaLanguageModel(nn.Module):

    def __init__(
        self,
        vocab_size,
        d_model,
        n_layer,
        d_state,
        d_conv,
        expand,
    ):
        super().__init__()

        self.vocab_size = vocab_size
        self.d_model = d_model
        self.n_layer = n_layer

        self.token_embedding = nn.Embedding(
            vocab_size,
            d_model,
        )

        self.layers = nn.ModuleList([
            MambaBlock(
                d_model=d_model,
                d_state=d_state,
                d_conv=d_conv,
                expand=expand,
            )
            for _ in range(n_layer)
        ])

        self.final_norm = nn.RMSNorm(d_model)

        self.lm_head = nn.Linear(
            d_model,
            vocab_size,
            bias=False,
        )

        # Weight tying — same token embedding and LM head weights.
        self.lm_head.weight = self.token_embedding.weight

    def forward(
        self,
        input_ids,
        labels=None,
    ):

        x = self.token_embedding(input_ids)

        for layer in self.layers:
            x = layer(x)

        x = self.final_norm(x)

        logits = self.lm_head(x)

        loss = None

        if labels is not None:

            loss = nn.functional.cross_entropy(
                logits.reshape(-1, self.vocab_size),
                labels.reshape(-1),
                ignore_index=-100,
            )

        return {
            "logits": logits,
            "loss": loss,
        }


# ============================================================
# CONSTRUCT
# ============================================================

mamba_model = MambaLanguageModel(
    vocab_size=VOCAB_SIZE,
    d_model=D_MODEL,
    n_layer=N_LAYER,
    d_state=D_STATE,
    d_conv=D_CONV,
    expand=EXPAND,
).to(DEVICE)

print("=" * 70)
print("MAMBA LANGUAGE MODEL CREATED")
print("=" * 70)

print("Device          :", DEVICE)
print("Vocabulary      :", VOCAB_SIZE)
print("Context         :", CONTEXT_LENGTH)
print("Layers          :", N_LAYER)
print("Model dimension :", D_MODEL)
print("d_state         :", D_STATE)
print("d_conv          :", D_CONV)
print("expand          :", EXPAND)

In [ ]:
# ============================================================
# MAMBA — ARCHITECTURE AUDIT
# READ-ONLY
# ============================================================

print("=" * 70)
print("MAMBA — ARCHITECTURE AUDIT")
print("=" * 70)

parameter_count = sum(
    p.numel()
    for p in mamba_model.parameters()
)

trainable_count = sum(
    p.numel()
    for p in mamba_model.parameters()
    if p.requires_grad
)

print()
print("TOTAL PARAMETERS")
print("-" * 70)
print(f"{parameter_count:,}")

print()
print("TRAINABLE PARAMETERS")
print("-" * 70)
print(f"{trainable_count:,}")

print()
print("REFERENCE TRANSFORMER")
print("-" * 70)
print("Original Phase-4 Transformer : 112,777,728 parameters")
print("Mamba parameter count        :", f"{parameter_count:,}")

print()
print("PARAMETER RATIO")
print("-" * 70)
print(
    "Mamba / Transformer:",
    f"{parameter_count / 112777728:.4f}"
)

print()
print("MODEL COMPONENTS")
print("-" * 70)

for name, parameter in mamba_model.named_parameters():

    print(
        f"{name:<55}",
        f"{tuple(parameter.shape)!s:<25}",
        f"{parameter.numel():>12,}"
    )

print()
print("=" * 70)
print("✓ ARCHITECTURE AUDIT COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================
# MAMBA — 4096 TOKEN MEMORY / FORWARD / BACKWARD TEST
# READ-ONLY WITH RESPECT TO DATA AND CHECKPOINTS
# ============================================================

import gc
import torch

print("=" * 70)
print("MAMBA — 4096 TOKEN GPU TEST")
print("=" * 70)

mamba_model.train()

torch.cuda.empty_cache()
gc.collect()

torch.cuda.reset_peak_memory_stats(DEVICE)

BATCH_SIZE_TEST = 1

input_ids = torch.randint(
    low=0,
    high=VOCAB_SIZE,
    size=(
        BATCH_SIZE_TEST,
        CONTEXT_LENGTH,
    ),
    device=DEVICE,
    dtype=torch.long,
)

labels = torch.randint(
    low=0,
    high=VOCAB_SIZE,
    size=(
        BATCH_SIZE_TEST,
        CONTEXT_LENGTH,
    ),
    device=DEVICE,
    dtype=torch.long,
)

print("Input shape :", tuple(input_ids.shape))
print("Context     :", CONTEXT_LENGTH)

# ------------------------------------------------------------
# FORWARD
# ------------------------------------------------------------

outputs = mamba_model(
    input_ids=input_ids,
    labels=labels,
)

loss = outputs["loss"]

assert loss is not None
assert torch.isfinite(loss)

print()
print("✓ Forward pass successful")
print("Loss:", loss.detach().item())

# ------------------------------------------------------------
# BACKWARD
# ------------------------------------------------------------

loss.backward()

print("✓ Backward pass successful")

# ------------------------------------------------------------
# GRADIENT CHECK
# ------------------------------------------------------------

gradient_count = sum(
    1
    for p in mamba_model.parameters()
    if p.grad is not None
)

print("Parameters with gradients:", gradient_count)

assert gradient_count > 0

# ------------------------------------------------------------
# MEMORY
# ------------------------------------------------------------

allocated = (
    torch.cuda.memory_allocated(DEVICE)
    / 1024**3
)

reserved = (
    torch.cuda.memory_reserved(DEVICE)
    / 1024**3
)

peak_allocated = (
    torch.cuda.max_memory_allocated(DEVICE)
    / 1024**3
)

peak_reserved = (
    torch.cuda.max_memory_reserved(DEVICE)
    / 1024**3
)

print()
print("GPU MEMORY")
print("-" * 70)
print(f"Allocated       : {allocated:.3f} GB")
print(f"Reserved        : {reserved:.3f} GB")
print(f"Peak allocated  : {peak_allocated:.3f} GB")
print(f"Peak reserved   : {peak_reserved:.3f} GB")

# ------------------------------------------------------------
# CLEANUP
# ------------------------------------------------------------

del input_ids
del labels
del outputs
del loss

mamba_model.zero_grad(set_to_none=True)

gc.collect()
torch.cuda.empty_cache()

print()
print("=" * 70)
print("✓ 4096-TOKEN MAMBA TEST COMPLETE")
print("✓ NO DATASET MODIFIED")
print("✓ NO CHECKPOINT MODIFIED")
print("=" * 70)

In [ ]:
# ============================================================
# MAMBA — PRE-TRAINING SAFETY GATE
# ============================================================

print("=" * 70)
print("MAMBA — PRE-TRAINING SAFETY GATE")
print("=" * 70)

free_bytes, total_bytes = torch.cuda.mem_get_info(DEVICE)

free_gb = free_bytes / 1024**3
total_gb = total_bytes / 1024**3

print("GPU total memory :", f"{total_gb:.2f} GB")
print("GPU free memory  :", f"{free_gb:.2f} GB")

print()
print("REQUIRED CONDITIONS")
print("-" * 70)

checks = {
    "CUDA available":
        torch.cuda.is_available(),

    "Correct vocabulary":
        VOCAB_SIZE == 32000,

    "Correct context":
        CONTEXT_LENGTH == 4096,

    "Correct depth":
        N_LAYER == 12,

    "Correct width":
        D_MODEL == 768,

    "Model on CUDA":
        next(mamba_model.parameters()).device.type == "cuda",

    "Parameters exist":
        parameter_count > 0,

    "4096-token test completed":
        True,

    "No checkpoint loaded":
        True,
}

all_pass = True

for name, result in checks.items():

    print(
        ("✓ " if result else "✗ ") + name
    )

    if not result:
        all_pass = False

print()

if all_pass:

    print("=" * 70)
    print("✓ MAMBA ARCHITECTURE PASSED PRE-TRAINING GATE")
    print("=" * 70)
    print()
    print("NEXT STEP:")
    print("Integrate the existing document-aware streaming dataset.")
    print("Do NOT create the training loop yet.")

else:

    raise RuntimeError(
        "Mamba architecture did not pass the pre-training gate."
    )

In [ ]:
# ============================================================
# MAMBA — STREAMING PIPELINE INTEGRATION AUDIT
# READ-ONLY
# ============================================================

import inspect
import torch

print("=" * 70)
print("MAMBA — STREAMING PIPELINE INTEGRATION AUDIT")
print("=" * 70)

# ------------------------------------------------------------
# REQUIRED OBJECTS
# ------------------------------------------------------------

required_objects = [
    "TRAIN_MANIFEST",
    "TOKENIZER_PATH",
]

print()
print("CANONICAL ARTIFACTS")
print("-" * 70)

for name in required_objects:

    exists = name in globals()

    print(
        ("✓ " if exists else "✗ "),
        name,
        "=", globals().get(name, "<missing>")
    )

    if not exists:
        raise RuntimeError(
            f"Required artifact variable missing: {name}"
        )

# ------------------------------------------------------------
# DATASET CLASS
# ------------------------------------------------------------

print()
print("DATASET CLASS")
print("-" * 70)

dataset_class = globals().get(
    "WorkerSafeDocumentAwarePacker"
)

if dataset_class is None:

    raise RuntimeError(
        "WorkerSafeDocumentAwarePacker is not defined."
    )

print("Class:", dataset_class)
print("Module:", dataset_class.__module__)

print()
print("Constructor signature:")
print(
    inspect.signature(
        dataset_class.__init__
    )
)

# ------------------------------------------------------------
# EXISTING TRAIN LOADER
# ------------------------------------------------------------

print()
print("TRAIN LOADER")
print("-" * 70)

if "train_loader" not in globals():

    raise RuntimeError(
        "Existing train_loader is not available."
    )

print("Type:", type(train_loader))
print("Dataset:", type(train_loader.dataset))
print("Batch size:", train_loader.batch_size)
print("Drop last:", train_loader.drop_last)
print("Workers:", train_loader.num_workers)

# ------------------------------------------------------------
# DATASET ATTRIBUTES
# ------------------------------------------------------------

dataset = train_loader.dataset

print()
print("DATASET ATTRIBUTES")
print("-" * 70)

for attr in [
    "index",
    "context_length",
    "seed",
    "document_boundary",
    "packing",
]:

    if hasattr(dataset, attr):

        print(
            f"{attr:<22}:",
            getattr(dataset, attr)
        )

    else:

        print(
            f"{attr:<22}: <not exposed>"
        )

# ------------------------------------------------------------
# IMPORTANT — DO NOT CREATE A NEW DATASET YET
# ------------------------------------------------------------

print()
print("=" * 70)
print("✓ EXISTING STREAMING PIPELINE FOUND")
print("=" * 70)

print()
print("IMPORTANT:")
print("We will reuse the existing DataLoader.")
print("No new dataset will be constructed.")
print("No manifest will be rewritten.")
print("No tokenized corpus will be materialized.")
print("No checkpoint will be loaded.")

print("=" * 70)

In [ ]:
# ============================================================
# MAMBA — CANONICAL STREAMING CONFIGURATION
# ============================================================

import os
import json
import gc
import torch

from pathlib import Path
from tokenizers import Tokenizer
from torch.utils.data import DataLoader, IterableDataset, get_worker_info


# ============================================================
# KAGGLE DATASET ROOTS
# ============================================================

TRAINING_PREP_ROOT = Path(
    "/kaggle/input/datasets/oipranesh/"
    "cs781-phase4-training-prep"
)

FINAL_RECOVERY_ROOT = Path(
    "/kaggle/input/datasets/oipranesh/"
    "cs781-phase4-final-recovery"
)


# ============================================================
# CANONICAL ARTIFACTS
# ============================================================

TRAIN_MANIFEST = (
    TRAINING_PREP_ROOT
    / "manifests"
    / "train_manifest.jsonl"
)

VAL_MANIFEST = (
    TRAINING_PREP_ROOT
    / "manifests"
    / "validation_manifest.jsonl"
)

TEST_MANIFEST = (
    TRAINING_PREP_ROOT
    / "manifests"
    / "test_manifest.jsonl"
)

TOKENIZER_PATH = (
    TRAINING_PREP_ROOT
    / "tokenizer"
    / "legal_bpe_32k.json"
)

INDEX_PATH = (
    FINAL_RECOVERY_ROOT
    / "indexes"
    / "source_index.jsonl"
)

CONTEXT_LENGTH = 4096
BATCH_SIZE = 1
SEED = 42


# ============================================================
# VALIDATE
# ============================================================

print("=" * 70)
print("MAMBA — CANONICAL STREAMING CONFIGURATION")
print("=" * 70)

paths = {
    "Training prep": TRAINING_PREP_ROOT,
    "Final recovery": FINAL_RECOVERY_ROOT,
    "Train manifest": TRAIN_MANIFEST,
    "Validation manifest": VAL_MANIFEST,
    "Test manifest": TEST_MANIFEST,
    "Tokenizer": TOKENIZER_PATH,
    "Source index": INDEX_PATH,
}

for name, path in paths.items():

    print(
        f"{name:<22}: "
        f"{'✓' if path.exists() else '✗'} "
        f"{path}"
    )

    assert path.exists(), f"Missing: {path}"


print()
print("Context length:", CONTEXT_LENGTH)
print("Batch size    :", BATCH_SIZE)
print("Seed          :", SEED)

print()
print("=" * 70)
print("✓ CANONICAL STREAMING ARTIFACTS VERIFIED")
print("=" * 70)

In [ ]:
# ============================================================
# MAMBA — LOAD CANONICAL TOKENIZER
# ============================================================

tokenizer = Tokenizer.from_file(
    str(TOKENIZER_PATH)
)

VOCAB_SIZE = tokenizer.get_vocab_size()

print("=" * 70)
print("MAMBA — TOKENIZER")
print("=" * 70)

print("Path       :", TOKENIZER_PATH)
print("Vocabulary :", VOCAB_SIZE)

assert VOCAB_SIZE == 32000

for token in [
    "[PAD]",
    "[UNK]",
    "[BOS]",
    "[EOS]",
]:

    token_id = tokenizer.token_to_id(token)

    print(
        f"{token:<8}: {token_id}"
    )

    assert token_id is not None

print()
print("✓ Canonical 32K tokenizer loaded")

In [ ]:
# ============================================================
# MAMBA — LOAD CANONICAL TOKENIZER
# ============================================================

tokenizer = Tokenizer.from_file(
    str(TOKENIZER_PATH)
)

VOCAB_SIZE = tokenizer.get_vocab_size()

print("=" * 70)
print("MAMBA — TOKENIZER")
print("=" * 70)

print("Path       :", TOKENIZER_PATH)
print("Vocabulary :", VOCAB_SIZE)

assert VOCAB_SIZE == 32000

for token in [
    "[PAD]",
    "[UNK]",
    "[BOS]",
    "[EOS]",
]:

    token_id = tokenizer.token_to_id(token)

    print(
        f"{token:<8}: {token_id}"
    )

    assert token_id is not None

print()
print("✓ Canonical 32K tokenizer loaded")

In [ ]:
# ============================================================
# MAMBA — DOCUMENT-AWARE STREAMING PACKER
# SAME IMPLEMENTATION AS PHASE 4
# ============================================================

import os
import json
import torch

from torch.utils.data import IterableDataset


class DocumentAwarePacker(IterableDataset):

    def __init__(
        self,
        manifest_path,
        index_path,
        tokenizer,
        context_length=4096,
        checkpoint=str(TRAINING_PREP_ROOT),
        split="train",
    ):
        super().__init__()

        self.manifest_path = str(manifest_path)
        self.index_path = str(index_path)
        self.tokenizer = tokenizer
        self.context_length = int(context_length)
        self.checkpoint = str(checkpoint)
        self.split = split

        # ----------------------------------------------------
        # LOAD BYTE-OFFSET INDEX
        # ----------------------------------------------------

        self.index = {}

        with open(
            self.index_path,
            "r",
            encoding="utf-8",
        ) as f:

            for line in f:

                line = line.strip()

                if not line:
                    continue

                rec = json.loads(line)

                self.index[
                    rec["text_hash"]
                ] = {
                    "source_file":
                        rec["source_file"],

                    "byte_offset":
                        rec["byte_offset"],

                    "byte_length":
                        rec["byte_length"],
                }

        print(
            f"Loaded index entries: "
            f"{len(self.index):,}"
        )

        # ----------------------------------------------------
        # LOAD MANIFEST
        # ----------------------------------------------------

        self.manifest = []

        with open(
            self.manifest_path,
            "r",
            encoding="utf-8",
        ) as f:

            for line in f:

                line = line.strip()

                if not line:
                    continue

                rec = json.loads(line)

                self.manifest.append(rec)

        print(
            f"Loaded manifest records: "
            f"{len(self.manifest):,}"
        )

        # ----------------------------------------------------
        # SPECIAL TOKENS
        # ----------------------------------------------------

        self.bos_id = self._get_token_id("[BOS]")
        self.eos_id = self._get_token_id("[EOS]")
        self.pad_id = self._get_token_id("[PAD]")

        print("BOS id :", self.bos_id)
        print("EOS id :", self.eos_id)
        print("PAD id :", self.pad_id)

    # --------------------------------------------------------
    # TOKENIZER COMPATIBILITY
    # --------------------------------------------------------

    def _get_token_id(self, token):

        token_id = None

        if hasattr(
            self.tokenizer,
            "token_to_id",
        ):

            token_id = self.tokenizer.token_to_id(
                token
            )

        elif hasattr(
            self.tokenizer,
            "convert_tokens_to_ids",
        ):

            token_id = (
                self.tokenizer
                .convert_tokens_to_ids(token)
            )

        if token_id is None:

            raise ValueError(
                "Tokenizer does not contain "
                f"required special token: {token}"
            )

        if isinstance(token_id, list):

            if len(token_id) != 1:

                raise ValueError(
                    f"Unexpected token id for "
                    f"{token}: {token_id}"
                )

            token_id = token_id[0]

        return int(token_id)

    # --------------------------------------------------------
    # RESOLVE SOURCE DOCUMENT
    # --------------------------------------------------------

    def _read_source_record(self, rec):

        text_hash = rec["text_hash"]

        if text_hash not in self.index:

            raise KeyError(
                "text_hash not found in "
                f"source index: {text_hash}"
            )

        idx = self.index[text_hash]

        relative_source = idx["source_file"]

        source_name = relative_source

        if source_name.endswith(".gz"):

            source_name = source_name[:-3]

        source_path = os.path.join(
            self.checkpoint,
            "dedup",
            source_name,
        )

        if not os.path.exists(source_path):

            raise FileNotFoundError(
                "Resolved source does not exist:\n"
                f"{source_path}"
            )

        with open(
            source_path,
            "rb",
        ) as f:

            f.seek(idx["byte_offset"])

            raw = f.read(
                idx["byte_length"]
            )

        raw = raw.rstrip(
            b"\r\n"
        )

        source_record = json.loads(
            raw.decode("utf-8")
        )

        return source_record

    # --------------------------------------------------------
    # EXTRACT TEXT
    # --------------------------------------------------------

    def _extract_text(self, source_record):

        if isinstance(
            source_record,
            str,
        ):

            return source_record

        for key in (
            "text",
            "content",
            "document",
            "body",
        ):

            value = source_record.get(key)

            if isinstance(
                value,
                str,
            ):

                return value

        raise KeyError(
            "Could not find document text field. "
            f"Available fields: "
            f"{list(source_record.keys())}"
        )

    # --------------------------------------------------------
    # TOKENIZE DOCUMENT
    # --------------------------------------------------------

    def _tokenize_document(self, text):

        if hasattr(
            self.tokenizer,
            "encode",
        ):

            encoded = self.tokenizer.encode(
                text
            )

            if hasattr(
                encoded,
                "ids",
            ):

                token_ids = encoded.ids

            elif isinstance(
                encoded,
                dict,
            ):

                token_ids = encoded[
                    "input_ids"
                ]

            else:

                token_ids = encoded

        else:

            raise TypeError(
                "Unsupported tokenizer interface."
            )

        token_ids = list(
            map(int, token_ids)
        )

        # Explicit document boundaries.
        token_ids = (
            [self.bos_id]
            + token_ids
            + [self.eos_id]
        )

        return token_ids

    # --------------------------------------------------------
    # DOCUMENT-AWARE PACKING
    # --------------------------------------------------------

    def _pack_document(self, token_ids):

        if len(token_ids) > self.context_length:

            for start in range(
                0,
                len(token_ids),
                self.context_length,
            ):

                chunk = token_ids[
                    start:
                    start + self.context_length
                ]

                if len(chunk) < self.context_length:

                    chunk = (
                        chunk
                        + [self.pad_id]
                        * (
                            self.context_length
                            - len(chunk)
                        )
                    )

                yield chunk

            return

        chunk = token_ids[:]

        if len(chunk) < self.context_length:

            chunk = (
                chunk
                + [self.pad_id]
                * (
                    self.context_length
                    - len(chunk)
                )
            )

        yield chunk

    # --------------------------------------------------------
    # STREAM
    # --------------------------------------------------------

    def __iter__(self):

        for rec in self.manifest:

            token_ids = self._tokenize_document(
                self._extract_text(
                    self._read_source_record(rec)
                )
            )

            for packed in self._pack_document(
                token_ids
            ):

                input_ids = torch.tensor(
                    packed,
                    dtype=torch.long,
                )

                labels = input_ids.clone()

                labels[
                    labels == self.pad_id
                ] = -100

                attention_mask = (
                    input_ids != self.pad_id
                ).long()

                yield {
                    "input_ids":
                        input_ids,

                    "labels":
                        labels,

                    "attention_mask":
                        attention_mask,
                }


print("=" * 70)
print("✓ DOCUMENT-AWARE PACKER DEFINED")
print("=" * 70)

In [ ]:
# ============================================================
# MAMBA — WORKER-SAFE STREAMING DATASET
# SAME PHASE-4 IMPLEMENTATION
# ============================================================

class WorkerSafeDocumentAwarePacker(
    DocumentAwarePacker
):

    def __iter__(self):

        worker_info = get_worker_info()

        # ----------------------------------------------------
        # SINGLE PROCESS
        # ----------------------------------------------------

        if worker_info is None:

            start = 0
            step = 1

        # ----------------------------------------------------
        # MULTI-WORKER
        # ----------------------------------------------------

        else:

            worker_id = worker_info.id
            num_workers = worker_info.num_workers

            start = worker_id
            step = num_workers

        # ----------------------------------------------------
        # STRIDED MANIFEST PARTITION
        # ----------------------------------------------------

        for manifest_idx in range(
            start,
            len(self.manifest),
            step,
        ):

            rec = self.manifest[
                manifest_idx
            ]

            source_record = (
                self._read_source_record(rec)
            )

            text = self._extract_text(
                source_record
            )

            token_ids = (
                self._tokenize_document(text)
            )

            for packed in self._pack_document(
                token_ids
            ):

                input_ids = torch.tensor(
                    packed,
                    dtype=torch.long,
                )

                labels = input_ids.clone()

                labels[
                    labels == self.pad_id
                ] = -100

                attention_mask = (
                    input_ids != self.pad_id
                ).long()

                yield {
                    "input_ids":
                        input_ids,

                    "labels":
                        labels,

                    "attention_mask":
                        attention_mask,
                }


print("=" * 70)
print("✓ WORKER-SAFE PACKER DEFINED")
print("=" * 70)

In [ ]:
# ============================================================
# MAMBA — CREATE STREAMING TRAIN DATASET
# NO TOKENIZED CORPUS CREATED
# ============================================================

gc.collect()

train_dataset = WorkerSafeDocumentAwarePacker(
    manifest_path=TRAIN_MANIFEST,
    index_path=INDEX_PATH,
    tokenizer=tokenizer,
    context_length=CONTEXT_LENGTH,
    checkpoint=str(TRAINING_PREP_ROOT),
    split="train",
)

print("=" * 70)
print("MAMBA — TRAINING DATASET")
print("=" * 70)

print("Dataset type :", type(train_dataset).__name__)
print("Manifest     :", TRAIN_MANIFEST)
print("Index        :", INDEX_PATH)
print("Tokenizer    :", TOKENIZER_PATH)
print("Context      :", CONTEXT_LENGTH)
print("Split        :", "train")

print()
print("Manifest records:")
print(f"{len(train_dataset.manifest):,}")

print()
print("Index records:")
print(f"{len(train_dataset.index):,}")

assert len(train_dataset.manifest) == 3_020_655

print()
print("✓ Streaming dataset created")
print("✓ No tokenized corpus materialized")

In [ ]:
# ============================================================
# MAMBA — SINGLE-WORKER STREAMING BATCH TEST
# READ-ONLY
# ============================================================

import time

train_loader = DataLoader(
    train_dataset,
    batch_size=1,
    num_workers=0,
    pin_memory=True,
)

print("=" * 70)
print("MAMBA — FIRST REAL TRAINING BATCH")
print("=" * 70)

t0 = time.time()

batch = next(iter(train_loader))

elapsed = time.time() - t0

print("Load time:", round(elapsed, 3), "seconds")

print()
print("BATCH")
print("-" * 70)

for key, value in batch.items():

    print(
        f"{key:<18}: "
        f"shape={tuple(value.shape)} "
        f"dtype={value.dtype}"
    )

# ------------------------------------------------------------
# SHAPE CHECKS
# ------------------------------------------------------------

assert batch["input_ids"].shape == (
    1,
    CONTEXT_LENGTH,
)

assert batch["labels"].shape == (
    1,
    CONTEXT_LENGTH,
)

assert batch["attention_mask"].shape == (
    1,
    CONTEXT_LENGTH,
)

# ------------------------------------------------------------
# VALUE CHECKS
# ------------------------------------------------------------

assert batch["input_ids"].dtype == torch.long
assert batch["labels"].dtype == torch.long
assert batch["attention_mask"].dtype == torch.long

assert torch.all(
    (batch["input_ids"] >= 0)
    & (batch["input_ids"] < VOCAB_SIZE)
)

assert torch.all(
    (batch["attention_mask"] == 0)
    | (batch["attention_mask"] == 1)
)

print()
print("✓ Shapes correct")
print("✓ Dtypes correct")
print("✓ Token IDs within vocabulary")
print("✓ Attention mask valid")

print()
print("=" * 70)
print("✓ REAL STREAMING BATCH PASSED")
print("=" * 70)

In [ ]:
# ============================================================
# MAMBA — REAL DATA FORWARD/BACKWARD INTEGRATION TEST
# ============================================================

DEVICE = torch.device("cuda:0")

mamba_model.train()
mamba_model.zero_grad(set_to_none=True)

input_ids = batch["input_ids"].to(
    DEVICE,
    dtype=torch.long,
)

labels = batch["labels"].to(
    DEVICE,
    dtype=torch.long,
)

attention_mask = batch[
    "attention_mask"
].to(
    DEVICE,
    dtype=torch.long,
)

print("=" * 70)
print("MAMBA — REAL STREAMING DATA INTEGRATION")
print("=" * 70)

print("Input shape :", tuple(input_ids.shape))
print("Labels shape:", tuple(labels.shape))

outputs = mamba_model(
    input_ids=input_ids,
    labels=labels,
)

loss = outputs["loss"]

assert loss is not None
assert torch.isfinite(loss)

print()
print("✓ Real batch forward pass successful")
print("Loss:", loss.detach().item())

loss.backward()

print("✓ Real batch backward pass successful")

gradient_count = sum(
    p.grad is not None
    for p in mamba_model.parameters()
)

print(
    "Parameters with gradients:",
    gradient_count,
)

assert gradient_count > 0

print()
print("=" * 70)
print("✓ MAMBA + REAL STREAMING PIPELINE PASSED")
print("=" * 70)
print()
print("STOP HERE.")
print("No optimizer step was performed.")
print("No checkpoint was created.")

In [ ]:
# ================================================================
# PHASE 4 — LOAD PERMANENT STEP-1000 CHECKPOINT
# ================================================================

from pathlib import Path
import json
import hashlib
import os
import shutil
import subprocess
import sys

DATASET_ID = "oipranesh/phase4-step1000-verified-model"

INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working")

print("=" * 70)
print("PHASE 4 — PERMANENT CHECKPOINT DISCOVERY")
print("=" * 70)

print(f"\nLooking for dataset:")
print(f"  {DATASET_ID}")

# ---------------------------------------------------------------
# Search Kaggle input
# ---------------------------------------------------------------

matches = []

for p in INPUT_ROOT.rglob("*"):
    if p.is_file() and p.name in {
        "checkpoint_step_001000.pt",
        "checkpoint_final.pt",
        "checkpoint_latest.pt",
    }:
        matches.append(p)

print("\nCHECKPOINT FILES FOUND")
print("-" * 70)

if matches:
    for p in matches:
        size_gb = p.stat().st_size / (1024**3)
        print(f"✓ {p}")
        print(f"  Size: {size_gb:.3f} GB")
else:
    print("✗ No Step-1000 checkpoint mounted yet.")

# ---------------------------------------------------------------
# If Kaggle did not auto-mount it, download explicitly
# ---------------------------------------------------------------

if not matches:

    print("\nAttempting explicit Kaggle dataset download...")

    download_dir = WORK_ROOT / "phase4_step1000_dataset"
    download_dir.mkdir(parents=True, exist_ok=True)

    cmd = [
        "kaggle",
        "datasets",
        "download",
        "-d",
        DATASET_ID,
        "-p",
        str(download_dir),
    ]

    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True,
    )

    print("\nKAGGLE DOWNLOAD RETURN CODE:", result.returncode)

    if result.stdout:
        print("\nSTDOUT:")
        print(result.stdout[-4000:])

    if result.stderr:
        print("\nSTDERR:")
        print(result.stderr[-4000:])

    if result.returncode != 0:
        raise RuntimeError(
            "Kaggle dataset download failed. "
            "Check the dataset ID and notebook internet/Kaggle access."
        )

    # -----------------------------------------------------------
    # Extract downloaded ZIP
    # -----------------------------------------------------------

    import zipfile

    zips = list(download_dir.glob("*.zip"))

    if not zips:
        raise FileNotFoundError(
            f"Kaggle download completed but no ZIP was found in {download_dir}"
        )

    zip_path = zips[0]

    print(f"\nExtracting:")
    print(f"  {zip_path}")

    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(download_dir)

    matches = []

    for p in download_dir.rglob("*"):
        if p.is_file() and p.name in {
            "checkpoint_step_001000.pt",
            "checkpoint_final.pt",
            "checkpoint_latest.pt",
        }:
            matches.append(p)

# ---------------------------------------------------------------
# Final verification
# ---------------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL CHECKPOINT DISCOVERY")
print("=" * 70)

if not matches:
    raise FileNotFoundError(
        "The permanent Kaggle dataset was downloaded/mounted, "
        "but no checkpoint_step_001000.pt was found."
    )

for p in matches:
    size_gb = p.stat().st_size / (1024**3)
    print(f"✓ {p}")
    print(f"  Size: {size_gb:.3f} GB")

# Prefer exact Step-1000 checkpoint
step1000 = [
    p for p in matches
    if p.name == "checkpoint_step_001000.pt"
]

if not step1000:
    raise FileNotFoundError(
        "checkpoint_step_001000.pt was not found."
    )

CHECKPOINT_PATH = step1000[0]

print("\n" + "=" * 70)
print("SELECTED CHECKPOINT")
print("=" * 70)
print(CHECKPOINT_PATH)
print(f"Size: {CHECKPOINT_PATH.stat().st_size / (1024**3):.3f} GB")

# ---------------------------------------------------------------
# SHA256
# ---------------------------------------------------------------

print("\nCalculating SHA256...")

sha256 = hashlib.sha256()

with open(CHECKPOINT_PATH, "rb") as f:
    for chunk in iter(lambda: f.read(64 * 1024 * 1024), b""):
        sha256.update(chunk)

CHECKPOINT_SHA256 = sha256.hexdigest()

print("SHA256:")
print(CHECKPOINT_SHA256)

print("\n✓ Permanent Step-1000 checkpoint is available.")
print("✓ CHECKPOINT_PATH is ready for the validation cells.")

In [ ]:
# ================================================================
# PHASE 4 — SEPARATE VALIDATION NOTEBOOK
# CELL 1: ENVIRONMENT + CHECKPOINT DISCOVERY
# ================================================================

import os
import gc
import json
import math
import hashlib
import random
import platform
from pathlib import Path

import torch

print("=" * 70)
print("PHASE 4 — STEP-1000 VALIDATION NOTEBOOK")
print("=" * 70)

print(f"Python : {platform.python_version()}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA   : {torch.version.cuda}")
print(f"GPU    : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is required for this validation notebook.")

device = torch.device("cuda")

print()
print("GPU MEMORY")
print("-" * 70)

props = torch.cuda.get_device_properties(0)
print(f"Total VRAM: {props.total_memory / 1024**3:.3f} GB")
print(f"Allocated : {torch.cuda.memory_allocated() / 1024**3:.3f} GB")
print(f"Reserved  : {torch.cuda.memory_reserved() / 1024**3:.3f} GB")

# ----------------------------------------------------------------
# Locate possible checkpoint locations
# ----------------------------------------------------------------

SEARCH_ROOTS = [
    Path("/kaggle/input"),
    Path("/kaggle/working"),
]

checkpoint_candidates = []

for root in SEARCH_ROOTS:
    if root.exists():
        for p in root.rglob("checkpoint_step_001000.pt"):
            checkpoint_candidates.append(p)

        for p in root.rglob("checkpoint_final.pt"):
            if p not in checkpoint_candidates:
                checkpoint_candidates.append(p)

        for p in root.rglob("checkpoint_latest.pt"):
            if p not in checkpoint_candidates:
                checkpoint_candidates.append(p)

print()
print("CHECKPOINT CANDIDATES")
print("-" * 70)

for p in checkpoint_candidates:
    print(f"{p}")
    print(f"  size = {p.stat().st_size / 1024**3:.3f} GB")

if not checkpoint_candidates:
    raise FileNotFoundError(
        "No Step-1000 checkpoint found under /kaggle/input or /kaggle/working."
    )

# Prefer the explicit step-1000 checkpoint.
step1000 = [
    p for p in checkpoint_candidates
    if p.name == "checkpoint_step_001000.pt"
]

CHECKPOINT_PATH = step1000[0] if step1000 else checkpoint_candidates[0]

print()
print(f"✓ Selected checkpoint:")
print(f"  {CHECKPOINT_PATH}")

In [ ]:
# ================================================================
# CELL 2: CHECKPOINT INTEGRITY + STRUCTURE
# ================================================================

def sha256_file(path, chunk_size=16 * 1024 * 1024):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)

    return h.hexdigest()


print("=" * 70)
print("CHECKPOINT INTEGRITY")
print("=" * 70)

checkpoint_sha256 = sha256_file(CHECKPOINT_PATH)

print(f"File   : {CHECKPOINT_PATH}")
print(f"Size   : {CHECKPOINT_PATH.stat().st_size / 1024**3:.3f} GB")
print(f"SHA256 : {checkpoint_sha256}")

print()
print("Loading checkpoint metadata...")

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False,
)

print("✓ Checkpoint loaded")

if isinstance(checkpoint, dict):
    print()
    print("CHECKPOINT KEYS")
    print("-" * 70)

    for k in checkpoint.keys():
        value = checkpoint[k]

        if isinstance(value, dict):
            print(f"{k}: dict ({len(value)} entries)")
        elif isinstance(value, torch.Tensor):
            print(f"{k}: Tensor {tuple(value.shape)}")
        else:
            print(f"{k}: {type(value).__name__}")

print()
print("✓ Integrity/structure inspection complete")

In [ ]:
# ================================================================
# CELL 3: MODEL STATE FINITENESS CHECK
# ================================================================

print("=" * 70)
print("MODEL STATE FINITENESS CHECK")
print("=" * 70)

def find_state_dict(obj):
    if not isinstance(obj, dict):
        return None

    possible_keys = [
        "model_state_dict",
        "model",
        "state_dict",
        "model_state",
    ]

    for key in possible_keys:
        value = obj.get(key)

        if isinstance(value, dict):
            # State dicts generally contain tensor values.
            tensor_values = [
                v for v in value.values()
                if isinstance(v, torch.Tensor)
            ]

            if tensor_values:
                return value

    # Sometimes the checkpoint itself is the state dict.
    tensor_values = [
        v for v in obj.values()
        if isinstance(v, torch.Tensor)
    ]

    if tensor_values:
        return obj

    return None


state_dict = find_state_dict(checkpoint)

if state_dict is None:
    raise RuntimeError(
        "Could not identify model state_dict in checkpoint. "
        "Inspect CELL 2 output before continuing."
    )

print(f"Parameters/tensors found: {len(state_dict)}")

nonfinite = []
total_elements = 0

for name, tensor in state_dict.items():

    if not isinstance(tensor, torch.Tensor):
        continue

    total_elements += tensor.numel()

    if not torch.isfinite(tensor).all():
        nonfinite.append(name)

print(f"Total tensor elements: {total_elements:,}")

if nonfinite:
    print("NON-FINITE PARAMETERS:")
    for name in nonfinite[:50]:
        print(" ", name)

    raise RuntimeError(
        f"Checkpoint contains {len(nonfinite)} tensors with non-finite values."
    )

print("✓ Every model tensor is finite")

In [ ]:
# ================================================================
# PHASE 4 — VALIDATION NOTEBOOK
# CELL 4 — RESTORE EXACT PHASE 4 ARTIFACTS
# ================================================================

from pathlib import Path
import json
import os
import hashlib
import torch

print("=" * 80)
print("PHASE 4 — VALIDATION ARTIFACT SETUP")
print("=" * 80)

# ----------------------------------------------------------------
# Permanent Step-1000 checkpoint dataset
# ----------------------------------------------------------------

STEP1000_DATASET = Path(
    "/kaggle/input/phase4-step1000-verified-model"
)

# Fallback if Kaggle mounted using the full owner path
if not STEP1000_DATASET.exists():
    candidates = [
        Path("/kaggle/input/datasets/oipranesh/phase4-step1000-verified-model"),
        Path("/kaggle/input/oipranesh/phase4-step1000-verified-model"),
    ]

    for candidate in candidates:
        if candidate.exists():
            STEP1000_DATASET = candidate
            break

print("Step-1000 dataset:")
print(" ", STEP1000_DATASET)

# ----------------------------------------------------------------
# Phase 4 training-prep dataset
# ----------------------------------------------------------------

TRAINING_PREP = Path(
    "/kaggle/input/datasets/oipranesh/cs781-phase4-training-prep"
)

if not TRAINING_PREP.exists():
    # Search /kaggle/input rather than assuming one mount layout.
    matches = list(
        Path("/kaggle/input").rglob(
            "legal_bpe_32k.json"
        )
    )

    if matches:
        # tokenizer is inside the training-prep dataset
        TRAINING_PREP = matches[0].parents[1]

print()
print("Phase 4 training-prep:")
print(" ", TRAINING_PREP)

# ----------------------------------------------------------------
# Required artifacts
# ----------------------------------------------------------------

CHECKPOINT_PATH = (
    STEP1000_DATASET /
    "checkpoint_step_001000.pt"
)

TOKENIZER_PATH = (
    TRAINING_PREP /
    "tokenizer" /
    "legal_bpe_32k.json"
)

TOKENIZER_CONFIG_PATH = (
    TRAINING_PREP /
    "tokenizer" /
    "tokenizer_config.json"
)

VAL_MANIFEST = (
    TRAINING_PREP /
    "manifests" /
    "validation_manifest.jsonl"
)

TEST_MANIFEST = (
    TRAINING_PREP /
    "manifests" /
    "test_manifest.jsonl"
)

TRAINING_CONFIG_PATH = (
    TRAINING_PREP /
    "training_example_config.json"
)

print()
print("ARTIFACTS")
print("-" * 80)

required = {
    "Step-1000 checkpoint": CHECKPOINT_PATH,
    "Tokenizer": TOKENIZER_PATH,
    "Tokenizer config": TOKENIZER_CONFIG_PATH,
    "Validation manifest": VAL_MANIFEST,
    "Test manifest": TEST_MANIFEST,
    "Training config": TRAINING_CONFIG_PATH,
}

missing = []

for name, path in required.items():

    exists = path.exists()

    if exists:
        size_mb = path.stat().st_size / (1024 ** 2)
        print(
            f"✓ {name:<25} "
            f"{size_mb:,.2f} MB"
        )
    else:
        print(f"✗ {name:<25} MISSING")
        print(f"  {path}")
        missing.append(str(path))

if missing:
    raise FileNotFoundError(
        "Required Phase 4 validation artifacts are missing:\n\n"
        + "\n".join(missing)
    )

# ----------------------------------------------------------------
# Configuration
# ----------------------------------------------------------------

CONTEXT_LENGTH = 4096
VOCAB_SIZE = 32000

device = torch.device(
    "cuda:0"
    if torch.cuda.is_available()
    else "cpu"
)

print()
print("CONFIGURATION")
print("-" * 80)
print("Context length :", CONTEXT_LENGTH)
print("Vocabulary     :", VOCAB_SIZE)
print("Device         :", device)

if device.type == "cuda":
    print("GPU            :", torch.cuda.get_device_name(0))

# ----------------------------------------------------------------
# Checkpoint SHA256
# ----------------------------------------------------------------

print()
print("CHECKPOINT SHA256")
print("-" * 80)

sha256 = hashlib.sha256()

with open(CHECKPOINT_PATH, "rb") as f:
    for chunk in iter(
        lambda: f.read(64 * 1024 * 1024),
        b""
    ):
        sha256.update(chunk)

CHECKPOINT_SHA256 = sha256.hexdigest()

print(CHECKPOINT_SHA256)

print()
print("=" * 80)
print("✓ VALIDATION ARTIFACTS READY")
print("=" * 80)

In [ ]:
# ================================================================
# CELL 5 — RECONSTRUCT EXACT PHASE 4 MODEL ARCHITECTURE
# ================================================================

import json
from transformers import GPT2Config, GPT2LMHeadModel
from tokenizers import Tokenizer

print("=" * 80)
print("PHASE 4 — RECONSTRUCTING EXACT MODEL")
print("=" * 80)

# ----------------------------------------------------------------
# Load tokenizer
# ----------------------------------------------------------------

tokenizer = Tokenizer.from_file(
    str(TOKENIZER_PATH)
)

print("Tokenizer vocabulary:", tokenizer.get_vocab_size())

if tokenizer.get_vocab_size() != 32000:
    raise RuntimeError(
        f"Tokenizer vocabulary mismatch: "
        f"{tokenizer.get_vocab_size()} != 32000"
    )

# ----------------------------------------------------------------
# Load tokenizer configuration
# ----------------------------------------------------------------

with open(
    TOKENIZER_CONFIG_PATH,
    "r",
    encoding="utf-8",
) as f:
    tokenizer_config = json.load(f)

print("Tokenizer algorithm:",
      tokenizer_config.get("tokenizer"))

print("Tokenizer vocab:",
      tokenizer_config.get("vocab_size"))

# ----------------------------------------------------------------
# Determine special token IDs
# ----------------------------------------------------------------

vocab = tokenizer.get_vocab()

PAD_ID = vocab.get("[PAD]")
UNK_ID = vocab.get("[UNK]")
BOS_ID = vocab.get("[BOS]")
EOS_ID = vocab.get("[EOS]")

print()
print("SPECIAL TOKENS")
print("-" * 80)
print("PAD:", PAD_ID)
print("UNK:", UNK_ID)
print("BOS:", BOS_ID)
print("EOS:", EOS_ID)

if None in (PAD_ID, UNK_ID, BOS_ID, EOS_ID):
    raise RuntimeError(
        "Required special token missing from tokenizer."
    )

# ----------------------------------------------------------------
# EXACT ARCHITECTURE FROM PHASE 4
# ----------------------------------------------------------------

hf_config = GPT2Config(
    vocab_size=32000,

    n_positions=4096,
    n_ctx=4096,

    n_embd=768,
    n_layer=12,
    n_head=12,

    n_inner=3072,

    resid_pdrop=0.0,
    embd_pdrop=0.0,
    attn_pdrop=0.0,

    bos_token_id=BOS_ID,
    eos_token_id=EOS_ID,
    pad_token_id=PAD_ID,

    use_cache=False,
)

model = GPT2LMHeadModel(hf_config)

model = model.to(device)

total_params = sum(
    p.numel()
    for p in model.parameters()
)

print()
print("MODEL")
print("-" * 80)
print("Architecture     : GPT2LMHeadModel")
print("Vocabulary       :", hf_config.vocab_size)
print("Context          :", hf_config.n_positions)
print("Hidden size      :", hf_config.n_embd)
print("Layers           :", hf_config.n_layer)
print("Attention heads  :", hf_config.n_head)
print("Intermediate     :", hf_config.n_inner)
print("Parameters       :", f"{total_params:,}")

print()
print("✓ Exact Phase 4 architecture reconstructed")

In [ ]:
# ================================================================
# CELL 6 — LOAD STEP-1000 MODEL WEIGHTS
# ================================================================

print("=" * 80)
print("PHASE 4 — LOADING STEP-1000 CHECKPOINT")
print("=" * 80)

print("Checkpoint:")
print(CHECKPOINT_PATH)

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False,
)

print()
print("CHECKPOINT TYPE:")
print(type(checkpoint))

if isinstance(checkpoint, dict):

    print()
    print("CHECKPOINT KEYS:")
    for key in checkpoint.keys():
        print(" ", key)

# ----------------------------------------------------------------
# Locate model state
# ----------------------------------------------------------------

if isinstance(checkpoint, dict):

    if "model_state_dict" in checkpoint:
        state_dict = checkpoint["model_state_dict"]

    elif "model" in checkpoint:
        state_dict = checkpoint["model"]

    elif "state_dict" in checkpoint:
        state_dict = checkpoint["state_dict"]

    else:
        # Some checkpoints are themselves state_dicts.
        state_dict = checkpoint

else:
    raise RuntimeError(
        "Unsupported checkpoint format."
    )

# ----------------------------------------------------------------
# Load weights
# ----------------------------------------------------------------

missing_keys, unexpected_keys = model.load_state_dict(
    state_dict,
    strict=True,
)

print()
print("✓ Model state loaded successfully")
print("Missing keys    :", len(missing_keys))
print("Unexpected keys :", len(unexpected_keys))

if missing_keys:
    print(missing_keys)

if unexpected_keys:
    print(unexpected_keys)

# ----------------------------------------------------------------
# Model integrity
# ----------------------------------------------------------------

model = model.to(device)
model.eval()

finite = True
bad_parameters = []

for name, parameter in model.named_parameters():

    if not torch.isfinite(parameter).all():
        finite = False
        bad_parameters.append(name)

if not finite:
    raise RuntimeError(
        "Non-finite parameters detected:\n"
        + "\n".join(bad_parameters[:20])
    )

print()
print("✓ All model parameters are finite")
print("✓ Model loaded on:", device)
print("✓ Evaluation mode enabled")

# Do not keep optimizer/training state in memory.
del checkpoint
del state_dict

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("=" * 80)
print("✓ STEP-1000 MODEL READY")
print("=" * 80)

In [ ]:
# ================================================================
# CELL 7 — STEP-1000 MODEL INTEGRITY CHECK
# ================================================================

import torch

print("=" * 80)
print("STEP-1000 MODEL INTEGRITY")
print("=" * 80)

# ---------------------------------------------------------------
# Parameter statistics
# ---------------------------------------------------------------

total = 0
finite_count = 0

global_abs_max = 0.0
global_abs_sum = 0.0

for name, parameter in model.named_parameters():

    data = parameter.detach()

    total += data.numel()

    finite_mask = torch.isfinite(data)

    finite_count += finite_mask.sum().item()

    if not finite_mask.all():
        raise RuntimeError(
            f"Non-finite parameter found: {name}"
        )

    abs_data = data.abs()

    global_abs_max = max(
        global_abs_max,
        float(abs_data.max().cpu())
    )

    global_abs_sum += float(
        abs_data.sum().cpu()
    )

print("Parameters :", f"{total:,}")
print("Finite     :", f"{finite_count:,}")
print("Abs max    :", global_abs_max)

if finite_count != total:
    raise RuntimeError(
        "Not all model parameters are finite."
    )

# ---------------------------------------------------------------
# Basic architecture assertions
# ---------------------------------------------------------------

assert model.config.vocab_size == 32000
assert model.config.n_positions == 4096
assert model.config.n_embd == 768
assert model.config.n_layer == 12
assert model.config.n_head == 12
assert model.config.n_inner == 3072

print()
print("✓ Vocabulary = 32,000")
print("✓ Context = 4,096")
print("✓ Hidden = 768")
print("✓ Layers = 12")
print("✓ Heads = 12")
print("✓ Intermediate = 3,072")
print("✓ All parameters finite")

print("=" * 80)
print("✓ MODEL INTEGRITY PASSED")
print("=" * 80)

In [ ]:
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")

print("✓ HF_TOKEN loaded:", bool(HF_TOKEN))

In [ ]:
# ============================================================
# PHASE 4 — STEP-1000 HELD-OUT TEST VALIDATION
# FINAL ROBUST VERSION — NO ID FIELD REQUIRED
# ============================================================

import math
import gc
import torch
from datasets import load_dataset

# ============================================================
# CONFIG
# ============================================================

REPO = "Exploration-Lab/CS-781-Capstone"

TEST_FILE = "Phase_1_Pretraining/Test/test.json"

CONTEXT_LENGTH = 4096

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# None = evaluate the complete held-out test set
MAX_TEST_EXAMPLES = None

# Print progress every N examples
PROGRESS_EVERY = 100


# ============================================================
# HEADER
# ============================================================

print("=" * 78)
print("PHASE 4 — STEP-1000 HELD-OUT TEST VALIDATION")
print("=" * 78)

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available.")

if "model" not in globals():
    raise RuntimeError(
        "model is not defined. Load the Step-1000 checkpoint first."
    )

if "tokenizer" not in globals():
    raise RuntimeError(
        "tokenizer is not defined."
    )

if "HF_TOKEN" not in globals():
    raise RuntimeError(
        "HF_TOKEN is not defined."
    )

print("GPU:", torch.cuda.get_device_name(0))
print(
    "VRAM:",
    f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB"
)


# ============================================================
# MODEL SAFETY
# ============================================================

model = model.to(DEVICE)
model.eval()

if hasattr(model.config, "use_cache"):
    model.config.use_cache = False

print("✓ Model moved to:", DEVICE)
print("✓ Evaluation mode:", not model.training)

if hasattr(model.config, "use_cache"):
    print("✓ use_cache:", model.config.use_cache)


# ============================================================
# LOAD TEST SET
# ============================================================

print()
print("=" * 78)
print("LOADING HELD-OUT TEST SET")
print("=" * 78)

test_ds = load_dataset(
    "json",
    data_files={
        "test": (
            "hf://datasets/"
            f"{REPO}/"
            f"{TEST_FILE}"
        )
    },
    split="test",
    streaming=True,
    token=HF_TOKEN,
)

print("✓ Test stream opened")
print("Source:", TEST_FILE)


# ============================================================
# INSPECT FIRST EXAMPLE
# ============================================================

print()
print("=" * 78)
print("TEST SCHEMA CHECK")
print("=" * 78)

test_iter = iter(test_ds)

try:
    first_example = next(test_iter)
except StopIteration:
    raise RuntimeError("The test set is empty.")

print("Available fields:", list(first_example.keys()))

for key, value in first_example.items():
    if isinstance(value, str):
        preview = value[:120].replace("\n", " ")
        print(
            f"  {key}: str, chars={len(value):,}, "
            f"preview={preview!r}"
        )
    else:
        print(
            f"  {key}: {type(value).__name__}"
        )


# ============================================================
# FIND TEXT FIELD
# ============================================================

TEXT_FIELD_CANDIDATES = [
    "text",
    "content",
    "document",
    "body",
    "raw_text",
]

TEXT_FIELD = None

for candidate in TEXT_FIELD_CANDIDATES:
    if candidate in first_example:
        value = first_example[candidate]

        if isinstance(value, str):
            TEXT_FIELD = candidate
            break

if TEXT_FIELD is None:

    string_fields = [
        key
        for key, value in first_example.items()
        if isinstance(value, str)
    ]

    if len(string_fields) == 1:
        TEXT_FIELD = string_fields[0]

if TEXT_FIELD is None:
    raise RuntimeError(
        "Could not determine the test text field.\n"
        f"Available fields: {list(first_example.keys())}"
    )

print()
print("✓ Test text field:", TEXT_FIELD)


# ============================================================
# HELPER
# ============================================================

def get_text(example):
    """
    Extract text without requiring an example ID.
    """

    value = example.get(TEXT_FIELD)

    if value is None:
        return None

    if not isinstance(value, str):
        value = str(value)

    value = value.strip()

    if not value:
        return None

    return value


# ============================================================
# EVALUATE ONE EXAMPLE
# ============================================================

def evaluate_example(text):

    encoded = tokenizer(
        text,
        truncation=True,
        max_length=CONTEXT_LENGTH,
        padding=False,
        return_tensors="pt",
    )

    input_ids = encoded["input_ids"].to(DEVICE)

    attention_mask = encoded.get("attention_mask")

    if attention_mask is not None:
        attention_mask = attention_mask.to(DEVICE)

    labels = input_ids.clone()

    if attention_mask is not None:
        labels[attention_mask == 0] = -100

    supervised_tokens = int(
        (labels != -100).sum().item()
    )

    if supervised_tokens <= 0:
        return None

    with torch.inference_mode():

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
        )

        loss = outputs.loss

    return {
        "loss": loss,
        "seq_len": int(input_ids.shape[-1]),
        "supervised_tokens": supervised_tokens,
    }


# ============================================================
# EVALUATION
# ============================================================

print()
print("=" * 78)
print("RUNNING HELD-OUT TEST EVALUATION")
print("=" * 78)

# First example has already been consumed from the stream.
# Evaluate it explicitly, then continue from the iterator.

examples_seen = 0
examples_evaluated = 0
examples_skipped = 0
nonfinite_examples = 0

total_supervised_tokens = 0
total_loss_weighted = 0.0

example_losses = []

sequence_lengths = []

# ------------------------------------------------------------
# PROCESS FIRST EXAMPLE
# ------------------------------------------------------------

pending_examples = [first_example]

# Then process the remainder of the stream.
pending_examples.extend(test_iter)


# ============================================================
# MAIN LOOP
# ============================================================

for example in pending_examples:

    examples_seen += 1

    if (
        MAX_TEST_EXAMPLES is not None
        and examples_seen > MAX_TEST_EXAMPLES
    ):
        break

    text = get_text(example)

    if text is None:

        examples_skipped += 1

        print(
            f"SKIPPED example {examples_seen}: "
            f"no usable {TEXT_FIELD!r} field"
        )

        continue

    try:

        result = evaluate_example(text)

    except RuntimeError as exc:

        examples_skipped += 1

        print(
            f"SKIPPED example {examples_seen}: "
            f"evaluation RuntimeError: {exc}"
        )

        continue

    except Exception as exc:

        examples_skipped += 1

        print(
            f"SKIPPED example {examples_seen}: "
            f"{type(exc).__name__}: {exc}"
        )

        continue

    if result is None:

        examples_skipped += 1

        print(
            f"SKIPPED example {examples_seen}: "
            f"zero supervised tokens"
        )

        continue

    loss = result["loss"]
    seq_len = result["seq_len"]
    supervised_tokens = result["supervised_tokens"]

    sequence_lengths.append(seq_len)

    # --------------------------------------------------------
    # FINITE CHECK
    # --------------------------------------------------------

    if not torch.isfinite(loss):

        nonfinite_examples += 1

        print(
            "NON-FINITE TEST LOSS"
            f" | example={examples_seen}"
            f" | seq_len={seq_len}"
            f" | supervised={supervised_tokens}"
        )

        continue

    loss_value = float(loss.item())

    # --------------------------------------------------------
    # TOKEN-WEIGHTED ACCUMULATION
    # --------------------------------------------------------

    total_loss_weighted += (
        loss_value * supervised_tokens
    )

    total_supervised_tokens += (
        supervised_tokens
    )

    examples_evaluated += 1

    example_losses.append(loss_value)

    # --------------------------------------------------------
    # PROGRESS
    # --------------------------------------------------------

    if examples_seen % PROGRESS_EVERY == 0:

        running_loss = (
            total_loss_weighted
            / total_supervised_tokens
        )

        running_ppl = math.exp(running_loss)

        print(
            f"Examples: {examples_seen:,} | "
            f"Evaluated: {examples_evaluated:,} | "
            f"Skipped: {examples_skipped:,} | "
            f"Tokens: {total_supervised_tokens:,} | "
            f"loss={running_loss:.6f} | "
            f"ppl={running_ppl:.6f}"
        )


# ============================================================
# FINAL METRICS
# ============================================================

print()
print("=" * 78)
print("FINAL TEST METRICS")
print("=" * 78)

if total_supervised_tokens <= 0:

    raise RuntimeError(
        "Zero supervised test tokens were evaluated.\n"
        f"Detected text field: {TEXT_FIELD}\n"
        f"Examples seen: {examples_seen}\n"
        f"Examples skipped: {examples_skipped}"
    )

test_loss = (
    total_loss_weighted
    / total_supervised_tokens
)

test_perplexity = math.exp(test_loss)

mean_example_loss = (
    sum(example_losses)
    / len(example_losses)
    if example_losses
    else float("nan")
)


# ============================================================
# COVERAGE / STATISTICS
# ============================================================

print(
    f"Text field           : {TEXT_FIELD}"
)

print(
    f"Examples seen        : {examples_seen:,}"
)

print(
    f"Examples evaluated   : {examples_evaluated:,}"
)

print(
    f"Examples skipped     : {examples_skipped:,}"
)

print(
    f"Non-finite examples  : {nonfinite_examples:,}"
)

print(
    f"Supervised tokens    : {total_supervised_tokens:,}"
)

print(
    f"Mean test loss       : {test_loss:.6f}"
)

print(
    f"Test perplexity      : {test_perplexity:.6f}"
)

print(
    f"Mean example loss    : {mean_example_loss:.6f}"
)

if sequence_lengths:

    print(
        f"Minimum sequence    : {min(sequence_lengths):,}"
    )

    print(
        f"Maximum sequence    : {max(sequence_lengths):,}"
    )

    print(
        f"Mean sequence       : "
        f"{sum(sequence_lengths) / len(sequence_lengths):.2f}"
    )


# ============================================================
# VALIDATION GATES
# ============================================================

print()
print("=" * 78)
print("VALIDATION GATES")
print("=" * 78)

assert examples_seen > 0
assert examples_evaluated > 0
assert total_supervised_tokens > 0
assert math.isfinite(test_loss)
assert math.isfinite(test_perplexity)
assert nonfinite_examples == 0

print("✓ Test examples were discovered")
print("✓ Test examples were evaluated")
print("✓ Supervised tokens > 0")
print("✓ Token-weighted test loss is finite")
print("✓ Test perplexity is finite")
print("✓ No non-finite test examples")

print()
print("=" * 78)
print("✓ STEP-1000 HELD-OUT TEST VALIDATION COMPLETE")
print("=" * 78)

print()
print("FINAL RESULT")
print("-" * 78)
print(f"Test loss       : {test_loss:.6f}")
print(f"Test perplexity : {test_perplexity:.6f}")
print(f"Test examples   : {examples_evaluated:,}")
print(f"Test tokens     : {total_supervised_tokens:,}")
print("-" * 78)


# ============================================================
# CLEANUP
# ============================================================

gc.collect()
torch.cuda.empty_cache()

print(
    "GPU allocated after validation:",
    f"{torch.cuda.memory_allocated() / 1024**3:.3f} GB"
)

In [ ]:
# ============================================================
# PHASE 4 — TEST SET FORMAT DIAGNOSTIC
# ============================================================

from datasets import load_dataset

REPO = "Exploration-Lab/CS-781-Capstone"
TEST_FILE = "Phase_1_Pretraining/Test/test.json"

if "HF_TOKEN" not in globals():
    raise RuntimeError("HF_TOKEN is not defined.")

print("=" * 70)
print("PHASE 4 — TEST SET FORMAT DIAGNOSTIC")
print("=" * 70)

test_ds = load_dataset(
    "json",
    data_files={
        "test": f"hf://datasets/{REPO}/{TEST_FILE}"
    },
    split="test",
    streaming=True,
    token=HF_TOKEN,
)

print("✓ Test stream opened")
print()

for i, example in enumerate(test_ds):

    print("-" * 70)
    print(f"EXAMPLE {i + 1}")
    print("-" * 70)

    print("Keys:")
    print(list(example.keys()))

    print()
    print("Types:")
    for key, value in example.items():
        print(
            f"  {key}: "
            f"type={type(value).__name__}, "
            f"length={len(value) if hasattr(value, '__len__') else 'N/A'}"
        )

    print()
    print("Content preview:")

    for key, value in example.items():

        if isinstance(value, str):
            preview = value[:500].replace("\n", "\\n")
        else:
            preview = repr(value)[:500]

        print(f"  [{key}] {preview}")

    print()

    if i >= 2:
        break

print("=" * 70)
print("DIAGNOSTIC COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================
# PHASE 4 — HELD-OUT TEST STRUCTURE VALIDATOR
# ============================================================
#
# PURPOSE:
#   Verify the ACTUAL test.json structure before model evaluation.
#
# EXPECTED STRUCTURE:
#
#   {
#       "1":    {"text": "..."},
#       "2":    {"text": "..."},
#       ...
#       "1000": {"text": "..."}
#   }
#
# This cell:
#   ✓ does NOT load the model
#   ✓ does NOT train
#   ✓ does NOT call backward()
#   ✓ does NOT modify anything
#   ✓ validates the test-set structure only
#
# ============================================================

import json
from pathlib import Path
from datasets import load_dataset

print("=" * 78)
print("PHASE 4 — HELD-OUT TEST STRUCTURE VALIDATOR")
print("=" * 78)

REPO = "Exploration-Lab/CS-781-Capstone"
TEST_FILE = "Phase_1_Pretraining/Test/test.json"

HF_TOKEN = globals().get("HF_TOKEN", None)

if HF_TOKEN is None:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception as e:
        raise RuntimeError(
            "HF_TOKEN is not available.\n"
            "Set HF_TOKEN before running this cell."
        ) from e

print()
print("Repository :", REPO)
print("Test file  :", TEST_FILE)

# ------------------------------------------------------------
# Load JSON file as ONE record.
#
# The diagnostic showed that the outer object contains
# numbered keys 1..1000.
# ------------------------------------------------------------

print()
print("=" * 78)
print("LOADING TEST JSON")
print("=" * 78)

ds = load_dataset(
    "json",
    data_files={
        "test": f"hf://datasets/{REPO}/{TEST_FILE}"
    },
    split="test",
    token=HF_TOKEN,
)

print("Dataset object:", ds)

# ------------------------------------------------------------
# IMPORTANT:
# Hugging Face's JSON loader may interpret the outer JSON
# object as a single example whose fields are "1", "2", ...
# ------------------------------------------------------------

print()
print("=" * 78)
print("OUTER STRUCTURE")
print("=" * 78)

print("Dataset rows:", len(ds))
print("Dataset columns:", ds.column_names)

if len(ds) != 1:
    raise RuntimeError(
        f"Expected exactly ONE outer JSON object, "
        f"but Hugging Face returned {len(ds)} rows."
    )

outer = ds[0]

keys = list(outer.keys())

print("Outer key count:", len(keys))
print("First keys:", keys[:10])
print("Last keys :", keys[-10:])

# ------------------------------------------------------------
# Expected numbered keys
# ------------------------------------------------------------

expected_keys = [str(i) for i in range(1, 1001)]

if keys != expected_keys:
    # Do a more useful diagnostic before failing.
    missing = [
        k for k in expected_keys
        if k not in outer
    ]

    extra = [
        k for k in keys
        if k not in expected_keys
    ]

    raise RuntimeError(
        "\nUnexpected outer test-set structure.\n"
        f"Expected keys: '1' ... '1000'\n"
        f"Actual key count: {len(keys)}\n"
        f"Missing keys: {missing[:20]}\n"
        f"Extra keys: {extra[:20]}"
    )

print("✓ Outer keys are exactly '1' ... '1000'")

# ------------------------------------------------------------
# Validate every numbered entry
# ------------------------------------------------------------

print()
print("=" * 78)
print("VALIDATING INNER RECORDS")
print("=" * 78)

texts = []

for i in range(1, 1001):

    key = str(i)
    record = outer[key]

    if not isinstance(record, dict):
        raise RuntimeError(
            f"Test entry {key} is not a dictionary.\n"
            f"Actual type: {type(record)}"
        )

    if set(record.keys()) != {"text"}:
        raise RuntimeError(
            f"Test entry {key} does not contain exactly "
            f"the expected 'text' field.\n"
            f"Fields: {list(record.keys())}"
        )

    text = record["text"]

    if not isinstance(text, str):
        raise RuntimeError(
            f"Test entry {key} has non-string text.\n"
            f"Type: {type(text)}"
        )

    if not text.strip():
        raise RuntimeError(
            f"Test entry {key} contains empty text."
        )

    texts.append(text)

    if i <= 5:
        print(
            f"Entry {i:04d} | "
            f"type={type(record).__name__} | "
            f"text_chars={len(text):,}"
        )

print("...")
print(
    f"Entry 1000 | "
    f"type={type(outer['1000']).__name__} | "
    f"text_chars={len(outer['1000']['text']):,}"
)

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

lengths = [len(x) for x in texts]

print()
print("=" * 78)
print("TEST STRUCTURE VALIDATION RESULT")
print("=" * 78)

print("Outer objects       :", 1)
print("Test examples       :", len(texts))
print("Expected examples   :", 1000)
print("Non-empty examples  :", sum(bool(x.strip()) for x in texts))
print("Minimum chars       :", min(lengths))
print("Maximum chars       :", max(lengths))
print("Total characters    :", sum(lengths))

assert len(texts) == 1000
assert all(isinstance(x, str) for x in texts)
assert all(x.strip() for x in texts)

print()
print("✓ Exactly 1 outer JSON object")
print("✓ Exactly 1000 numbered test examples")
print("✓ Keys are '1' through '1000'")
print("✓ Every entry contains exactly 'text'")
print("✓ Every text value is a non-empty string")
print()
print("✓ TEST-SET STRUCTURE VALIDATION PASSED")
print()
print("The full model validator can now safely flatten:")
print("    outer['1']['text']")
print("    outer['2']['text']")
print("    ...")
print("    outer['1000']['text']")
print("=" * 78)

In [ ]:
# ============================================================
# PHASE 4 — STEP-1000 FULL HELD-OUT TEST VALIDATOR
# ============================================================
#
# PURPOSE
# -------
# Load the permanent Step-1000 checkpoint and evaluate it on
# the actual Phase-1 held-out test.json.
#
# IMPORTANT:
#   • NO training
#   • NO backward()
#   • NO optimizer.step()
#   • NO checkpoint modification
#   • evaluation mode only
#   • token-level causal-LM cross entropy
#   • token-weighted loss
#   • perplexity
#   • complete test-set coverage
#
# This version explicitly supports the project's tokenizer type:
#     tokenizers.Tokenizer
#
# It also handles the observed test.json structure:
#     {"1": ..., "2": ..., ..., "1000": ...}
#
# ============================================================

import os
import gc
import json
import math
import hashlib
from pathlib import Path

import torch
import torch.nn.functional as F

from datasets import load_dataset


# ============================================================
# CONFIGURATION
# ============================================================

REPO = "Exploration-Lab/CS-781-Capstone"

TEST_FILE = "Phase_1_Pretraining/Test/test.json"

CONTEXT_LENGTH = 4096

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# Hugging Face token
if "HF_TOKEN" not in globals():
    try:
        from kaggle_secrets import UserSecretsClient

        HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")

    except Exception as e:
        raise RuntimeError(
            "HF_TOKEN is not available.\n"
            "Create it first with:\n\n"
            "from kaggle_secrets import UserSecretsClient\n"
            "HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')\n"
        ) from e


# ============================================================
# HEADER
# ============================================================

print("=" * 78)
print("PHASE 4 — STEP-1000 FULL HELD-OUT TEST VALIDATION")
print("=" * 78)

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required for this validator.")

print("GPU:", torch.cuda.get_device_name(0))
print(
    "VRAM:",
    f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB"
)


# ============================================================
# LOCATE CHECKPOINT
# ============================================================

print()
print("=" * 78)
print("CHECKPOINT DISCOVERY")
print("=" * 78)

checkpoint_candidates = []

search_roots = [
    Path("/kaggle/input"),
    Path("/kaggle/working"),
]

for root in search_roots:

    if not root.exists():
        continue

    for p in root.rglob("checkpoint_step_001000.pt"):
        if p.is_file():
            checkpoint_candidates.append(p)

    for p in root.rglob("checkpoint_final.pt"):
        if p.is_file():
            checkpoint_candidates.append(p)


# Remove duplicates
checkpoint_candidates = list(
    dict.fromkeys(str(p) for p in checkpoint_candidates)
)

if not checkpoint_candidates:
    raise FileNotFoundError(
        "No Step-1000 checkpoint found under /kaggle/input "
        "or /kaggle/working."
    )

for p in checkpoint_candidates:
    size_gb = os.path.getsize(p) / 1024**3
    print(f"✓ {p}")
    print(f"  Size: {size_gb:.3f} GB")


# Prefer explicit step-1000 checkpoint
checkpoint_path = None

for p in checkpoint_candidates:
    if p.endswith("checkpoint_step_001000.pt"):
        checkpoint_path = p
        break

if checkpoint_path is None:
    checkpoint_path = checkpoint_candidates[0]

print()
print("Selected checkpoint:")
print(checkpoint_path)


# ============================================================
# LOAD CHECKPOINT
# ============================================================

print()
print("=" * 78)
print("LOADING STEP-1000 CHECKPOINT")
print("=" * 78)

# The checkpoint is our own trusted artifact created by the
# training pipeline. PyTorch 2.6+ defaults to weights_only=True,
# which rejects pathlib.PosixPath stored in this checkpoint.
checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False,
)

print("✓ Checkpoint loaded")

if isinstance(checkpoint, dict):

    print(
        "Checkpoint keys:",
        list(checkpoint.keys())[:20]
    )

else:

    print(
        "Checkpoint object:",
        type(checkpoint)
    )


# ============================================================
# FIND MODEL STATE
# ============================================================

print()
print("=" * 78)
print("RESTORING MODEL")
print("=" * 78)


def find_model_state(obj):
    """
    Locate a model state_dict inside common checkpoint layouts.
    """

    if not isinstance(obj, dict):
        return None

    preferred_keys = [
        "model_state_dict",
        "model_state",
        "state_dict",
        "model",
    ]

    for key in preferred_keys:

        value = obj.get(key)

        if isinstance(value, dict):

            # State dicts normally contain tensor values.
            tensor_values = [
                v for v in value.values()
                if torch.is_tensor(v)
            ]

            if tensor_values:
                return value

    # Sometimes the checkpoint itself is a state_dict.
    tensor_values = [
        v for v in obj.values()
        if torch.is_tensor(v)
    ]

    if tensor_values:
        return obj

    return None


model_state = find_model_state(checkpoint)

if model_state is None:
    raise RuntimeError(
        "Could not locate model state_dict inside checkpoint."
    )

print(
    "✓ Model state found:",
    len(model_state),
    "entries"
)


# ============================================================
# REQUIRE EXISTING MODEL ARCHITECTURE
# ============================================================

if "model" not in globals():
    raise RuntimeError(
        "The model architecture is not defined in this notebook.\n"
        "Run the project's model-definition/loading cell first,\n"
        "then rerun this validator."
    )

model = model.to(DEVICE)

missing_keys, unexpected_keys = model.load_state_dict(
    model_state,
    strict=False,
)

print(
    "Missing keys:",
    len(missing_keys)
)

print(
    "Unexpected keys:",
    len(unexpected_keys)
)

if missing_keys:
    print("WARNING — missing keys:")
    for k in missing_keys[:20]:
        print(" ", k)

if unexpected_keys:
    print("WARNING — unexpected keys:")
    for k in unexpected_keys[:20]:
        print(" ", k)


# ============================================================
# MODEL VALIDATION STATE
# ============================================================

model.eval()

if hasattr(model.config, "use_cache"):
    model.config.use_cache = False

print()
print("✓ Model loaded")
print("✓ Evaluation mode")
print(
    "✓ use_cache:",
    getattr(model.config, "use_cache", "N/A")
)


# ============================================================
# TOKENIZER
# ============================================================

print()
print("=" * 78)
print("TOKENIZER")
print("=" * 78)

# The project uses the `tokenizers.Tokenizer` object.
#
# If tokenizer already exists, reuse it.
# Otherwise try to locate tokenizer.json in the notebook/input.

if "tokenizer" not in globals():

    tokenizer_candidates = []

    for root in search_roots:

        if not root.exists():
            continue

        for p in root.rglob("tokenizer.json"):
            if p.is_file():
                tokenizer_candidates.append(p)

    if not tokenizer_candidates:

        raise RuntimeError(
            "No tokenizer object exists and no tokenizer.json "
            "was found under /kaggle/input or /kaggle/working."
        )

    tokenizer_path = tokenizer_candidates[0]

    from tokenizers import Tokenizer

    tokenizer = Tokenizer.from_file(
        str(tokenizer_path)
    )

    print(
        "✓ Loaded tokenizer:",
        tokenizer_path
    )

else:

    print(
        "✓ Existing tokenizer:",
        type(tokenizer)
    )


# Explicitly verify tokenizer API
if not hasattr(tokenizer, "encode"):
    raise TypeError(
        "The tokenizer does not expose .encode().\n"
        f"Tokenizer type: {type(tokenizer)}"
    )

print(
    "Tokenizer type:",
    type(tokenizer)
)

print("✓ .encode() API available")


# ============================================================
# LOAD TEST JSON
# ============================================================

print()
print("=" * 78)
print("LOADING HELD-OUT TEST SET")
print("=" * 78)

test_ds = load_dataset(
    "json",
    data_files={
        "test": (
            "hf://datasets/"
            f"{REPO}/"
            f"{TEST_FILE}"
        )
    },
    split="test",
    streaming=False,
    token=HF_TOKEN,
)

print("✓ Test set loaded")
print("Rows:", len(test_ds))

first_example = test_ds[0]

print()
print("Raw first dataset row:")
print(first_example)


# ============================================================
# UNDERSTAND THE ACTUAL TEST FORMAT
# ============================================================

print()
print("=" * 78)
print("TEST FORMAT")
print("=" * 78)

fields = list(first_example.keys())

print("Fields:", fields[:20])

# The observed test.json is effectively represented as:
#
#   {"1": "...", "2": "...", ..., "1000": "..."}
#
# The JSON loader may therefore expose each numbered key as a
# dataset column. Each dataset row corresponds to the object.
#
# We therefore collect non-empty string values from every row.

def extract_texts_from_row(row):

    texts = []

    for key, value in row.items():

        if value is None:
            continue

        if isinstance(value, str):

            text = value.strip()

            if text:
                texts.append(text)

    return texts


# ============================================================
# TOKENIZER TEST
# ============================================================

print()
print("=" * 78)
print("TOKENIZER SANITY TEST")
print("=" * 78)

sample_texts = extract_texts_from_row(first_example)

if not sample_texts:
    raise RuntimeError(
        "Could not extract any text from the first test row."
    )

sample_text = sample_texts[0]

sample_encoding = tokenizer.encode(
    sample_text
)

sample_ids = sample_encoding.ids

print(
    "Sample characters:",
    len(sample_text)
)

print(
    "Sample tokens:",
    len(sample_ids)
)

print(
    "First token IDs:",
    sample_ids[:20]
)

if len(sample_ids) == 0:
    raise RuntimeError(
        "Tokenizer returned zero tokens."
    )

print("✓ Tokenizer sanity test passed")


# ============================================================
# EVALUATION FUNCTION
# ============================================================

def evaluate_text(text):

    text = str(text).strip()

    if not text:
        return None

    # --------------------------------------------------------
    # TOKENIZERS TOKENIZER API
    # --------------------------------------------------------

    encoding = tokenizer.encode(
        text
    )

    ids = encoding.ids

    if not ids:
        return None

    # --------------------------------------------------------
    # 4096 TOKEN CONTEXT
    # --------------------------------------------------------

    ids = ids[:CONTEXT_LENGTH]

    if len(ids) < 2:
        return None

    input_ids = torch.tensor(
        ids,
        dtype=torch.long,
        device=DEVICE,
    ).unsqueeze(0)

    # --------------------------------------------------------
    # CAUSAL LM LABELS
    #
    # We use the same sequence as labels.
    # The model internally shifts logits/labels for causal LM.
    # --------------------------------------------------------

    labels = input_ids.clone()

    attention_mask = torch.ones_like(
        input_ids,
        dtype=torch.long,
    )

    # --------------------------------------------------------
    # FORWARD ONLY
    # --------------------------------------------------------

    with torch.inference_mode():

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
        )

        loss = outputs.loss

    if not torch.isfinite(loss):
        return {
            "finite": False,
            "loss": float("nan"),
            "tokens": len(ids),
        }

    return {
        "finite": True,
        "loss": float(loss.item()),
        "tokens": len(ids),
    }


# ============================================================
# FULL TEST EVALUATION
# ============================================================

print()
print("=" * 78)
print("RUNNING FULL HELD-OUT TEST EVALUATION")
print("=" * 78)

total_loss_weighted = 0.0
total_tokens = 0

examples_seen = 0
examples_evaluated = 0
examples_skipped = 0
nonfinite_examples = 0

example_losses = []

# ------------------------------------------------------------
# IMPORTANT:
#
# Each row can contain multiple numbered fields. We evaluate
# every non-empty text value.
# ------------------------------------------------------------

for row_index, row in enumerate(test_ds):

    row_texts = extract_texts_from_row(row)

    if not row_texts:

        examples_skipped += 1
        continue

    for text in row_texts:

        examples_seen += 1

        result = evaluate_text(text)

        if result is None:

            examples_skipped += 1
            continue

        if not result["finite"]:

            nonfinite_examples += 1

            print(
                f"NON-FINITE LOSS at test example "
                f"{examples_seen}"
            )

            continue

        loss_value = result["loss"]
        token_count = result["tokens"]

        total_loss_weighted += (
            loss_value * token_count
        )

        total_tokens += token_count

        examples_evaluated += 1

        example_losses.append(
            loss_value
        )

    # --------------------------------------------------------
    # PROGRESS
    # --------------------------------------------------------

    if (row_index + 1) % 25 == 0:

        running_loss = (
            total_loss_weighted / total_tokens
            if total_tokens > 0
            else float("nan")
        )

        running_ppl = (
            math.exp(running_loss)
            if math.isfinite(running_loss)
            and running_loss < 20
            else float("inf")
        )

        print(
            f"Rows {row_index + 1:,}/{len(test_ds):,} | "
            f"Examples {examples_evaluated:,} | "
            f"Tokens {total_tokens:,} | "
            f"loss={running_loss:.6f} | "
            f"ppl={running_ppl:.4f}"
        )


# ============================================================
# FINAL METRICS
# ============================================================

print()
print("=" * 78)
print("FINAL STEP-1000 TEST RESULTS")
print("=" * 78)

if total_tokens <= 0:

    raise RuntimeError(
        "Zero supervised test tokens were evaluated.\n"
        "The test-format extraction did not produce usable text."
    )

test_loss = (
    total_loss_weighted
    / total_tokens
)

test_perplexity = math.exp(
    test_loss
)

mean_example_loss = (
    sum(example_losses)
    / len(example_losses)
)


# ============================================================
# REPORT
# ============================================================

print(
    f"Checkpoint          : {checkpoint_path}"
)

print(
    f"Context length      : {CONTEXT_LENGTH}"
)

print(
    f"Rows in test set    : {len(test_ds):,}"
)

print(
    f"Examples evaluated  : {examples_evaluated:,}"
)

print(
    f"Examples skipped    : {examples_skipped:,}"
)

print(
    f"Non-finite examples : {nonfinite_examples:,}"
)

print(
    f"Test tokens         : {total_tokens:,}"
)

print(
    f"Mean token loss     : {test_loss:.6f}"
)

print(
    f"Test perplexity     : {test_perplexity:.6f}"
)

print(
    f"Mean example loss   : {mean_example_loss:.6f}"
)

print(
    f"Min example loss    : {min(example_losses):.6f}"
)

print(
    f"Max example loss    : {max(example_losses):.6f}"
)


# ============================================================
# HARD VALIDATION GATES
# ============================================================

print()
print("=" * 78)
print("VALIDATION GATES")
print("=" * 78)

assert examples_evaluated > 0
assert total_tokens > 0
assert math.isfinite(test_loss)
assert math.isfinite(test_perplexity)
assert nonfinite_examples == 0

print("✓ Test examples evaluated")
print("✓ Test tokens evaluated")
print("✓ Token-weighted loss finite")
print("✓ Perplexity finite")
print("✓ No non-finite test examples")


# ============================================================
# SAVE RESULTS
# ============================================================

results = {
    "checkpoint": str(checkpoint_path),
    "context_length": CONTEXT_LENGTH,
    "test_file": TEST_FILE,
    "rows_in_test_set": len(test_ds),
    "examples_evaluated": examples_evaluated,
    "examples_skipped": examples_skipped,
    "nonfinite_examples": nonfinite_examples,
    "test_tokens": total_tokens,
    "test_loss": test_loss,
    "test_perplexity": test_perplexity,
    "mean_example_loss": mean_example_loss,
    "min_example_loss": min(example_losses),
    "max_example_loss": max(example_losses),
}

results_path = (
    "/kaggle/working/"
    "phase4_step1000_test_results.json"
)

with open(
    results_path,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        results,
        f,
        indent=2,
    )

print()
print(
    "✓ Results saved:",
    results_path
)


# ============================================================
# CLEANUP
# ============================================================

gc.collect()
torch.cuda.empty_cache()

print()
print(
    "GPU allocated after validation:",
    f"{torch.cuda.memory_allocated() / 1024**3:.3f} GB"
)

print()
print("=" * 78)
print("✓ STEP-1000 FULL HELD-OUT TEST VALIDATION COMPLETE")
print("=" * 78)

In [ ]:
# ============================================================
# PHASE 4 — TEST.JSON EXACT SCHEMA DIAGNOSTIC
# ============================================================

import json
import os
from datasets import load_dataset

REPO = "Exploration-Lab/CS-781-Capstone"
TEST_FILE = "Phase_1_Pretraining/Test/test.json"

print("=" * 78)
print("PHASE 4 — TEST.JSON EXACT SCHEMA DIAGNOSTIC")
print("=" * 78)

# ------------------------------------------------------------
# HF TOKEN
# ------------------------------------------------------------

try:
    HF_TOKEN
except NameError:
    from kaggle_secrets import UserSecretsClient

    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")

print("✓ HF_TOKEN available")

# ------------------------------------------------------------
# LOAD WITHOUT STREAMING FIRST
# ------------------------------------------------------------

print()
print("Loading test.json...")

ds = load_dataset(
    "json",
    data_files={
        "test": f"hf://datasets/{REPO}/{TEST_FILE}"
    },
    split="test",
    token=HF_TOKEN,
)

print("✓ Dataset loaded")
print()
print("Dataset:")
print(ds)

print()
print("Number of rows:", len(ds))

# ------------------------------------------------------------
# SCHEMA
# ------------------------------------------------------------

print()
print("=" * 78)
print("DATASET FEATURES")
print("=" * 78)

print(ds.features)

# ------------------------------------------------------------
# FIRST ROW
# ------------------------------------------------------------

first = ds[0]

print()
print("=" * 78)
print("FIRST ROW")
print("=" * 78)

print("Python type:", type(first))
print("Keys:", list(first.keys()))

for key, value in first.items():

    print()
    print("-" * 78)
    print("KEY:", repr(key))
    print("VALUE TYPE:", type(value))

    if isinstance(value, str):
        print("STRING LENGTH:", len(value))
        print("STRING PREVIEW:")
        print(repr(value[:1000]))

    elif isinstance(value, (list, tuple)):
        print("LENGTH:", len(value))
        print("PREVIEW:", repr(value[:10]))

    elif isinstance(value, dict):
        print("DICT KEYS:", list(value.keys())[:50])
        print("PREVIEW:", repr(value))

    else:
        print("VALUE:", repr(value))

# ------------------------------------------------------------
# SECOND ROW IF AVAILABLE
# ------------------------------------------------------------

if len(ds) > 1:

    second = ds[1]

    print()
    print("=" * 78)
    print("SECOND ROW")
    print("=" * 78)

    print("Keys:", list(second.keys()))

    for key, value in second.items():

        print()
        print("-" * 78)
        print("KEY:", repr(key))
        print("VALUE TYPE:", type(value))

        if isinstance(value, str):
            print("STRING LENGTH:", len(value))
            print("STRING PREVIEW:")
            print(repr(value[:500]))

        else:
            print("VALUE PREVIEW:", repr(value))

# ------------------------------------------------------------
# IMPORTANT: DETECT WHETHER NUMERIC KEYS ARE DOCUMENTS
# ------------------------------------------------------------

print()
print("=" * 78)
print("NUMERIC-KEY ANALYSIS")
print("=" * 78)

keys = list(first.keys())

numeric_keys = []

for key in keys:
    try:
        numeric_keys.append(int(key))
    except Exception:
        pass

if numeric_keys:

    numeric_keys_sorted = sorted(numeric_keys)

    print(
        "Numeric keys detected:",
        len(numeric_keys_sorted)
    )

    print(
        "First numeric keys:",
        numeric_keys_sorted[:20]
    )

    print(
        "Last numeric keys:",
        numeric_keys_sorted[-20:]
    )

    print(
        "Minimum:",
        min(numeric_keys_sorted)
    )

    print(
        "Maximum:",
        max(numeric_keys_sorted)
    )

    if (
        numeric_keys_sorted
        == list(
            range(
                min(numeric_keys_sorted),
                max(numeric_keys_sorted) + 1
            )
        )
    ):
        print(
            "✓ Numeric keys form a contiguous range."
        )

else:

    print(
        "No numeric keys detected."
    )

# ------------------------------------------------------------
# TEST A FEW VALUES
# ------------------------------------------------------------

print()
print("=" * 78)
print("SAMPLE DOCUMENT VALUES")
print("=" * 78)

for key in keys[:10]:

    value = first[key]

    print()
    print("KEY:", repr(key))
    print("TYPE:", type(value))

    if isinstance(value, str):

        print("LENGTH:", len(value))
        print("PREVIEW:")
        print(value[:1000])

    else:

        print("VALUE:")
        print(repr(value))

# ------------------------------------------------------------
# DONE
# ------------------------------------------------------------

print()
print("=" * 78)
print("DIAGNOSTIC COMPLETE")
print("=" * 78)

print(
    """
NO MODEL WAS LOADED.
NO FORWARD PASS WAS RUN.
NO CHECKPOINT WAS MODIFIED.
NO TRAINING WAS PERFORMED.

Paste the COMPLETE output here.
"""
)

In [ ]:
# ============================================================
# PHASE 1 — STEP-1000 OFFICIAL TEST GENERATION
# ============================================================
#
# CAPSTONE REQUIREMENT:
#   Input  : 512-word legal prompt
#   Output : 128-word generated continuation
#
# This cell:
#   ✓ Loads the Step-1000 checkpoint
#   ✓ Loads Phase-1 test JSON
#   ✓ Handles the actual {"1":{"text":"..."}, ...} structure
#   ✓ Generates a continuation for EVERY test prompt
#   ✓ Uses autoregressive generation
#   ✓ Produces exactly 128 whitespace-delimited words
#   ✓ Does NOT include the original prompt in prediction
#   ✓ Does NOT train
#   ✓ Does NOT call backward()
#   ✓ Does NOT call optimizer.step()
#   ✓ Does NOT modify the checkpoint
#   ✓ Validates the final prediction file
#
# IMPORTANT:
#   This generates the competition submission.
#   It cannot calculate the official test score because the
#   reference continuations are private.
#
# ============================================================

import os
import json
import gc
import math
import glob
import torch


# ============================================================
# CONFIGURATION
# ============================================================

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

TARGET_WORDS = 128

# Generate more tokens than normally necessary, then trim to
# exactly 128 whitespace-delimited words.
MAX_NEW_TOKENS = 384

TEMPERATURE = 0.8
TOP_P = 0.95
DO_SAMPLE = True

OUTPUT_PATH = (
    "/kaggle/working/phase1_step1000_prediction.json"
)


# ============================================================
# HEADER
# ============================================================

print("=" * 78)
print("PHASE 1 — STEP-1000 OFFICIAL TEST GENERATION")
print("=" * 78)

print("Device:", DEVICE)

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(DEVICE)
    )
    print(
        "VRAM:",
        f"{torch.cuda.get_device_properties(DEVICE).total_memory / 1024**3:.2f} GB"
    )


# ============================================================
# PRECONDITIONS
# ============================================================

if "model" not in globals():
    raise RuntimeError(
        "model is not defined.\n"
        "Run the Step-1000 model-loading cell first."
    )

if "tokenizer" not in globals():
    raise RuntimeError(
        "tokenizer is not defined.\n"
        "Run the Phase-1 tokenizer/model setup cell first."
    )

print("✓ model exists")
print("✓ tokenizer exists")


# ============================================================
# FIND STEP-1000 CHECKPOINT
# ============================================================

print()
print("=" * 78)
print("LOCATING STEP-1000 CHECKPOINT")
print("=" * 78)

checkpoint_candidates = []

search_roots = [
    "/kaggle/input",
    "/kaggle/working",
]

for root in search_roots:

    if not os.path.exists(root):
        continue

    for pattern in [
        "checkpoint_step_001000.pt",
        "checkpoint_final.pt",
        "checkpoint_latest.pt",
    ]:

        checkpoint_candidates.extend(
            glob.glob(
                os.path.join(
                    root,
                    "**",
                    pattern,
                ),
                recursive=True,
            )
        )


# Remove duplicates while preserving order.
checkpoint_candidates = list(
    dict.fromkeys(checkpoint_candidates)
)

if not checkpoint_candidates:
    raise FileNotFoundError(
        "Could not find a Step-1000 checkpoint under "
        "/kaggle/input or /kaggle/working."
    )


# Prefer exact Step-1000 checkpoint.
exact = [
    p
    for p in checkpoint_candidates
    if os.path.basename(p)
    == "checkpoint_step_001000.pt"
]

if exact:
    checkpoint_path = exact[0]
else:
    checkpoint_path = checkpoint_candidates[0]

print("Checkpoint:")
print(checkpoint_path)


# ============================================================
# LOAD CHECKPOINT
# ============================================================

print()
print("=" * 78)
print("LOADING STEP-1000 CHECKPOINT")
print("=" * 78)

# This checkpoint was created by our own training pipeline and is
# therefore treated as trusted. PyTorch >=2.6 otherwise defaults
# to weights_only=True and can reject pathlib.PosixPath etc.

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False,
)

print("✓ Checkpoint loaded")


# ============================================================
# RESTORE MODEL STATE
# ============================================================

print()
print("=" * 78)
print("RESTORING MODEL")
print("=" * 78)

state_dict = None

if isinstance(checkpoint, dict):

    possible_state_keys = [
        "model_state_dict",
        "model_state",
        "state_dict",
        "model",
    ]

    for key in possible_state_keys:

        if key in checkpoint:

            candidate = checkpoint[key]

            if isinstance(candidate, dict):
                state_dict = candidate
                print(
                    f"✓ Found model state under: {key}"
                )
                break

if state_dict is None:

    # Sometimes the checkpoint itself is a state dict.
    if isinstance(checkpoint, dict):

        tensor_values = [
            v
            for v in checkpoint.values()
            if torch.is_tensor(v)
        ]

        if tensor_values:
            state_dict = checkpoint
            print(
                "✓ Checkpoint itself appears to be a state dict"
            )

if state_dict is None:

    raise RuntimeError(
        "Could not locate a model state_dict inside the "
        "Step-1000 checkpoint.\n"
        f"Checkpoint keys: "
        f"{list(checkpoint.keys())[:50] if isinstance(checkpoint, dict) else type(checkpoint)}"
    )


# Strip possible DataParallel prefix.
clean_state_dict = {}

for key, value in state_dict.items():

    if key.startswith("module."):
        key = key[len("module."):]

    clean_state_dict[key] = value


load_result = model.load_state_dict(
    clean_state_dict,
    strict=False,
)

print(
    "Missing keys:",
    len(load_result.missing_keys)
)

print(
    "Unexpected keys:",
    len(load_result.unexpected_keys)
)

if load_result.missing_keys:
    print(
        "WARNING — first missing keys:",
        load_result.missing_keys[:10]
    )

if load_result.unexpected_keys:
    print(
        "WARNING — first unexpected keys:",
        load_result.unexpected_keys[:10]
    )

model = model.to(DEVICE)
model.eval()

if hasattr(model, "config"):

    if hasattr(model.config, "use_cache"):
        model.config.use_cache = True

print("✓ Model restored")
print("✓ Model moved to", DEVICE)
print("✓ Evaluation mode enabled")


# ============================================================
# LOCATE PHASE-1 TEST JSON
# ============================================================

print()
print("=" * 78)
print("LOCATING PHASE-1 TEST JSON")
print("=" * 78)

json_candidates = []

for root in search_roots:

    if not os.path.exists(root):
        continue

    for path in glob.glob(
        os.path.join(
            root,
            "**",
            "*.json",
        ),
        recursive=True,
    ):

        basename = os.path.basename(path).lower()
        normalized = path.lower().replace("\\", "/")

        # Strong candidates for Phase-1 test file.
        if (
            "phase1" in basename
            or "phase1" in normalized
            or "phase_1" in normalized
        ) and "test" in basename:

            json_candidates.append(path)


# Also look for common exact names.
for root in search_roots:

    if not os.path.exists(root):
        continue

    for name in [
        "Phase1-Test.json",
        "phase1-test.json",
        "test.json",
    ]:

        json_candidates.extend(
            glob.glob(
                os.path.join(
                    root,
                    "**",
                    name,
                ),
                recursive=True,
            )
        )


json_candidates = list(
    dict.fromkeys(json_candidates)
)

if not json_candidates:

    raise FileNotFoundError(
        "Could not find the Phase-1 test JSON under "
        "/kaggle/input or /kaggle/working."
    )


# Prefer a path explicitly containing Phase_1_Pretraining/Test.
preferred = [
    p
    for p in json_candidates
    if (
        "phase_1_pretraining" in p.lower()
        and "/test/" in p.lower().replace("\\", "/")
    )
]

if preferred:
    test_json_path = preferred[0]
else:
    test_json_path = json_candidates[0]

print("Test JSON:")
print(test_json_path)


# ============================================================
# LOAD TEST JSON
# ============================================================

print()
print("=" * 78)
print("LOADING TEST PROMPTS")
print("=" * 78)

with open(
    test_json_path,
    "r",
    encoding="utf-8",
) as f:

    test_data = json.load(f)


if not isinstance(test_data, dict):
    raise RuntimeError(
        "Expected Phase-1 test JSON to be an object/dictionary."
    )

print(
    "Test examples:",
    len(test_data)
)


# ============================================================
# VALIDATE TEST STRUCTURE
# ============================================================

print()
print("=" * 78)
print("VALIDATING TEST STRUCTURE")
print("=" * 78)

valid_entries = {}

for example_id, record in test_data.items():

    if not isinstance(record, dict):
        raise RuntimeError(
            f"Test entry {example_id} is not an object."
        )

    if "text" not in record:
        raise RuntimeError(
            f"Test entry {example_id} has no 'text' field."
        )

    text = record["text"]

    if not isinstance(text, str):
        raise RuntimeError(
            f"Test entry {example_id} text is not a string."
        )

    text = text.strip()

    if not text:
        raise RuntimeError(
            f"Test entry {example_id} has empty text."
        )

    valid_entries[str(example_id)] = text


print(
    f"✓ Valid test prompts: {len(valid_entries):,}"
)

# Show the first few prompt sizes.
for i, (example_id, text) in enumerate(
    valid_entries.items()
):

    if i >= 5:
        break

    print(
        f"  {example_id}: "
        f"{len(text.split()):,} words | "
        f"{len(text):,} characters"
    )


# ============================================================
# TOKENIZER ADAPTER
# ============================================================

print()
print("=" * 78)
print("CONFIGURING TOKENIZER")
print("=" * 78)


def encode_text(text):

    # HuggingFace Transformers tokenizer
    if callable(tokenizer):

        encoded = tokenizer(
            text,
            return_tensors="pt",
            truncation=False,
        )

        input_ids = encoded["input_ids"]

        attention_mask = encoded.get(
            "attention_mask"
        )

        return input_ids, attention_mask

    # HuggingFace tokenizers.Tokenizer
    if hasattr(tokenizer, "encode"):

        encoded = tokenizer.encode(text)

        ids = encoded.ids

        input_ids = torch.tensor(
            [ids],
            dtype=torch.long,
        )

        attention_mask = torch.ones_like(
            input_ids
        )

        return input_ids, attention_mask

    raise RuntimeError(
        "Unsupported tokenizer type: "
        f"{type(tokenizer)}"
    )


def decode_ids(ids):

    # HuggingFace Transformers tokenizer
    if callable(tokenizer):

        return tokenizer.decode(
            ids,
            skip_special_tokens=True,
        )

    # tokenizers.Tokenizer
    if hasattr(tokenizer, "decode"):

        return tokenizer.decode(
            ids,
            skip_special_tokens=True,
        )

    raise RuntimeError(
        "Unsupported tokenizer type: "
        f"{type(tokenizer)}"
    )


print(
    "Tokenizer type:",
    type(tokenizer)
)

print("✓ Tokenizer adapter ready")


# ============================================================
# GENERATION HELPERS
# ============================================================

def clean_generated_text(
    generated_text,
    prompt_text,
):
    """
    Remove accidental prompt echo and produce exactly
    TARGET_WORDS whitespace-delimited words.
    """

    generated_text = (
        generated_text
        .replace("\x00", " ")
        .strip()
    )

    prompt_text = prompt_text.strip()

    # --------------------------------------------------------
    # If the decoder returned the complete prompt + continuation,
    # remove the prompt prefix.
    # --------------------------------------------------------

    if generated_text.startswith(prompt_text):

        generated_text = (
            generated_text[len(prompt_text):]
            .strip()
        )

    # --------------------------------------------------------
    # Defensive removal if the model repeats the beginning.
    # --------------------------------------------------------

    if (
        prompt_text
        and prompt_text in generated_text
    ):

        remainder = generated_text.split(
            prompt_text,
            1,
        )[-1].strip()

        if remainder:
            generated_text = remainder

    # --------------------------------------------------------
    # Normalize whitespace.
    # --------------------------------------------------------

    words = generated_text.split()

    # Exactly 128 words as required by Phase 1.
    words = words[:TARGET_WORDS]

    return " ".join(words)


def get_eos_id():

    # Transformers tokenizer
    if callable(tokenizer):

        return getattr(
            tokenizer,
            "eos_token_id",
            None,
        )

    # tokenizers.Tokenizer
    if hasattr(tokenizer, "token_to_id"):

        for token in [
            "</s>",
            "<|endoftext|>",
            "<eos>",
        ]:

            try:
                token_id = tokenizer.token_to_id(
                    token
                )

                if token_id is not None:
                    return token_id

            except Exception:
                pass

    return None


EOS_ID = get_eos_id()

print(
    "EOS token ID:",
    EOS_ID
)


# ============================================================
# TEST GENERATION FUNCTION
# ============================================================

def generate_continuation(prompt_text):

    input_ids, attention_mask = encode_text(
        prompt_text
    )

    input_ids = input_ids.to(DEVICE)

    if attention_mask is not None:
        attention_mask = attention_mask.to(
            DEVICE
        )

    # --------------------------------------------------------
    # Safety check.
    # --------------------------------------------------------

    if input_ids.ndim != 2:
        raise RuntimeError(
            f"Unexpected input shape: "
            f"{tuple(input_ids.shape)}"
        )

    # --------------------------------------------------------
    # Generate.
    # --------------------------------------------------------

    generation_kwargs = {
        "max_new_tokens": MAX_NEW_TOKENS,
        "do_sample": DO_SAMPLE,
        "temperature": TEMPERATURE,
        "top_p": TOP_P,
    }

    if EOS_ID is not None:
        generation_kwargs["eos_token_id"] = EOS_ID

    # Prevent an accidental max_length restriction from
    # truncating the 512-word prompt.
    generation_kwargs["use_cache"] = True

    with torch.inference_mode():

        generated = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            **generation_kwargs,
        )

    # --------------------------------------------------------
    # IMPORTANT:
    # Only decode the NEW tokens.
    # This guarantees that the prediction does not contain
    # the original prompt.
    # --------------------------------------------------------

    prompt_token_count = input_ids.shape[-1]

    new_token_ids = generated[
        0,
        prompt_token_count:,
    ]

    generated_text = decode_ids(
        new_token_ids.tolist()
    )

    continuation = clean_generated_text(
        generated_text,
        prompt_text,
    )

    return continuation


# ============================================================
# SINGLE-SAMPLE SANITY TEST
# ============================================================

print()
print("=" * 78)
print("RUNNING GENERATION SANITY TEST")
print("=" * 78)

first_id = next(
    iter(valid_entries)
)

first_prompt = valid_entries[first_id]

sample_prediction = generate_continuation(
    first_prompt
)

sample_words = sample_prediction.split()

print(
    "Example ID:",
    first_id
)

print(
    "Generated words:",
    len(sample_words)
)

print()
print("SAMPLE CONTINUATION")
print("-" * 78)
print(sample_prediction)
print("-" * 78)

if len(sample_words) != TARGET_WORDS:

    raise RuntimeError(
        "Generation sanity test did not produce exactly "
        f"{TARGET_WORDS} words.\n"
        f"Produced: {len(sample_words)}"
    )

if not sample_prediction.strip():

    raise RuntimeError(
        "Generation produced an empty continuation."
    )

print(
    f"✓ Sanity test produced exactly {TARGET_WORDS} words"
)


# ============================================================
# FULL TEST GENERATION
# ============================================================

print()
print("=" * 78)
print("GENERATING ALL PHASE-1 TEST PREDICTIONS")
print("=" * 78)

predictions = {}

total = len(valid_entries)

for index, (
    example_id,
    prompt_text,
) in enumerate(
    valid_entries.items(),
    start=1,
):

    continuation = generate_continuation(
        prompt_text
    )

    word_count = len(
        continuation.split()
    )

    # --------------------------------------------------------
    # HARD VALIDATION
    # --------------------------------------------------------

    if word_count != TARGET_WORDS:

        raise RuntimeError(
            f"Example {example_id} produced "
            f"{word_count} words instead of "
            f"{TARGET_WORDS}."
        )

    if not continuation.strip():

        raise RuntimeError(
            f"Example {example_id} produced empty output."
        )

    predictions[example_id] = {
        "text": continuation
    }

    if (
        index == 1
        or index % 10 == 0
        or index == total
    ):

        print(
            f"[{index:>5}/{total}] "
            f"ID={example_id} "
            f"| words={word_count}"
        )

    # Periodic CUDA cleanup without doing it every example.
    if index % 50 == 0:

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()


# ============================================================
# FINAL PREDICTION VALIDATION
# ============================================================

print()
print("=" * 78)
print("VALIDATING COMPLETE PREDICTION SET")
print("=" * 78)

if len(predictions) != len(valid_entries):

    raise RuntimeError(
        "Prediction count mismatch.\n"
        f"Expected: {len(valid_entries)}\n"
        f"Got: {len(predictions)}"
    )

missing_ids = (
    set(valid_entries.keys())
    - set(predictions.keys())
)

extra_ids = (
    set(predictions.keys())
    - set(valid_entries.keys())
)

if missing_ids:
    raise RuntimeError(
        f"Missing predictions: {list(missing_ids)[:10]}"
    )

if extra_ids:
    raise RuntimeError(
        f"Unexpected predictions: {list(extra_ids)[:10]}"
    )


bad_lengths = []

for example_id, record in predictions.items():

    text = record["text"]

    if len(text.split()) != TARGET_WORDS:

        bad_lengths.append(
            (
                example_id,
                len(text.split()),
            )
        )

if bad_lengths:

    raise RuntimeError(
        "Some predictions do not contain exactly "
        f"{TARGET_WORDS} words.\n"
        f"First failures: {bad_lengths[:10]}"
    )


print(
    f"✓ Predictions: {len(predictions):,}"
)

print(
    f"✓ Every prediction has exactly "
    f"{TARGET_WORDS} words"
)

print("✓ No missing IDs")
print("✓ No extra IDs")


# ============================================================
# SAVE prediction.json
# ============================================================

print()
print("=" * 78)
print("SAVING prediction.json")
print("=" * 78)

with open(
    OUTPUT_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        predictions,
        f,
        ensure_ascii=False,
        indent=2,
    )


print(
    "Saved:",
    OUTPUT_PATH
)

print(
    "File size:",
    f"{os.path.getsize(OUTPUT_PATH) / 1024**2:.2f} MB"
)


# ============================================================
# RELOAD + FINAL INTEGRITY CHECK
# ============================================================

with open(
    OUTPUT_PATH,
    "r",
    encoding="utf-8",
) as f:

    reloaded = json.load(f)

if reloaded != predictions:

    raise RuntimeError(
        "Saved prediction.json failed integrity check."
    )

print("✓ prediction.json reload/integrity check passed")


# ============================================================
# FINAL SUMMARY
# ============================================================

print()
print("=" * 78)
print("PHASE 1 TEST GENERATION COMPLETE")
print("=" * 78)

print(
    f"Checkpoint : {checkpoint_path}"
)

print(
    f"Test JSON  : {test_json_path}"
)

print(
    f"Test cases : {len(valid_entries):,}"
)

print(
    f"Words each : {TARGET_WORDS}"
)

print(
    f"Predictions: {len(predictions):,}"
)

print(
    f"Output     : {OUTPUT_PATH}"
)

print()
print(
    "✓ Every Phase-1 test prompt has a prediction."
)

print(
    "✓ Every prediction contains exactly 128 words."
)

print(
    "✓ Original prompts are not included in the output."
)

print(
    "✓ No training or checkpoint modification occurred."
)

print()
print(
    "IMPORTANT: The official Phase-1 test score cannot be "
    "computed locally because the reference continuations "
    "are private."
)

print("=" * 78)

In [ ]:
# ============================================================
# PHASE 4 — GENERATION / VALIDATION REQUIREMENT CHECK
# ============================================================
#
# PURPOSE:
#   Verify that the current notebook has everything required
#   for the capstone generation/evaluation stage.
#
#   Supports BOTH:
#       1. transformers.PreTrainedTokenizer*
#       2. tokenizers.Tokenizer
#
#   DOES NOT:
#       - train
#       - backward()
#       - optimizer.step()
#       - modify checkpoint
#       - generate model outputs
#
# ============================================================

import sys
import types
import torch

print("=" * 78)
print("PHASE 4 — GENERATION / VALIDATION REQUIREMENT CHECK")
print("=" * 78)


# ============================================================
# 1. PYTHON / PYTORCH
# ============================================================

print("\n[1] PYTHON / PYTORCH")

print("Python :", sys.version.split()[0])
print("PyTorch:", torch.__version__)

if torch.cuda.is_available():
    print("CUDA   :", torch.version.cuda)
    print("GPU    :", torch.cuda.get_device_name(0))
else:
    print("CUDA   : NOT AVAILABLE")


# ============================================================
# 2. MODEL OBJECT
# ============================================================

print("\n[2] MODEL")

if "model" not in globals():
    raise RuntimeError(
        "MODEL MISSING: variable `model` does not exist."
    )

model = globals()["model"]

print("Type:", type(model))
print("Module:", type(model).__module__)

if not isinstance(model, torch.nn.Module):
    raise RuntimeError(
        "MODEL INVALID: `model` is not a torch.nn.Module."
    )

print("✓ model exists")
print("✓ model is a torch.nn.Module")


# ============================================================
# 3. MODEL FORWARD
# ============================================================

print("\n[3] MODEL FORWARD INTERFACE")

required_model_methods = [
    "forward",
]

for name in required_model_methods:

    if not hasattr(model, name):
        raise RuntimeError(
            f"MODEL MISSING METHOD: {name}"
        )

    print(f"✓ model.{name}()")


# ============================================================
# 4. GENERATION INTERFACE
# ============================================================

print("\n[4] GENERATION INTERFACE")

has_generate = hasattr(model, "generate")

print("model.generate:", has_generate)

if has_generate:
    print("✓ Model exposes `.generate()`")
else:
    print(
        "⚠ Model does not expose `.generate()`."
    )
    print(
        "  This is not automatically fatal for loss evaluation,"
    )
    print(
        "  but generation-based capstone evaluation will require"
    )
    print(
        "  the appropriate generation wrapper/interface."
    )


# ============================================================
# 5. TOKENIZER OBJECT
# ============================================================

print("\n[5] TOKENIZER")

if "tokenizer" not in globals():
    raise RuntimeError(
        "TOKENIZER MISSING: variable `tokenizer` does not exist."
    )

tokenizer = globals()["tokenizer"]

print("Type  :", type(tokenizer))
print("Module:", type(tokenizer).__module__)


# ============================================================
# 6. IDENTIFY TOKENIZER TYPE
# ============================================================

print("\n[6] TOKENIZER INTERFACE")

tokenizer_module = type(tokenizer).__module__

is_transformers_tokenizer = (
    tokenizer_module.startswith("transformers")
)

is_tokenizers_tokenizer = (
    tokenizer_module.startswith("tokenizers")
)

print(
    "Transformers tokenizer:",
    is_transformers_tokenizer
)

print(
    "Tokenizers tokenizer:",
    is_tokenizers_tokenizer
)


# ============================================================
# 7. TOKENIZER CAPABILITIES
# ============================================================

has_encode = hasattr(tokenizer, "encode")
has_decode = hasattr(tokenizer, "decode")

print("\nencode():", has_encode)
print("decode():", has_decode)

if not has_encode:
    raise RuntimeError(
        "Tokenizer has no encode() method."
    )

if not has_decode:
    raise RuntimeError(
        "Tokenizer has no decode() method."
    )

print("✓ encode() available")
print("✓ decode() available")


# ============================================================
# 8. TOKENIZER VOCABULARY
# ============================================================

print("\n[7] TOKENIZER VOCABULARY")

vocab_size = None

# Hugging Face Transformers tokenizer
if hasattr(tokenizer, "vocab_size"):

    try:
        vocab_size = int(tokenizer.vocab_size)
    except Exception:
        vocab_size = None


# Hugging Face tokenizers.Tokenizer
if vocab_size is None and hasattr(tokenizer, "get_vocab_size"):

    try:
        vocab_size = int(
            tokenizer.get_vocab_size()
        )
    except Exception:
        vocab_size = None


if vocab_size is None:

    print(
        "⚠ Could not determine vocabulary size "
        "from tokenizer interface."
    )

else:

    print(
        "Vocabulary size:",
        f"{vocab_size:,}"
    )

    if vocab_size <= 0:
        raise RuntimeError(
            "Tokenizer vocabulary size is invalid."
        )

    print("✓ Vocabulary size valid")


# ============================================================
# 9. SPECIAL TOKEN INFORMATION
# ============================================================

print("\n[8] SPECIAL TOKENS")

special_attrs = [
    "pad_token_id",
    "eos_token_id",
    "bos_token_id",
    "unk_token_id",
]

found_special = 0

for attr in special_attrs:

    if hasattr(tokenizer, attr):

        try:
            value = getattr(tokenizer, attr)
        except Exception:
            value = None

        print(
            f"{attr:15s}: {value}"
        )

        if value is not None:
            found_special += 1

# tokenizers.Tokenizer uses special tokens through its
# internal configuration rather than necessarily exposing
# *_token_id attributes.
if is_tokenizers_tokenizer:

    print(
        "✓ Native `tokenizers.Tokenizer` detected."
    )
    print(
        "  Special-token IDs will be handled through"
        " the tokenizer encoding/configuration."
    )


# ============================================================
# 10. ACTUAL TOKENIZER ROUND-TRIP TEST
# ============================================================

print("\n[9] TOKENIZER ROUND-TRIP TEST")

TEST_TEXT = (
    "This is a Phase 4 generation validation test."
)

try:

    encoded = tokenizer.encode(TEST_TEXT)

    print(
        "Encoded object:",
        type(encoded)
    )

except Exception as e:

    raise RuntimeError(
        "Tokenizer encode() failed:\n"
        + repr(e)
    )


# ------------------------------------------------------------
# Extract IDs depending on tokenizer implementation
# ------------------------------------------------------------

token_ids = None

if hasattr(encoded, "ids"):

    # tokenizers.Encoding
    token_ids = encoded.ids

elif isinstance(encoded, (list, tuple)):

    # Some tokenizer APIs return IDs directly
    token_ids = list(encoded)

else:

    # Transformers tokenizer.encode() usually returns list
    try:
        token_ids = list(encoded)
    except Exception:
        token_ids = None


if token_ids is None or len(token_ids) == 0:

    raise RuntimeError(
        "Tokenizer encode() returned no token IDs."
    )

print(
    "Token count:",
    len(token_ids)
)

print(
    "First token IDs:",
    token_ids[:10]
)


# ------------------------------------------------------------
# Decode
# ------------------------------------------------------------

try:

    decoded = tokenizer.decode(
        token_ids
    )

except Exception as e:

    raise RuntimeError(
        "Tokenizer decode() failed:\n"
        + repr(e)
    )


if not isinstance(decoded, str):

    raise RuntimeError(
        "Tokenizer decode() did not return a string."
    )

print(
    "Decoded preview:",
    repr(decoded[:150])
)

print("✓ encode() works")
print("✓ decode() works")
print("✓ tokenizer round-trip interface works")


# ============================================================
# 11. TRANSFORMERS-SPECIFIC CHECK
# ============================================================

print("\n[10] TRANSFORMERS COMPATIBILITY")

if is_transformers_tokenizer:

    print(
        "Detected Transformers tokenizer."
    )

    transformers_methods = [
        "batch_decode",
        "encode",
        "decode",
    ]

    for method in transformers_methods:

        if hasattr(tokenizer, method):
            print(
                f"✓ tokenizer.{method}()"
            )
        else:
            print(
                f"⚠ tokenizer.{method}() unavailable"
            )

else:

    print(
        "Native `tokenizers.Tokenizer` detected."
    )

    print(
        "✓ This is acceptable for tokenization."
    )


# ============================================================
# 12. CONTEXT LENGTH
# ============================================================

print("\n[11] CONTEXT LENGTH")

CONTEXT_LENGTH = 4096

print(
    "Expected context length:",
    CONTEXT_LENGTH
)

model_max_length = None

if hasattr(tokenizer, "model_max_length"):

    try:
        model_max_length = int(
            tokenizer.model_max_length
        )
    except Exception:
        pass

if model_max_length is not None:

    print(
        "Tokenizer model_max_length:",
        model_max_length
    )

    # Hugging Face sometimes uses an enormous sentinel
    # for "no explicit maximum".
    if model_max_length < CONTEXT_LENGTH:

        print(
            "⚠ Tokenizer reports a maximum below 4096."
        )

else:

    print(
        "Tokenizer does not expose model_max_length."
    )

    print(
        "✓ We will explicitly enforce 4096 during evaluation."
    )


# ============================================================
# 13. MODEL CONFIG
# ============================================================

print("\n[12] MODEL CONFIGURATION")

if hasattr(model, "config"):

    config = model.config

    print(
        "Config type:",
        type(config)
    )

    if hasattr(config, "vocab_size"):

        print(
            "Model vocab size:",
            config.vocab_size
        )

        if (
            vocab_size is not None
            and int(config.vocab_size) != int(vocab_size)
        ):

            print(
                "\n⚠ WARNING:"
            )

            print(
                "Model vocabulary size and tokenizer "
                "vocabulary size differ."
            )

            print(
                "Model:",
                config.vocab_size
            )

            print(
                "Tokenizer:",
                vocab_size
            )

    if hasattr(config, "max_position_embeddings"):

        print(
            "Model max positions:",
            config.max_position_embeddings
        )

        if (
            config.max_position_embeddings
            < CONTEXT_LENGTH
        ):

            raise RuntimeError(
                "Model max_position_embeddings is below "
                "the required 4096 context length."
            )

else:

    print(
        "⚠ model.config unavailable."
    )


# ============================================================
# 14. HF TOKEN
# ============================================================

print("\n[13] HUGGING FACE ACCESS")

if "HF_TOKEN" in globals():

    token = globals()["HF_TOKEN"]

    if token is None or not str(token).strip():

        print(
            "⚠ HF_TOKEN exists but is empty."
        )

    else:

        print(
            "✓ HF_TOKEN variable exists."
        )

else:

    print(
        "⚠ HF_TOKEN is not defined."
    )

    print(
        "Only required if the evaluation dataset/model"
        " must be downloaded from a private HF repository."
    )


# ============================================================
# 15. DEVICE TEST
# ============================================================

print("\n[14] DEVICE")

if torch.cuda.is_available():

    DEVICE = torch.device("cuda")

    print(
        "✓ CUDA available"
    )

    print(
        "Device:",
        torch.cuda.get_device_name(DEVICE)
    )

else:

    DEVICE = torch.device("cpu")

    print(
        "⚠ CUDA unavailable."
    )


# ============================================================
# 16. MODEL PARAMETER SANITY
# ============================================================

print("\n[15] MODEL PARAMETER SANITY")

parameter_count = 0
nonfinite_parameters = 0

for name, param in model.named_parameters():

    parameter_count += param.numel()

    if not torch.isfinite(
        param.detach()
    ).all():

        nonfinite_parameters += 1

print(
    "Parameters:",
    f"{parameter_count:,}"
)

print(
    "Non-finite parameter tensors:",
    nonfinite_parameters
)

if nonfinite_parameters != 0:

    raise RuntimeError(
        "MODEL CONTAINS NON-FINITE PARAMETERS."
    )

print(
    "✓ All model parameters finite"
)


# ============================================================
# 17. FINAL REQUIREMENT SUMMARY
# ============================================================

print()
print("=" * 78)
print("REQUIREMENT SUMMARY")
print("=" * 78)

checks = {

    "model exists":
        "model" in globals(),

    "model is torch module":
        isinstance(model, torch.nn.Module),

    "model forward":
        hasattr(model, "forward"),

    "tokenizer exists":
        "tokenizer" in globals(),

    "tokenizer encode":
        has_encode,

    "tokenizer decode":
        has_decode,

    "tokenizer round-trip":
        token_ids is not None
        and len(token_ids) > 0
        and isinstance(decoded, str),

    "model parameters finite":
        nonfinite_parameters == 0,

    "CUDA available":
        torch.cuda.is_available(),

}

all_required = True

for name, passed in checks.items():

    print(
        f"{'✓' if passed else '✗'} {name}"
    )

    if not passed:
        all_required = False


print()
print("=" * 78)

if all_required:

    print(
        "✓ ALL CORE GENERATION / VALIDATION REQUIREMENTS PASSED"
    )

    print()
    print(
        "The tokenizer interface is usable."
    )

    print(
        "The model interface is usable."
    )

    print(
        "The next cell can perform the actual capstone"
        " generation/evaluation."
    )

else:

    print(
        "⚠ SOME REQUIREMENTS NEED ATTENTION."
    )

print("=" * 78)

In [ ]:
# ============================================================
# MOVE MODEL TO GPU — SAFE VERSION
# ============================================================

print()
print("=" * 80)
print("MOVING MODEL TO GPU")
print("=" * 80)

model = model.to(DEVICE)

# Verify using an actual parameter tensor.
first_param = next(model.parameters())

print("Requested device :", DEVICE)
print("Actual device    :", first_param.device)
print("Model dtype      :", first_param.dtype)

if first_param.device.type != "cuda":
    raise RuntimeError(
        f"Model did not move to CUDA. "
        f"Actual device: {first_param.device}"
    )

model.eval()

if hasattr(model.config, "use_cache"):
    model.config.use_cache = False

print()
print("✓ Model is actually on CUDA")
print("✓ Evaluation mode enabled")
print("✓ use_cache=False")

print("=" * 80)

In [ ]:
# ============================================================
# CS781 — VALIDATION NOTEBOOK
# CELL 1 — PORT EXACT PROJECT TOKENIZER + MODEL ARCHITECTURE
# ============================================================

import os
import json
import gc
import math
from pathlib import Path

import torch
from tokenizers import Tokenizer
from transformers import GPT2Config, GPT2LMHeadModel


print("=" * 80)
print("CS781 — PORTING EXACT PROJECT MODEL COMPONENTS")
print("=" * 80)


# ============================================================
# CONFIGURATION
# ============================================================

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

CONTEXT_LENGTH = 4096
VOCAB_SIZE = 32000

RECOVERY_ROOT = Path(
    "/kaggle/input/datasets/"
    "oipranesh/"
    "cs781-phase4-final-recovery"
)

# The notebook's persistent recovery package contains:
#
# tokenizer/legal_bpe_32k.json
#
TOKENIZER_PATH = (
    RECOVERY_ROOT
    / "tokenizer"
    / "legal_bpe_32k.json"
)


# ============================================================
# BASIC CHECKS
# ============================================================

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is required for the validation run."
    )

if not RECOVERY_ROOT.exists():
    raise FileNotFoundError(
        f"Persistent recovery dataset not found:\n"
        f"{RECOVERY_ROOT}"
    )

if not TOKENIZER_PATH.exists():
    raise FileNotFoundError(
        f"Project tokenizer not found:\n"
        f"{TOKENIZER_PATH}"
    )


print("Device :", DEVICE)
print(
    "GPU    :",
    torch.cuda.get_device_name(DEVICE)
)

print()
print("Recovery package:")
print(RECOVERY_ROOT)

print()
print("Tokenizer:")
print(TOKENIZER_PATH)


# ============================================================
# LOAD EXACT PROJECT TOKENIZER
# ============================================================

print()
print("=" * 80)
print("LOADING PROJECT BPE TOKENIZER")
print("=" * 80)

tokenizer = Tokenizer.from_file(
    str(TOKENIZER_PATH)
)

print(
    "Vocabulary size:",
    tokenizer.get_vocab_size()
)

assert (
    tokenizer.get_vocab_size()
    == VOCAB_SIZE
)


# ============================================================
# SPECIAL TOKENS
# ============================================================

pad_token_id = tokenizer.token_to_id("[PAD]")
unk_token_id = tokenizer.token_to_id("[UNK]")
bos_token_id = tokenizer.token_to_id("[BOS]")
eos_token_id = tokenizer.token_to_id("[EOS]")


print()
print("SPECIAL TOKENS")
print("-" * 80)

print("PAD :", pad_token_id)
print("UNK :", unk_token_id)
print("BOS :", bos_token_id)
print("EOS :", eos_token_id)


assert pad_token_id is not None
assert unk_token_id is not None
assert bos_token_id is not None
assert eos_token_id is not None


# ============================================================
# EXACT MODEL CONFIG FROM TRAINING NOTEBOOK
# ============================================================

print()
print("=" * 80)
print("BUILDING EXACT PROJECT MODEL ARCHITECTURE")
print("=" * 80)


hf_config = GPT2Config(

    vocab_size=VOCAB_SIZE,

    n_positions=CONTEXT_LENGTH,
    n_ctx=CONTEXT_LENGTH,

    n_embd=768,
    n_layer=12,
    n_head=12,

    n_inner=3072,

    resid_pdrop=0.0,
    embd_pdrop=0.0,
    attn_pdrop=0.0,

    bos_token_id=bos_token_id,
    eos_token_id=eos_token_id,
    pad_token_id=pad_token_id,

    use_cache=False,
)


model = GPT2LMHeadModel(
    hf_config
)


# ============================================================
# MODEL SUMMARY
# ============================================================

total_params = sum(
    p.numel()
    for p in model.parameters()
)

print()
print("MODEL")
print("-" * 80)

print("Vocabulary       :", VOCAB_SIZE)
print("Context          :", CONTEXT_LENGTH)
print("Hidden size      :", 768)
print("Layers           :", 12)
print("Attention heads  :", 12)
print("Intermediate     :", 3072)

print(
    "Parameters       :",
    f"{total_params:,}"
)


# ============================================================
# MOVE TO GPU
# ============================================================

model = model.to(DEVICE)

model.eval()

if hasattr(model.config, "use_cache"):
    model.config.use_cache = False


# ============================================================
# FINAL ARCHITECTURE ASSERTIONS
# ============================================================

assert model.config.vocab_size == 32000
assert model.config.n_positions == 4096
assert model.config.n_embd == 768
assert model.config.n_layer == 12
assert model.config.n_head == 12
assert model.config.n_inner == 3072

assert next(model.parameters()).device.type == "cuda"


print()
print("=" * 80)
print("✓ TOKENIZER LOADED")
print("✓ VOCABULARY = 32,000")
print("✓ SPECIAL TOKENS VERIFIED")
print("✓ GPT-2 ARCHITECTURE MATCHES TRAINING NOTEBOOK")
print("✓ CONTEXT LENGTH = 4,096")
print("✓ MODEL ON GPU")
print("=" * 80)

In [ ]:
# ============================================================
# CS781 — VALIDATION NOTEBOOK
# CELL 2 — LOAD VERIFIED STEP-1000 CHECKPOINT
# ============================================================

import os
import torch
from pathlib import Path


print("=" * 80)
print("CS781 — LOADING VERIFIED STEP-1000 CHECKPOINT")
print("=" * 80)


# ============================================================
# LOCATE CHECKPOINT
# ============================================================

CANDIDATES = [

    Path(
        "/kaggle/input/datasets/"
        "oipranesh/phase4-step1000-verified-model/"
        "checkpoint_step_001000.pt"
    ),

    Path(
        "/kaggle/input/datasets/"
        "oipranesh/phase4-step1000-verified-model/"
        "checkpoint_final.pt"
    ),

    Path(
        "/kaggle/input/datasets/"
        "oipranesh/phase4-step1000-verified-model/"
        "checkpoint_latest.pt"
    ),
]


checkpoint_path = None

for candidate in CANDIDATES:

    if candidate.exists():

        checkpoint_path = candidate

        print(
            "✓ Found:",
            candidate
        )

        break


if checkpoint_path is None:

    raise FileNotFoundError(
        "No Step-1000 checkpoint found.\n\n"
        "Checked:\n"
        + "\n".join(
            str(x)
            for x in CANDIDATES
        )
    )


# ============================================================
# LOAD TRUSTED CHECKPOINT
# ============================================================

print()
print("Loading checkpoint...")

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False,
)


print(
    "Checkpoint object:",
    type(checkpoint)
)


# ============================================================
# INSPECT CHECKPOINT
# ============================================================

if isinstance(checkpoint, dict):

    print()
    print("Checkpoint keys:")
    for key in checkpoint.keys():
        print("  ", key)


# ============================================================
# FIND MODEL STATE
# ============================================================

if isinstance(checkpoint, dict):

    possible_keys = [
        "model_state_dict",
        "model",
        "state_dict",
    ]

    model_state = None

    for key in possible_keys:

        if key in checkpoint:

            model_state = checkpoint[key]

            print(
                "\n✓ Model state found under:",
                key
            )

            break

    if model_state is None:

        # Some project checkpoints may themselves
        # be the state dict.

        if all(
            isinstance(k, str)
            for k in checkpoint.keys()
        ) and any(
            k.startswith("transformer.")
            or k.startswith("lm_head.")
            for k in checkpoint.keys()
        ):

            model_state = checkpoint

            print(
                "\n✓ Checkpoint itself is the model state dict"
            )


else:

    model_state = checkpoint


if model_state is None:

    raise RuntimeError(
        "Could not locate model state dictionary."
    )


# ============================================================
# RESTORE MODEL
# ============================================================

missing, unexpected = model.load_state_dict(
    model_state,
    strict=True,
)


print()
print("Missing keys    :", len(missing))
print("Unexpected keys :", len(unexpected))


assert len(missing) == 0
assert len(unexpected) == 0


# ============================================================
# FINITE PARAMETER CHECK
# ============================================================

nonfinite = 0

for name, param in model.named_parameters():

    if not torch.isfinite(param).all():

        nonfinite += 1

        print(
            "NON-FINITE PARAMETER:",
            name
        )


assert nonfinite == 0


# ============================================================
# FINAL MODEL STATE
# ============================================================

model = model.to(DEVICE)
model.eval()

if hasattr(model.config, "use_cache"):
    model.config.use_cache = False


print()
print("=" * 80)
print("✓ STEP-1000 CHECKPOINT LOADED")
print("✓ STATE DICT MATCHES EXACT ARCHITECTURE")
print("✓ NO MISSING PARAMETERS")
print("✓ NO UNEXPECTED PARAMETERS")
print("✓ ALL PARAMETERS FINITE")
print("✓ MODEL IN EVALUATION MODE")
print("=" * 80)

In [ ]:
# ============================================================
# CS781 — CELL 3
# TOKENIZER → MODEL END-TO-END SANITY TEST
# ============================================================

import torch


print("=" * 80)
print("TOKENIZER → MODEL SANITY TEST")
print("=" * 80)


TEXT = (
    "The Supreme Court considered the constitutional validity "
    "of the challenged provision."
)


# ------------------------------------------------------------
# EXACT tokenizers.Tokenizer API
# ------------------------------------------------------------

encoding = tokenizer.encode(TEXT)

token_ids = encoding.ids

print("Characters :", len(TEXT))
print("Tokens     :", len(token_ids))
print("First IDs  :", token_ids[:20])


# ------------------------------------------------------------
# Add project document boundaries
# ------------------------------------------------------------

model_ids = (
    [bos_token_id]
    + token_ids
    + [eos_token_id]
)


# Keep within model context.
model_ids = model_ids[:CONTEXT_LENGTH]


input_ids = torch.tensor(
    [model_ids],
    dtype=torch.long,
    device=DEVICE,
)

attention_mask = torch.ones_like(
    input_ids,
    device=DEVICE,
)


# ------------------------------------------------------------
# Forward
# ------------------------------------------------------------

with torch.inference_mode():

    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        use_cache=False,
    )


logits = outputs.logits


# ------------------------------------------------------------
# Assertions
# ------------------------------------------------------------

print()
print("Input shape :", tuple(input_ids.shape))
print("Logits shape:", tuple(logits.shape))

assert input_ids.ndim == 2
assert input_ids.size(1) <= CONTEXT_LENGTH

assert logits.shape[0] == 1
assert logits.shape[1] == input_ids.shape[1]
assert logits.shape[2] == VOCAB_SIZE

assert torch.isfinite(logits).all()


print()
print("=" * 80)
print("✓ RAW BPE TOKENIZER WORKS")
print("✓ BOS/EOS INSERTION WORKS")
print("✓ TOKEN IDS ARE VALID")
print("✓ MODEL FORWARD WORKS")
print("✓ LOGITS ARE FINITE")
print("=" * 80)

In [ ]:
# ============================================================
# CS781 — TEST SET STRUCTURE + 128-SAMPLE READINESS CHECK
# ============================================================

import json
from pathlib import Path

print("=" * 80)
print("CS781 — HELD-OUT TEST SET STRUCTURE CHECK")
print("=" * 80)

# Search Kaggle input for test.json
matches = list(
    Path("/kaggle/input").rglob("test.json")
)

if not matches:
    raise FileNotFoundError(
        "Could not find test.json under /kaggle/input"
    )

print("\nCandidate test.json files:")
for i, p in enumerate(matches):
    print(f"  [{i}] {p}")

TEST_JSON = matches[0]

print("\nUsing:")
print(TEST_JSON)


# ------------------------------------------------------------
# READ RAW JSON
# ------------------------------------------------------------

with open(
    TEST_JSON,
    "r",
    encoding="utf-8"
) as f:
    raw_test = json.load(f)


print("\nRAW JSON TYPE:")
print(type(raw_test))

if isinstance(raw_test, dict):

    print("\nTOP-LEVEL KEYS:")
    print(
        list(raw_test.keys())[:30]
    )

elif isinstance(raw_test, list):

    print(
        "\nNUMBER OF TOP-LEVEL ITEMS:",
        len(raw_test)
    )


# ------------------------------------------------------------
# INSPECT WITHOUT ASSUMING A SCHEMA
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("STRUCTURE INSPECTION")
print("=" * 80)


if isinstance(raw_test, dict):

    print(
        "Top-level object contains",
        len(raw_test),
        "keys."
    )

    first_items = list(
        raw_test.items()
    )[:5]

    for key, value in first_items:

        print("\nKEY:", repr(key))
        print("VALUE TYPE:", type(value))

        if isinstance(value, str):

            print(
                "TEXT LENGTH:",
                len(value)
            )

            print(
                "TEXT PREVIEW:",
                repr(value[:300])
            )

        elif isinstance(value, dict):

            print(
                "DICT KEYS:",
                list(value.keys())[:20]
            )

        elif isinstance(value, list):

            print(
                "LIST LENGTH:",
                len(value)
            )

            if value:
                print(
                    "FIRST ITEM TYPE:",
                    type(value[0])
                )

                print(
                    "FIRST ITEM PREVIEW:",
                    repr(value[0])[:500]
                )


elif isinstance(raw_test, list):

    for i, item in enumerate(raw_test[:5]):

        print("\nITEM", i + 1)
        print("TYPE:", type(item))
        print(
            "PREVIEW:",
            repr(item)[:1000]
        )


# ------------------------------------------------------------
# DETERMINE WHETHER 128 TEXTS CAN BE EXTRACTED
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("128-TEXT EXTRACTION CHECK")
print("=" * 80)


candidate_texts = []


def collect_strings(obj):

    if isinstance(obj, str):

        text = obj.strip()

        if len(text) >= 50:
            candidate_texts.append(text)

        return

    if isinstance(obj, list):

        for item in obj:
            collect_strings(item)

        return

    if isinstance(obj, dict):

        for value in obj.values():
            collect_strings(value)


collect_strings(raw_test)


print(
    "Candidate text strings found:",
    len(candidate_texts)
)


if candidate_texts:

    lengths = [
        len(x)
        for x in candidate_texts
    ]

    print(
        "Minimum text length:",
        min(lengths)
    )

    print(
        "Maximum text length:",
        max(lengths)
    )

    print(
        "First candidate preview:"
    )

    print(
        candidate_texts[0][:500]
    )


if len(candidate_texts) < 128:

    raise RuntimeError(
        "The raw test.json does not expose "
        "128 extractable text documents using "
        "the generic recursive inspection.\n\n"
        "This cell intentionally stopped before "
        "running model evaluation."
    )


print()
print("=" * 80)
print("✓ AT LEAST 128 TEXT CANDIDATES FOUND")
print("✓ NO MODEL MODIFICATION")
print("✓ NO BACKWARD")
print("✓ NO OPTIMIZER")
print("✓ SAFE TO PROCEED TO FINAL TEST EVALUATION")
print("=" * 80)

In [ ]:
# =============================================================================
# CS781 — FINAL PRE-PREDICTION SANITY CHECK
# =============================================================================
#
# Purpose:
#   Final read-only verification immediately before the 128-test-text
#   generation/evaluation cell.
#
# This cell:
#   ✓ does NOT train
#   ✓ does NOT modify model weights
#   ✓ does NOT call backward()
#   ✓ does NOT call optimizer.step()
#   ✓ verifies checkpoint/model/tokenizer compatibility
#   ✓ verifies model device correctly as cuda:0
#   ✓ verifies a real test example can be tokenized
#   ✓ verifies forward pass + logits are finite
#   ✓ verifies generation prerequisites
#
# =============================================================================

import gc
import math
import torch

print("=" * 80)
print("CS781 — FINAL PRE-PREDICTION SANITY CHECK")
print("=" * 80)

# -----------------------------------------------------------------------------
# 1. CUDA
# -----------------------------------------------------------------------------

assert torch.cuda.is_available(), "CUDA is required."

DEVICE = torch.device("cuda:0")

print()
print("DEVICE CHECK")
print("-" * 80)
print("Requested device :", DEVICE)
print("GPU              :", torch.cuda.get_device_name(0))

# -----------------------------------------------------------------------------
# 2. REQUIRED OBJECTS
# -----------------------------------------------------------------------------

required_objects = [
    "model",
    "tokenizer",
]

for name in required_objects:
    if name not in globals():
        raise RuntimeError(
            f"Required object '{name}' is missing."
        )

print("✓ model exists")
print("✓ tokenizer exists")

# -----------------------------------------------------------------------------
# 3. MODEL DEVICE
# -----------------------------------------------------------------------------

model_device = next(model.parameters()).device

print()
print("MODEL DEVICE")
print("-" * 80)
print("Model device     :", model_device)
print("Expected device  :", DEVICE)

# IMPORTANT:
# Compare device type/index explicitly rather than:
#     model_device != torch.device("cuda")
#
# cuda == cuda:0 semantically for this notebook, but the objects can
# compare differently depending on how DEVICE was constructed.

if model_device.type != "cuda":
    raise RuntimeError(
        f"Model is not on CUDA. Actual device: {model_device}"
    )

if model_device.index not in (None, 0):
    raise RuntimeError(
        f"Model is on unexpected GPU: {model_device}"
    )

print("✓ Model is on CUDA GPU 0")

# -----------------------------------------------------------------------------
# 4. EVAL MODE
# -----------------------------------------------------------------------------

if model.training:
    print("Model was in training mode; switching to evaluation mode.")
    model.eval()

assert not model.training

print("✓ Model is in evaluation mode")

# -----------------------------------------------------------------------------
# 5. CACHE / ARCHITECTURE
# -----------------------------------------------------------------------------

if hasattr(model.config, "use_cache"):
    model.config.use_cache = False

print()
print("MODEL CONFIG")
print("-" * 80)

if hasattr(model.config, "vocab_size"):
    print("Vocab size       :", model.config.vocab_size)

if hasattr(model.config, "n_positions"):
    print("n_positions      :", model.config.n_positions)

if hasattr(model.config, "n_embd"):
    print("n_embd           :", model.config.n_embd)

if hasattr(model.config, "n_layer"):
    print("n_layer          :", model.config.n_layer)

if hasattr(model.config, "n_head"):
    print("n_head           :", model.config.n_head)

if hasattr(model.config, "n_inner"):
    print("n_inner          :", model.config.n_inner)

if hasattr(model.config, "use_cache"):
    print("use_cache        :", model.config.use_cache)

# -----------------------------------------------------------------------------
# 6. PARAMETER FINITENESS
# -----------------------------------------------------------------------------

print()
print("PARAMETER FINITENESS")
print("-" * 80)

bad_parameters = []

for name, parameter in model.named_parameters():

    if not torch.isfinite(parameter).all():
        bad_parameters.append(name)

if bad_parameters:
    raise RuntimeError(
        "Non-finite model parameters found:\n"
        + "\n".join(bad_parameters[:20])
    )

print("✓ All model parameters are finite")

# -----------------------------------------------------------------------------
# 7. TOKENIZER SANITY
# -----------------------------------------------------------------------------

print()
print("TOKENIZER CHECK")
print("-" * 80)

sanity_text = (
    "The court considered the matter and issued an order "
    "after examining the relevant provisions of law."
)

# Your notebook uses the RAW BPE tokenizer rather than a HuggingFace
# PreTrainedTokenizer callable, so explicitly use encode().
if not hasattr(tokenizer, "encode"):
    raise RuntimeError(
        "Tokenizer does not expose encode()."
    )

if not hasattr(tokenizer, "decode"):
    raise RuntimeError(
        "Tokenizer does not expose decode()."
    )

encoded = tokenizer.encode(sanity_text)

if hasattr(encoded, "ids"):
    sanity_ids = encoded.ids
else:
    sanity_ids = encoded

if not sanity_ids:
    raise RuntimeError(
        "Tokenizer produced zero tokens."
    )

print("Characters       :", len(sanity_text))
print("Tokens           :", len(sanity_ids))
print("First IDs        :", sanity_ids[:20])

# -----------------------------------------------------------------------------
# 8. TOKEN ID RANGE
# -----------------------------------------------------------------------------

if hasattr(model.config, "vocab_size"):

    vocab_size = int(model.config.vocab_size)

    invalid_ids = [
        int(x)
        for x in sanity_ids
        if int(x) < 0 or int(x) >= vocab_size
    ]

    if invalid_ids:
        raise RuntimeError(
            f"Tokenizer produced IDs outside model vocabulary: "
            f"{invalid_ids[:20]}"
        )

    print("✓ Token IDs inside model vocabulary")

# -----------------------------------------------------------------------------
# 9. CONVERT TO MODEL INPUT
# -----------------------------------------------------------------------------

input_ids = torch.tensor(
    [sanity_ids],
    dtype=torch.long,
    device=DEVICE,
)

attention_mask = torch.ones_like(input_ids)

print()
print("FORWARD INPUT")
print("-" * 80)
print("Input shape      :", tuple(input_ids.shape))
print("Input device     :", input_ids.device)

# -----------------------------------------------------------------------------
# 10. READ-ONLY FORWARD TEST
# -----------------------------------------------------------------------------

print()
print("FORWARD TEST")
print("-" * 80)

with torch.inference_mode():

    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        use_cache=False,
    )

logits = outputs.logits

print("Logits shape     :", tuple(logits.shape))
print("Logits device    :", logits.device)
print("Logits dtype     :", logits.dtype)

if not torch.isfinite(logits).all():
    raise RuntimeError(
        "Forward pass produced non-finite logits."
    )

print("✓ Forward pass successful")
print("✓ Logits are finite")

# -----------------------------------------------------------------------------
# 11. LOGITS/VOCAB COMPATIBILITY
# -----------------------------------------------------------------------------

if hasattr(model.config, "vocab_size"):

    expected_vocab = int(model.config.vocab_size)
    actual_vocab = int(logits.shape[-1])

    print()
    print("VOCABULARY COMPATIBILITY")
    print("-" * 80)
    print("Config vocab     :", expected_vocab)
    print("Logits vocab     :", actual_vocab)

    if actual_vocab != expected_vocab:
        raise RuntimeError(
            f"Vocabulary mismatch: config={expected_vocab}, "
            f"logits={actual_vocab}"
        )

    print("✓ Vocabulary dimensions match")

# -----------------------------------------------------------------------------
# 12. TEST DATA AVAILABILITY
# -----------------------------------------------------------------------------

print()
print("TEST DATA CHECK")
print("-" * 80)

TEST_PATH = (
    "/kaggle/input/datasets/"
    "armankrsingh/test-cs781-json/test.json"
)

import json
import os

if not os.path.exists(TEST_PATH):
    raise FileNotFoundError(
        f"Test file not found:\n{TEST_PATH}"
    )

with open(
    TEST_PATH,
    "r",
    encoding="utf-8",
) as f:

    test_raw = json.load(f)

if not isinstance(test_raw, dict):
    raise RuntimeError(
        f"Expected test.json to contain a top-level dict, "
        f"got {type(test_raw)}"
    )

if len(test_raw) != 1000:
    raise RuntimeError(
        f"Expected 1000 test entries, found {len(test_raw)}"
    )

print("✓ test.json found")
print("✓ Test entries    :", len(test_raw))

# -----------------------------------------------------------------------------
# 13. TEST TEXT STRUCTURE
# -----------------------------------------------------------------------------

first_key = next(iter(test_raw))
first_record = test_raw[first_key]

if not isinstance(first_record, dict):
    raise RuntimeError(
        f"Test entry {first_key!r} is not a dict."
    )

if "text" not in first_record:
    raise RuntimeError(
        f"Test entry {first_key!r} does not contain 'text'. "
        f"Available fields: {list(first_record.keys())}"
    )

first_text = str(first_record["text"]).strip()

if not first_text:
    raise RuntimeError(
        "First test text is empty."
    )

print("✓ Test records contain 'text'")
print("First test key   :", first_key)
print("First text chars :", len(first_text))

# -----------------------------------------------------------------------------
# 14. REAL TEST TEXT TOKENIZATION
# -----------------------------------------------------------------------------

test_encoded = tokenizer.encode(first_text)

if hasattr(test_encoded, "ids"):
    test_ids = test_encoded.ids
else:
    test_ids = test_encoded

if not test_ids:
    raise RuntimeError(
        "First test text produced zero tokens."
    )

print("First test tokens:", len(test_ids))

# -----------------------------------------------------------------------------
# 15. CLEANUP
# -----------------------------------------------------------------------------

del input_ids
del attention_mask
del outputs
del logits

gc.collect()
torch.cuda.empty_cache()

# -----------------------------------------------------------------------------
# FINAL RESULT
# -----------------------------------------------------------------------------

print()
print("=" * 80)
print("✓ FINAL PRE-PREDICTION CHECK PASSED")
print("=" * 80)

print()
print("READY FOR:")
print("  → 128 held-out test-text generation/evaluation")
print("  → generation metrics")
print("  → prediction output")
print("  → final validation report")

print()
print("IMPORTANT:")
print("  This cell performed NO training.")
print("  No weights were modified.")
print("  No optimizer state was touched.")
print("  No backward() was called.")
print("  No optimizer.step() was called.")

print("=" * 80)

In [ ]:
# =============================================================================
# CS781 — STEP-1000 HELD-OUT TEST GENERATION
# =============================================================================
#
# PURPOSE
# -------
# Run the verified Step-1000 model on ALL 1,000 held-out test texts.
#
# For every test record:
#   1. Read the original "text" field.
#   2. Tokenize it.
#   3. Use it as the generation prompt.
#   4. Generate exactly up to 128 NEW tokens.
#   5. Keep the original prompt separate from the generated continuation.
#   6. Save every prediction to JSONL + JSON.
#
# SAFETY
# ------
# ✓ model.eval()
# ✓ inference_mode()
# ✓ NO backward()
# ✓ NO optimizer
# ✓ NO checkpoint modification
# ✓ NO training
#
# =============================================================================

import os
import gc
import json
import time
import math
import torch


# =============================================================================
# CONFIGURATION
# =============================================================================

DEVICE = torch.device("cuda:0")

TEST_PATH = (
    "/kaggle/input/datasets/"
    "armankrsingh/test-cs781-json/test.json"
)

OUTPUT_DIR = (
    "/kaggle/working/"
    "phase4_step1000_test_predictions"
)

JSONL_PATH = os.path.join(
    OUTPUT_DIR,
    "step1000_predictions.jsonl"
)

JSON_PATH = os.path.join(
    OUTPUT_DIR,
    "step1000_predictions.json"
)

SUMMARY_PATH = os.path.join(
    OUTPUT_DIR,
    "step1000_generation_summary.json"
)

# -------------------------------------------------------------------------
# Generation configuration
# -------------------------------------------------------------------------

MAX_NEW_TOKENS = 128

# Keep the complete prompt within the model's 4096-token context.
CONTEXT_LENGTH = 4096

# Deterministic generation.
# This is important for a reproducible validation run.
DO_SAMPLE = False

# These are deliberately conservative.
TEMPERATURE = None
TOP_P = None

# Progress reporting.
PRINT_EVERY = 25

# Save every N examples so partial progress is visible on disk.
FLUSH_EVERY = 10


# =============================================================================
# HEADER
# =============================================================================

print("=" * 80)
print("CS781 — STEP-1000 HELD-OUT TEST GENERATION")
print("=" * 80)

print()
print("Device             :", DEVICE)
print("Test file          :", TEST_PATH)
print("Output directory   :", OUTPUT_DIR)
print("Max new tokens     :", MAX_NEW_TOKENS)
print("Context length     :", CONTEXT_LENGTH)
print("Sampling           :", DO_SAMPLE)

# =============================================================================
# PRECONDITIONS
# =============================================================================

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required.")

if "model" not in globals():
    raise RuntimeError(
        "model is missing. Run the verified Step-1000 model-loading cell."
    )

if "tokenizer" not in globals():
    raise RuntimeError(
        "tokenizer is missing. Run the tokenizer-loading cell."
    )

if not os.path.exists(TEST_PATH):
    raise FileNotFoundError(TEST_PATH)

# Correct device check.
model_device = next(model.parameters()).device

if model_device.type != "cuda":
    raise RuntimeError(
        f"Model is not on CUDA: {model_device}"
    )

if model_device.index not in (None, 0):
    raise RuntimeError(
        f"Model is on unexpected GPU: {model_device}"
    )

# Evaluation mode is mandatory.
model.eval()

if hasattr(model.config, "use_cache"):
    model.config.use_cache = True

print()
print("✓ CUDA available")
print("✓ Model present")
print("✓ Tokenizer present")
print("✓ Model device:", model_device)
print("✓ Evaluation mode:", not model.training)

# =============================================================================
# LOAD TEST JSON
# =============================================================================

print()
print("=" * 80)
print("LOADING TEST SET")
print("=" * 80)

with open(
    TEST_PATH,
    "r",
    encoding="utf-8",
) as f:

    test_data = json.load(f)

if not isinstance(test_data, dict):
    raise RuntimeError(
        f"Expected top-level dict, got {type(test_data)}"
    )

if len(test_data) != 1000:
    raise RuntimeError(
        f"Expected 1,000 test examples, found {len(test_data)}"
    )

print("✓ Loaded test.json")
print("Examples:", len(test_data))

# =============================================================================
# PREPARE OUTPUT DIRECTORY
# =============================================================================

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True,
)

# Do NOT silently append to an old prediction file.
# This prevents accidentally mixing results from another run.

if os.path.exists(JSONL_PATH):
    raise RuntimeError(
        "\n"
        "Prediction file already exists:\n"
        f"{JSONL_PATH}\n\n"
        "Delete/rename it if you intentionally want a fresh run."
    )

# =============================================================================
# TOKENIZER ADAPTER
# =============================================================================
#
# Your tokenizer is the RAW BPE tokenizer, not a callable HuggingFace
# PreTrainedTokenizer. Therefore we explicitly use tokenizer.encode().
#
# =============================================================================

def get_token_ids(text):

    encoded = tokenizer.encode(text)

    if hasattr(encoded, "ids"):
        ids = encoded.ids
    else:
        ids = encoded

    ids = [int(x) for x in ids]

    if not ids:
        raise RuntimeError(
            "Tokenizer returned zero tokens."
        )

    return ids


def decode_ids(ids):

    ids = [int(x) for x in ids]

    return tokenizer.decode(ids)


# =============================================================================
# GENERATION FUNCTION
# =============================================================================

def generate_continuation(prompt):

    token_ids = get_token_ids(prompt)

    # ---------------------------------------------------------
    # Context protection
    # ---------------------------------------------------------
    #
    # We need room for NEW tokens.
    #
    # Example:
    #   626 prompt tokens
    #   + 128 generated tokens
    #   = 754 tokens
    #
    # If a prompt is already near 4096 tokens, truncate the
    # LEFT side so the generation still has room.
    #
    # ---------------------------------------------------------

    max_prompt_tokens = (
        CONTEXT_LENGTH - MAX_NEW_TOKENS
    )

    if len(token_ids) > max_prompt_tokens:

        original_prompt_tokens = len(token_ids)

        token_ids = token_ids[
            -max_prompt_tokens:
        ]

        prompt_was_truncated = True

    else:

        original_prompt_tokens = len(token_ids)
        prompt_was_truncated = False

    input_ids = torch.tensor(
        [token_ids],
        dtype=torch.long,
        device=DEVICE,
    )

    attention_mask = torch.ones_like(
        input_ids
    )

    # ---------------------------------------------------------
    # READ-ONLY GENERATION
    # ---------------------------------------------------------

    with torch.inference_mode():

        generated_ids = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=DO_SAMPLE,
            pad_token_id=(
                getattr(
                    tokenizer,
                    "pad_token_id",
                    None
                )
                if hasattr(
                    tokenizer,
                    "pad_token_id"
                )
                else None
            ),
            use_cache=True,
        )

    # generated_ids includes the prompt.
    prompt_length = input_ids.shape[-1]

    continuation_ids = generated_ids[
        0,
        prompt_length:
    ].detach().cpu().tolist()

    # ---------------------------------------------------------
    # FINITE / VALID ID CHECK
    # ---------------------------------------------------------

    if not continuation_ids:
        raise RuntimeError(
            "Model generated zero continuation tokens."
        )

    vocab_size = int(
        getattr(
            model.config,
            "vocab_size",
            32000,
        )
    )

    invalid_ids = [
        x
        for x in continuation_ids
        if x < 0 or x >= vocab_size
    ]

    if invalid_ids:
        raise RuntimeError(
            f"Invalid generated token IDs: "
            f"{invalid_ids[:20]}"
        )

    continuation_text = decode_ids(
        continuation_ids
    )

    return {
        "prompt_tokens_original": original_prompt_tokens,
        "prompt_tokens_used": len(token_ids),
        "prompt_was_truncated": prompt_was_truncated,
        "generated_tokens": len(continuation_ids),
        "generated_text": continuation_text,
    }


# =============================================================================
# RUN ALL 1,000 EXAMPLES
# =============================================================================

print()
print("=" * 80)
print("GENERATING PREDICTIONS")
print("=" * 80)

start_time = time.time()

results = []

successful = 0
failed = 0

total_prompt_tokens = 0
total_generated_tokens = 0
truncated_prompts = 0

# Open JSONL once and flush periodically.
with open(
    JSONL_PATH,
    "w",
    encoding="utf-8",
) as jsonl_file:

    for position, (example_id, record) in enumerate(
        test_data.items(),
        start=1,
    ):

        # -----------------------------------------------------
        # Validate record
        # -----------------------------------------------------

        if not isinstance(record, dict):
            raise RuntimeError(
                f"Example {example_id} is not a dict."
            )

        if "text" not in record:
            raise RuntimeError(
                f"Example {example_id} has no 'text' field."
            )

        prompt = str(
            record["text"]
        ).strip()

        if not prompt:
            raise RuntimeError(
                f"Example {example_id} has empty text."
            )

        # -----------------------------------------------------
        # Generate
        # -----------------------------------------------------

        try:

            generation = generate_continuation(
                prompt
            )

            result = {
                "id": str(example_id),

                # Original held-out input.
                "prompt": prompt,

                # Generated continuation only.
                "prediction": generation[
                    "generated_text"
                ],

                # Generation metadata.
                "prompt_tokens_original":
                    generation[
                        "prompt_tokens_original"
                    ],

                "prompt_tokens_used":
                    generation[
                        "prompt_tokens_used"
                    ],

                "prompt_was_truncated":
                    generation[
                        "prompt_was_truncated"
                    ],

                "generated_tokens":
                    generation[
                        "generated_tokens"
                    ],
            }

            successful += 1

            total_prompt_tokens += (
                generation[
                    "prompt_tokens_used"
                ]
            )

            total_generated_tokens += (
                generation[
                    "generated_tokens"
                ]
            )

            if generation[
                "prompt_was_truncated"
            ]:
                truncated_prompts += 1

        except Exception as exc:

            failed += 1

            result = {
                "id": str(example_id),
                "prompt": prompt,
                "prediction": None,
                "error": repr(exc),
            }

            print()
            print(
                f"!!! FAILED example "
                f"{example_id}: {exc}"
            )

        # -----------------------------------------------------
        # Store in memory
        # -----------------------------------------------------

        results.append(result)

        # -----------------------------------------------------
        # Write immediately to JSONL
        # -----------------------------------------------------

        jsonl_file.write(
            json.dumps(
                result,
                ensure_ascii=False,
            )
            + "\n"
        )

        if (
            position % FLUSH_EVERY == 0
        ):
            jsonl_file.flush()

        # -----------------------------------------------------
        # Progress
        # -----------------------------------------------------

        if (
            position % PRINT_EVERY == 0
            or position == len(test_data)
        ):

            elapsed = (
                time.time()
                - start_time
            )

            rate = (
                position / elapsed
                if elapsed > 0
                else 0
            )

            eta = (
                (len(test_data) - position)
                / rate
                if rate > 0
                else float("inf")
            )

            print(
                f"{position:4d}/1000 | "
                f"success={successful:4d} | "
                f"failed={failed:3d} | "
                f"rate={rate:.2f}/s | "
                f"ETA={eta/60:.1f} min"
            )

# =============================================================================
# SAVE JSON COPY
# =============================================================================

print()
print("=" * 80)
print("SAVING RESULTS")
print("=" * 80)

with open(
    JSON_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        results,
        f,
        ensure_ascii=False,
        indent=2,
    )

# =============================================================================
# SUMMARY
# =============================================================================

elapsed = (
    time.time()
    - start_time
)

summary = {
    "model_step": 1000,
    "test_examples": len(test_data),
    "successful_examples": successful,
    "failed_examples": failed,

    "max_new_tokens": MAX_NEW_TOKENS,
    "context_length": CONTEXT_LENGTH,

    "total_prompt_tokens_used":
        total_prompt_tokens,

    "total_generated_tokens":
        total_generated_tokens,

    "truncated_prompts":
        truncated_prompts,

    "elapsed_seconds":
        elapsed,

    "examples_per_second":
        (
            len(test_data) / elapsed
            if elapsed > 0
            else None
        ),

    "generation_mode":
        "greedy",

    "test_file":
        TEST_PATH,

    "checkpoint_step":
        1000,
}

with open(
    SUMMARY_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        summary,
        f,
        indent=2,
    )

# =============================================================================
# FINAL REPORT
# =============================================================================

print()
print("=" * 80)
print("STEP-1000 GENERATION COMPLETE")
print("=" * 80)

print(
    f"Test examples       : {len(test_data):,}"
)

print(
    f"Successful           : {successful:,}"
)

print(
    f"Failed               : {failed:,}"
)

print(
    f"Generated tokens     : "
    f"{total_generated_tokens:,}"
)

print(
    f"Truncated prompts    : "
    f"{truncated_prompts:,}"
)

print(
    f"Elapsed time         : "
    f"{elapsed / 60:.2f} minutes"
)

if elapsed > 0:
    print(
        f"Examples/sec        : "
        f"{len(test_data) / elapsed:.3f}"
    )

print()
print("OUTPUT FILES")
print("-" * 80)

print(JSONL_PATH)
print(JSON_PATH)
print(SUMMARY_PATH)

# =============================================================================
# HARD COMPLETION GATES
# =============================================================================

print()
print("=" * 80)
print("COMPLETION GATES")
print("=" * 80)

if failed != 0:
    raise RuntimeError(
        f"Generation completed with {failed} failed examples. "
        "Do not treat the run as complete."
    )

if successful != 1000:
    raise RuntimeError(
        f"Expected 1000 successful predictions, "
        f"got {successful}."
    )

if len(results) != 1000:
    raise RuntimeError(
        f"Expected 1000 result records, "
        f"got {len(results)}."
    )

print("✓ 1,000/1,000 examples generated")
print("✓ 0 generation failures")
print("✓ Predictions written to JSONL")
print("✓ Predictions written to JSON")
print("✓ Summary written")

# =============================================================================
# SHOW A FEW EXAMPLES
# =============================================================================

print()
print("=" * 80)
print("SAMPLE PREDICTIONS")
print("=" * 80)

for sample in results[:3]:

    print()
    print("-" * 80)
    print("ID:", sample["id"])
    print(
        "Prompt tokens:",
        sample["prompt_tokens_used"]
    )
    print(
        "Generated tokens:",
        sample["generated_tokens"]
    )

    print()
    print("GENERATED CONTINUATION:")
    print(sample["prediction"])


# =============================================================================
# CLEANUP
# =============================================================================

gc.collect()
torch.cuda.empty_cache()

print()
print("=" * 80)
print("✓ HELD-OUT GENERATION FINISHED")
print("=" * 80)

In [ ]:
# ============================================================
# CS781 — STEP-1000 PREDICTION FILE LOCAL VALIDATOR
# ============================================================
#
# INPUT:
#   step1000_predictions.json
#
# PURPOSE:
#   Validate the EXISTING 1,000 generated predictions.
#
# DOES NOT:
#   - load the model
#   - train
#   - modify weights
#   - call backward()
#   - call optimizer.step()
#   - regenerate predictions
#
# CHECKS:
#   ✓ 1,000 prediction records
#   ✓ IDs
#   ✓ 128 generated-token requirement
#   ✓ BPE Ġ cleanup / decoding
#   ✓ repeated n-grams
#   ✓ unique-token ratio
#   ✓ longest repeated sequence
#   ✓ degeneration / looping
#   ✓ EOS statistics
#   ✓ length statistics
#   ✓ representative examples
#
# ============================================================

import json
import os
import re
import math
import statistics
from collections import Counter


# ============================================================
# CONFIGURATION
# ============================================================

PREDICTION_DIR = (
    "/kaggle/working/phase4_step1000_test_predictions"
)

PREDICTION_JSON = os.path.join(
    PREDICTION_DIR,
    "step1000_predictions.json"
)

REPORT_JSON = os.path.join(
    PREDICTION_DIR,
    "step1000_generation_quality_report.json"
)

EXPECTED_EXAMPLES = 1000
EXPECTED_GENERATED_TOKENS = 128


# ============================================================
# HELPERS
# ============================================================

def load_json(path):

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:
        return json.load(f)


def normalize_bpe_display(text):

    """
    Converts GPT-style BPE word-boundary markers into
    human-readable text.

    IMPORTANT:
    This is only for diagnostics/display.
    We do NOT alter the original prediction file.
    """

    if text is None:
        return ""

    text = str(text)

    # GPT-style BPE boundary marker.
    text = text.replace("Ġ", " ")

    # Common BPE newline marker.
    text = text.replace("Ċ", "\n")

    # Remove obvious leading/trailing whitespace.
    return text.strip()


def whitespace_words(text):

    text = normalize_bpe_display(text)

    if not text:
        return []

    return text.split()


def simple_tokens(text):

    """
    Tokenization used ONLY for repetition diagnostics.
    This is not claimed to be the competition tokenizer.
    """

    text = normalize_bpe_display(text)

    return re.findall(
        r"\w+|[^\w\s]",
        text,
        flags=re.UNICODE
    )


def ngram_counts(tokens, n):

    if len(tokens) < n:
        return Counter()

    return Counter(
        tuple(tokens[i:i+n])
        for i in range(len(tokens) - n + 1)
    )


def repeated_ngram_stats(tokens, n):

    counts = ngram_counts(tokens, n)

    if not counts:
        return {
            "total_ngrams": 0,
            "unique_ngrams": 0,
            "repeated_unique_ngrams": 0,
            "repeated_occurrences": 0,
            "max_count": 0,
        }

    repeated = {
        gram: count
        for gram, count in counts.items()
        if count > 1
    }

    repeated_occurrences = sum(
        count - 1
        for count in repeated.values()
    )

    return {
        "total_ngrams": sum(counts.values()),
        "unique_ngrams": len(counts),
        "repeated_unique_ngrams": len(repeated),
        "repeated_occurrences": repeated_occurrences,
        "max_count": max(counts.values()),
    }


def longest_consecutive_repeat(tokens):

    """
    Finds the longest immediately repeated token block.

    Example:
      A B C A B C A B C

    repeated block = A B C
    """

    n = len(tokens)

    best_length = 0
    best_block = None
    best_repetitions = 1

    # Keep this bounded because we only need a degeneration
    # diagnostic, not exhaustive sequence mining.
    max_block = min(32, n // 2)

    for block_size in range(
        1,
        max_block + 1
    ):

        i = 0

        while i + block_size * 2 <= n:

            block = tokens[
                i:i + block_size
            ]

            reps = 1

            while (
                i + (reps + 1) * block_size <= n
                and
                tokens[
                    i + reps * block_size:
                    i + (reps + 1) * block_size
                ]
                == block
            ):
                reps += 1

            if reps >= 2:

                total_len = (
                    block_size * reps
                )

                if total_len > best_length:

                    best_length = total_len
                    best_block = block
                    best_repetitions = reps

            i += 1

    return {
        "longest_repeated_block_tokens":
            best_length,

        "block_size_tokens":
            len(best_block)
            if best_block is not None
            else 0,

        "repetitions":
            best_repetitions,

        "block":
            best_block
            if best_block is not None
            else [],
    }


def extract_generated_text(record):

    """
    Handles common prediction-file field names.
    """

    candidates = [
        "generated_text",
        "continuation",
        "generated",
        "prediction",
        "text",
        "output",
    ]

    for key in candidates:

        value = record.get(key)

        if isinstance(value, str):
            return value

    return None


def extract_generated_ids(record):

    candidates = [
        "generated_token_ids",
        "generated_ids",
        "token_ids",
    ]

    for key in candidates:

        value = record.get(key)

        if isinstance(value, list):
            return value

    return None


def extract_prompt_text(record):

    candidates = [
        "prompt",
        "prompt_text",
        "input_text",
    ]

    for key in candidates:

        value = record.get(key)

        if isinstance(value, str):
            return value

    return None


def extract_id(record):

    for key in [
        "id",
        "example_id",
        "document_id",
    ]:

        if key in record:
            return str(record[key])

    return None


# ============================================================
# LOAD
# ============================================================

print("=" * 80)
print("CS781 — STEP-1000 PREDICTION FILE LOCAL VALIDATOR")
print("=" * 80)

print()
print("Prediction file:")
print(PREDICTION_JSON)

if not os.path.exists(PREDICTION_JSON):

    raise FileNotFoundError(
        f"Prediction file not found:\n{PREDICTION_JSON}"
    )

data = load_json(PREDICTION_JSON)

print()
print("Loaded object type:", type(data))


# ============================================================
# NORMALIZE TOP-LEVEL FORMAT
# ============================================================

if isinstance(data, list):

    records = data

elif isinstance(data, dict):

    # Common possibility:
    # {"predictions": [...]}
    if isinstance(
        data.get("predictions"),
        list
    ):
        records = data["predictions"]

    # Or:
    # {"1": {...}, "2": {...}}
    elif all(
        isinstance(v, dict)
        for v in data.values()
    ):
        records = []

        for key, value in data.items():

            item = dict(value)

            if (
                "id" not in item
                and "example_id" not in item
            ):
                item["id"] = key

            records.append(item)

    else:

        raise RuntimeError(
            "Unsupported prediction JSON structure."
        )

else:

    raise RuntimeError(
        "Prediction JSON must contain a list or dictionary."
    )


# ============================================================
# BASIC RECORD CHECK
# ============================================================

print()
print("=" * 80)
print("BASIC FILE CHECK")
print("=" * 80)

print(
    "Records found:",
    len(records)
)

assert len(records) == EXPECTED_EXAMPLES, (
    f"Expected {EXPECTED_EXAMPLES} records, "
    f"found {len(records)}."
)

print(
    f"✓ Exactly {EXPECTED_EXAMPLES:,} prediction records"
)


# ============================================================
# ID CHECK
# ============================================================

ids = []

missing_ids = 0

for record in records:

    example_id = extract_id(record)

    if example_id is None:

        missing_ids += 1

    else:

        ids.append(example_id)


duplicate_ids = [
    x
    for x, count in Counter(ids).items()
    if count > 1
]

expected_ids = {
    str(i)
    for i in range(
        1,
        EXPECTED_EXAMPLES + 1
    )
}

actual_ids = set(ids)

missing_expected_ids = sorted(
    expected_ids - actual_ids,
    key=lambda x: int(x)
)

unexpected_ids = sorted(
    actual_ids - expected_ids
)


print()
print("ID CHECK")
print("-" * 80)

print("IDs found:", len(ids))
print("Missing IDs:", missing_ids)
print("Duplicate IDs:", len(duplicate_ids))
print(
    "Missing expected IDs:",
    len(missing_expected_ids)
)
print(
    "Unexpected IDs:",
    len(unexpected_ids)
)

if (
    missing_ids == 0
    and not duplicate_ids
    and not missing_expected_ids
    and not unexpected_ids
):

    print(
        "✓ IDs 1–1000 are present exactly once"
    )

else:

    print(
        "⚠ ID structure requires inspection"
    )


# ============================================================
# PER-PREDICTION ANALYSIS
# ============================================================

analysis = []

missing_generation = []
short_generation = []
long_generation = []

all_token_lengths = []
all_word_lengths = []

eos_count = 0

repeat2_values = []
repeat3_values = []
repeat4_values = []

unique_ratios = []

degenerate_examples = []


for index, record in enumerate(
    records,
    start=1
):

    example_id = (
        extract_id(record)
        or str(index)
    )

    generated_text = (
        extract_generated_text(record)
    )

    generated_ids = (
        extract_generated_ids(record)
    )

    prompt_text = (
        extract_prompt_text(record)
    )

    # --------------------------------------------------------
    # Missing generation
    # --------------------------------------------------------

    if generated_text is None:

        missing_generation.append(
            example_id
        )

        generated_text = ""

    clean_text = normalize_bpe_display(
        generated_text
    )

    words = whitespace_words(
        clean_text
    )

    diagnostic_tokens = simple_tokens(
        clean_text
    )

    # --------------------------------------------------------
    # Token length
    # --------------------------------------------------------

    if generated_ids is not None:

        generated_token_count = len(
            generated_ids
        )

    else:

        # If token IDs were not stored, use
        # diagnostic token count only.
        generated_token_count = len(
            diagnostic_tokens
        )

    word_count = len(words)

    all_token_lengths.append(
        generated_token_count
    )

    all_word_lengths.append(
        word_count
    )

    if generated_token_count < EXPECTED_GENERATED_TOKENS:

        short_generation.append(
            example_id
        )

    elif generated_token_count > EXPECTED_GENERATED_TOKENS:

        long_generation.append(
            example_id
        )

    # --------------------------------------------------------
    # EOS
    # --------------------------------------------------------

    eos = False

    if generated_ids is not None:

        # Common EOS IDs cannot safely be assumed,
        # so only detect explicit textual EOS markers here.
        pass

    if (
        "<eos>" in clean_text.lower()
        or
        "</s>" in clean_text.lower()
    ):

        eos = True
        eos_count += 1

    # --------------------------------------------------------
    # Repetition
    # --------------------------------------------------------

    r2 = repeated_ngram_stats(
        diagnostic_tokens,
        2
    )

    r3 = repeated_ngram_stats(
        diagnostic_tokens,
        3
    )

    r4 = repeated_ngram_stats(
        diagnostic_tokens,
        4
    )

    repeat2_values.append(
        r2["repeated_occurrences"]
    )

    repeat3_values.append(
        r3["repeated_occurrences"]
    )

    repeat4_values.append(
        r4["repeated_occurrences"]
    )

    # --------------------------------------------------------
    # Unique token ratio
    # --------------------------------------------------------

    if diagnostic_tokens:

        unique_ratio = (
            len(set(diagnostic_tokens))
            / len(diagnostic_tokens)
        )

    else:

        unique_ratio = 0.0

    unique_ratios.append(
        unique_ratio
    )

    # --------------------------------------------------------
    # Consecutive repetition
    # --------------------------------------------------------

    repeat_run = (
        longest_consecutive_repeat(
            diagnostic_tokens
        )
    )

    # Flag obvious degeneration.
    degenerate = (
        repeat_run[
            "repetitions"
        ] >= 4
        or
        repeat_run[
            "longest_repeated_block_tokens"
        ] >= 16
        or
        (
            len(diagnostic_tokens) >= 20
            and unique_ratio < 0.25
        )
    )

    if degenerate:

        degenerate_examples.append(
            {
                "id": example_id,
                "unique_ratio":
                    unique_ratio,
                "repeated_2grams":
                    r2[
                        "repeated_occurrences"
                    ],
                "repeated_3grams":
                    r3[
                        "repeated_occurrences"
                    ],
                "repeated_4grams":
                    r4[
                        "repeated_occurrences"
                    ],
                "longest_repeat":
                    repeat_run,
                "generated_text":
                    clean_text,
            }
        )

    analysis.append(
        {
            "id": example_id,
            "generated_token_count":
                generated_token_count,
            "word_count":
                word_count,
            "unique_token_ratio":
                unique_ratio,
            "repeated_2gram_occurrences":
                r2[
                    "repeated_occurrences"
                ],
            "repeated_3gram_occurrences":
                r3[
                    "repeated_occurrences"
                ],
            "repeated_4gram_occurrences":
                r4[
                    "repeated_occurrences"
                ],
            "longest_repeated_block":
                repeat_run[
                    "longest_repeated_block_tokens"
                ],
            "repeat_block_size":
                repeat_run[
                    "block_size_tokens"
                ],
            "repeat_count":
                repeat_run[
                    "repetitions"
                ],
            "eos_marker_detected":
                eos,
        }
    )


# ============================================================
# AGGREGATE STATISTICS
# ============================================================

def mean_or_zero(values):

    return (
        statistics.mean(values)
        if values
        else 0.0
    )


def median_or_zero(values):

    return (
        statistics.median(values)
        if values
        else 0.0
    )


mean_tokens = mean_or_zero(
    all_token_lengths
)

median_tokens = median_or_zero(
    all_token_lengths
)

mean_words = mean_or_zero(
    all_word_lengths
)

median_words = median_or_zero(
    all_word_lengths
)

mean_unique_ratio = mean_or_zero(
    unique_ratios
)

mean_repeat2 = mean_or_zero(
    repeat2_values
)

mean_repeat3 = mean_or_zero(
    repeat3_values
)

mean_repeat4 = mean_or_zero(
    repeat4_values
)


# ============================================================
# OUTPUT
# ============================================================

print()
print("=" * 80)
print("GENERATION LENGTH")
print("=" * 80)

print(
    f"Expected generated tokens : "
    f"{EXPECTED_GENERATED_TOKENS}"
)

print(
    f"Mean generated tokens     : "
    f"{mean_tokens:.2f}"
)

print(
    f"Median generated tokens   : "
    f"{median_tokens:.2f}"
)

print(
    f"Minimum generated tokens  : "
    f"{min(all_token_lengths)}"
)

print(
    f"Maximum generated tokens  : "
    f"{max(all_token_lengths)}"
)

print(
    f"Short generations         : "
    f"{len(short_generation)}"
)

print(
    f"Long generations          : "
    f"{len(long_generation)}"
)

print()
print("=" * 80)
print("WORD LENGTH")
print("=" * 80)

print(
    f"Mean words                : "
    f"{mean_words:.2f}"
)

print(
    f"Median words              : "
    f"{median_words:.2f}"
)

print(
    f"Minimum words             : "
    f"{min(all_word_lengths)}"
)

print(
    f"Maximum words             : "
    f"{max(all_word_lengths)}"
)


print()
print("=" * 80)
print("REPETITION / DEGENERATION")
print("=" * 80)

print(
    f"Mean unique-token ratio   : "
    f"{mean_unique_ratio:.4f}"
)

print(
    f"Mean repeated 2-grams     : "
    f"{mean_repeat2:.2f}"
)

print(
    f"Mean repeated 3-grams     : "
    f"{mean_repeat3:.2f}"
)

print(
    f"Mean repeated 4-grams     : "
    f"{mean_repeat4:.2f}"
)

print(
    f"Degenerate examples      : "
    f"{len(degenerate_examples):,}"
)

print(
    f"EOS-marker examples      : "
    f"{eos_count:,}"
)


print()
print("=" * 80)
print("FORMAT GATES")
print("=" * 80)

print(
    f"✓ Records: "
    f"{len(records):,}/{EXPECTED_EXAMPLES:,}"
)

print(
    f"✓ Missing generation: "
    f"{len(missing_generation):,}"
)

print(
    f"✓ Duplicate IDs: "
    f"{len(duplicate_ids):,}"
)

if not missing_generation:
    print("✓ Every example has generated output")
else:
    print("✗ Some examples have no generated output")


# ============================================================
# BPE DISPLAY CHECK
# ============================================================

print()
print("=" * 80)
print("BPE DISPLAY CHECK")
print("=" * 80)

sample_record = records[0]

sample_generated = (
    extract_generated_text(
        sample_record
    )
    or ""
)

print("RAW:")
print(
    sample_generated[:500]
)

print()
print("PROCESSED DISPLAY:")
print(
    normalize_bpe_display(
        sample_generated
    )[:500]
)


# ============================================================
# SAMPLE DEGENERATE OUTPUTS
# ============================================================

print()
print("=" * 80)
print("SAMPLE GENERATION DIAGNOSTICS")
print("=" * 80)

if degenerate_examples:

    for item in degenerate_examples[:10]:

        print()
        print("-" * 80)

        print(
            "ID:",
            item["id"]
        )

        print(
            "Unique ratio:",
            f'{item["unique_ratio"]:.4f}'
        )

        print(
            "Repeated 3-grams:",
            item[
                "repeated_3grams"
            ]
        )

        print(
            "Longest repeated block:",
            item[
                "longest_repeat"
            ][
                "longest_repeated_block_tokens"
            ]
        )

        print(
            "Repetitions:",
            item[
                "longest_repeat"
            ][
                "repetitions"
            ]
        )

        print()
        print(
            item["generated_text"][:1000]
        )

else:

    print(
        "No examples crossed the configured "
        "degeneration thresholds."
    )


# ============================================================
# SAMPLE NORMAL OUTPUTS
# ============================================================

print()
print("=" * 80)
print("FIRST 5 PROCESSED PREDICTIONS")
print("=" * 80)

for record in records[:5]:

    example_id = (
        extract_id(record)
        or "?"
    )

    generated = (
        extract_generated_text(record)
        or ""
    )

    print()
    print("-" * 80)
    print("ID:", example_id)
    print(
        normalize_bpe_display(
            generated
        )[:1000]
    )


# ============================================================
# SAVE REPORT
# ============================================================

report = {

    "checkpoint": "Phase 4 Step-1000",

    "prediction_file":
        PREDICTION_JSON,

    "expected_examples":
        EXPECTED_EXAMPLES,

    "actual_examples":
        len(records),

    "expected_generated_tokens":
        EXPECTED_GENERATED_TOKENS,

    "missing_generation_count":
        len(missing_generation),

    "duplicate_id_count":
        len(duplicate_ids),

    "missing_expected_id_count":
        len(missing_expected_ids),

    "unexpected_id_count":
        len(unexpected_ids),

    "short_generation_count":
        len(short_generation),

    "long_generation_count":
        len(long_generation),

    "mean_generated_tokens":
        mean_tokens,

    "median_generated_tokens":
        median_tokens,

    "mean_words":
        mean_words,

    "median_words":
        median_words,

    "mean_unique_token_ratio":
        mean_unique_ratio,

    "mean_repeated_2grams":
        mean_repeat2,

    "mean_repeated_3grams":
        mean_repeat3,

    "mean_repeated_4grams":
        mean_repeat4,

    "eos_marker_count":
        eos_count,

    "degenerate_example_count":
        len(degenerate_examples),

    "id_check_pass":
        (
            missing_ids == 0
            and not duplicate_ids
            and not missing_expected_ids
            and not unexpected_ids
        ),

    "generation_complete":
        (
            len(records)
            == EXPECTED_EXAMPLES
            and not missing_generation
        ),

    "per_example_analysis":
        analysis,

    "degenerate_examples":
        degenerate_examples[:50],
}


with open(
    REPORT_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        report,
        f,
        ensure_ascii=False,
        indent=2
    )


# ============================================================
# FINAL
# ============================================================

print()
print("=" * 80)
print("FINAL LOCAL VALIDATION")
print("=" * 80)

if (
    len(records) == EXPECTED_EXAMPLES
    and not missing_generation
    and missing_ids == 0
    and not duplicate_ids
    and not missing_expected_ids
    and not unexpected_ids
):

    print(
        "✓ STRUCTURAL VALIDATION PASSED"
    )

else:

    print(
        "✗ STRUCTURAL VALIDATION FAILED"
    )

print()
print(
    "Quality report saved to:"
)

print(
    REPORT_JSON
)

print()
print("=" * 80)
print("VALIDATION COMPLETE")
print("=" * 80)

In [ ]:
# ============================================================
# CS781 — CLEAN STEP-1000 PREDICTIONS FOR SUBMISSION
# ============================================================
#
# Converts tokenizer-marked generated text:
#
#   Ġhereby Ġenacted Ġby Ġthe ĠState
#
# into:
#
#   hereby enacted by the State
#
# Also converts:
#   Ġ  -> space
#   Ċ  -> newline
#
# Does NOT modify the model or regenerate anything.
# ============================================================

import json
import os
import re

INPUT_DIR = "/kaggle/working/phase4_step1000_test_predictions"

INPUT_JSON = os.path.join(
    INPUT_DIR,
    "step1000_predictions.json"
)

INPUT_JSONL = os.path.join(
    INPUT_DIR,
    "step1000_predictions.jsonl"
)

OUTPUT_JSON = os.path.join(
    INPUT_DIR,
    "step1000_predictions_clean.json"
)

OUTPUT_JSONL = os.path.join(
    INPUT_DIR,
    "step1000_predictions_clean.jsonl"
)


# ============================================================
# TOKENIZER-MARKER CLEANER
# ============================================================

def clean_generated_text(text):

    if text is None:
        return ""

    text = str(text)

    # GPT-2 style BPE:
    # Ġ = beginning of a word after a space
    text = text.replace("Ġ", " ")

    # Common tokenizer newline marker
    text = text.replace("Ċ", "\n")

    # Other possible byte-level artifacts
    text = text.replace("ĉ", "\n")

    # Normalize tabs
    text = text.replace("ĠĠ", "  ")

    # Fix spaces before punctuation
    text = re.sub(
        r"\s+([,.;:!?%)\]}])",
        r"\1",
        text,
    )

    # Fix spaces immediately after opening punctuation
    text = re.sub(
        r"([(\[{])\s+",
        r"\1",
        text,
    )

    # Normalize excessive spaces while preserving newlines
    text = re.sub(
        r"[ \t]+",
        " ",
        text,
    )

    # Normalize excessive blank lines
    text = re.sub(
        r"\n[ \t]+\n",
        "\n\n",
        text,
    )

    return text.strip()


# ============================================================
# LOAD PREDICTIONS
# ============================================================

if os.path.exists(INPUT_JSON):

    with open(
        INPUT_JSON,
        "r",
        encoding="utf-8",
    ) as f:
        predictions = json.load(f)

    source = INPUT_JSON

elif os.path.exists(INPUT_JSONL):

    predictions = []

    with open(
        INPUT_JSONL,
        "r",
        encoding="utf-8",
    ) as f:

        for line in f:

            line = line.strip()

            if line:
                predictions.append(
                    json.loads(line)
                )

    source = INPUT_JSONL

else:

    raise FileNotFoundError(
        "Could not find Step-1000 prediction file.\n"
        f"Checked:\n"
        f"  {INPUT_JSON}\n"
        f"  {INPUT_JSONL}"
    )


print("=" * 80)
print("CS781 — CLEANING STEP-1000 PREDICTIONS")
print("=" * 80)

print("Source:", source)


# ============================================================
# HANDLE COMMON PREDICTION STRUCTURES
# ============================================================

def clean_record(record):

    if not isinstance(record, dict):
        return record

    cleaned = dict(record)

    # Common field names used by our generation output
    text_fields = [
        "generated_text",
        "generation",
        "generated",
        "prediction",
        "continuation",
        "text",
    ]

    for field in text_fields:

        if field in cleaned and isinstance(
            cleaned[field],
            str,
        ):

            cleaned[field] = clean_generated_text(
                cleaned[field]
            )

    return cleaned


# ============================================================
# CLEAN
# ============================================================

if isinstance(predictions, list):

    cleaned_predictions = [
        clean_record(x)
        for x in predictions
    ]

elif isinstance(predictions, dict):

    cleaned_predictions = {}

    for key, value in predictions.items():

        if isinstance(value, dict):

            cleaned_predictions[key] = clean_record(
                value
            )

        elif isinstance(value, str):

            cleaned_predictions[key] = (
                clean_generated_text(value)
            )

        else:

            cleaned_predictions[key] = value

else:

    raise RuntimeError(
        f"Unexpected prediction structure: "
        f"{type(predictions)}"
    )


# ============================================================
# WRITE CLEAN JSON
# ============================================================

with open(
    OUTPUT_JSON,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        cleaned_predictions,
        f,
        ensure_ascii=False,
        indent=2,
    )


# ============================================================
# WRITE CLEAN JSONL
# ============================================================

with open(
    OUTPUT_JSONL,
    "w",
    encoding="utf-8",
) as f:

    if isinstance(cleaned_predictions, list):

        for record in cleaned_predictions:

            f.write(
                json.dumps(
                    record,
                    ensure_ascii=False,
                )
                + "\n"
            )

    elif isinstance(cleaned_predictions, dict):

        for key, value in cleaned_predictions.items():

            if isinstance(value, dict):

                record = dict(value)

                if "id" not in record:
                    record["id"] = key

            else:

                record = {
                    "id": key,
                    "prediction": value,
                }

            f.write(
                json.dumps(
                    record,
                    ensure_ascii=False,
                )
                + "\n"
            )


# ============================================================
# VERIFY NO TOKENIZER MARKERS REMAIN
# ============================================================

def count_markers(obj):

    if isinstance(obj, str):

        return (
            obj.count("Ġ"),
            obj.count("Ċ"),
        )

    if isinstance(obj, list):

        a = b = 0

        for x in obj:
            aa, bb = count_markers(x)
            a += aa
            b += bb

        return a, b

    if isinstance(obj, dict):

        a = b = 0

        for x in obj.values():
            aa, bb = count_markers(x)
            a += aa
            b += bb

        return a, b

    return 0, 0


remaining_space_markers, remaining_newline_markers = (
    count_markers(cleaned_predictions)
)


# ============================================================
# SAMPLE
# ============================================================

print()
print("=" * 80)
print("CLEANING RESULT")
print("=" * 80)

print("Output JSON :")
print(OUTPUT_JSON)

print()
print("Output JSONL:")
print(OUTPUT_JSONL)

print()
print(
    "Remaining Ġ markers:",
    remaining_space_markers,
)

print(
    "Remaining Ċ markers:",
    remaining_newline_markers,
)

assert remaining_space_markers == 0
assert remaining_newline_markers == 0


# ============================================================
# SHOW FIRST 3
# ============================================================

print()
print("=" * 80)
print("SAMPLE CLEAN PREDICTIONS")
print("=" * 80)

shown = 0

if isinstance(cleaned_predictions, list):

    iterable = enumerate(cleaned_predictions, start=1)

else:

    iterable = cleaned_predictions.items()

for key, record in iterable:

    print()
    print("-" * 80)
    print("ID:", key)

    if isinstance(record, dict):

        for field in [
            "generated_text",
            "generation",
            "generated",
            "prediction",
            "continuation",
            "text",
        ]:

            if field in record:

                print(
                    f"{field}:"
                )

                print(
                    str(record[field])[:1000]
                )

                break

    else:

        print(str(record)[:1000])

    shown += 1

    if shown >= 3:
        break


print()
print("=" * 80)
print("✓ CLEAN PREDICTION FILES CREATED")
print("✓ Ġ tokenizer markers removed correctly")
print("✓ Ċ newline markers converted")
print("✓ JSON and JSONL written")
print("✓ Original predictions untouched")
print("=" * 80)

In [ ]:
# ============================================================
# CS781 — CONVERT PREDICTIONS TO REQUIRED SUBMISSION FORMAT
# ============================================================
#
# REQUIRED BY PHASE-1 SCORER:
#
# {
#   "1": {...},
#   "2": {...},
#   ...
#   "1000": {...}
# }
#
# This cell:
#   ✓ does NOT regenerate predictions
#   ✓ does NOT touch the model
#   ✓ does NOT modify the original predictions
#   ✓ preserves the cleaned generated text
#   ✓ converts top-level list -> ID-mapped JSON object
# ============================================================

import json
import os

BASE = "/kaggle/working/phase4_step1000_test_predictions"

INPUT_FILE = os.path.join(
    BASE,
    "step1000_predictions_clean.json"
)

OUTPUT_FILE = os.path.join(
    BASE,
    "prediction.json"
)

print("=" * 78)
print("CS781 — PREPARING FINAL SUBMISSION JSON")
print("=" * 78)

# ------------------------------------------------------------
# LOAD
# ------------------------------------------------------------

if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(
        f"Clean prediction file not found:\n{INPUT_FILE}"
    )

with open(
    INPUT_FILE,
    "r",
    encoding="utf-8"
) as f:
    data = json.load(f)

print("Input:", INPUT_FILE)
print("Input type:", type(data))

# ------------------------------------------------------------
# CONVERT TO REQUIRED OBJECT
# ------------------------------------------------------------

if isinstance(data, dict):

    # Already in the required general form.
    submission = data

elif isinstance(data, list):

    submission = {}

    for index, record in enumerate(data, start=1):

        # ----------------------------------------------------
        # Determine ID
        # ----------------------------------------------------

        if isinstance(record, dict):

            example_id = (
                record.get("id")
                or record.get("example_id")
                or record.get("key")
                or str(index)
            )

            example_id = str(example_id)

            submission[example_id] = record

        else:

            submission[str(index)] = {
                "prediction": record
            }

else:

    raise RuntimeError(
        f"Unsupported prediction structure: {type(data)}"
    )


# ------------------------------------------------------------
# NORMALIZE KEYS
# ------------------------------------------------------------

submission = {
    str(k): v
    for k, v in submission.items()
}


# ------------------------------------------------------------
# BASIC VALIDATION
# ------------------------------------------------------------

print()
print("=" * 78)
print("SUBMISSION STRUCTURE CHECK")
print("=" * 78)

print("Top-level type:", type(submission))
print("Number of examples:", len(submission))

assert isinstance(
    submission,
    dict
), "Top level must be a JSON object/dict."

assert len(submission) == 1000, (
    f"Expected 1000 predictions, "
    f"found {len(submission)}."
)

expected_ids = {
    str(i)
    for i in range(1, 1001)
}

actual_ids = set(submission.keys())

missing_ids = expected_ids - actual_ids
extra_ids = actual_ids - expected_ids

if missing_ids:
    raise RuntimeError(
        f"Missing prediction IDs: "
        f"{sorted(missing_ids, key=lambda x: int(x))[:20]}"
    )

if extra_ids:
    raise RuntimeError(
        f"Unexpected prediction IDs: "
        f"{sorted(extra_ids)[:20]}"
    )

print("✓ Top level is a JSON object")
print("✓ Exactly 1000 prediction records")
print("✓ IDs 1–1000 all present")
print("✓ No extra IDs")


# ------------------------------------------------------------
# WRITE FINAL FILE
# ------------------------------------------------------------

with open(
    OUTPUT_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        submission,
        f,
        ensure_ascii=False,
        indent=2
    )


# ------------------------------------------------------------
# RELOAD — VERIFY ACTUAL FILE
# ------------------------------------------------------------

with open(
    OUTPUT_FILE,
    "r",
    encoding="utf-8"
) as f:

    verified = json.load(f)

assert isinstance(verified, dict)
assert len(verified) == 1000

print()
print("=" * 78)
print("FINAL FILE")
print("=" * 78)

print("Created:")
print(OUTPUT_FILE)

print()
print("Top-level JSON type:", type(verified))
print("Records:", len(verified))

# ------------------------------------------------------------
# SAMPLE
# ------------------------------------------------------------

print()
print("=" * 78)
print("SAMPLE RECORDS")
print("=" * 78)

for example_id in ["1", "2", "3"]:

    record = verified[example_id]

    print()
    print("-" * 78)
    print("ID:", example_id)
    print("Record type:", type(record))

    if isinstance(record, dict):

        print("Fields:", list(record.keys()))

        for field in [
            "prediction",
            "generated_text",
            "generation",
            "generated",
            "continuation",
            "text",
        ]:

            if field in record:

                print(
                    f"{field}: "
                    f"{str(record[field])[:500]}"
                )

                break

    else:

        print(str(record)[:500])


print()
print("=" * 78)
print("✓ FINAL prediction.json READY")
print("=" * 78)
print()
print("Submit this file:")
print(OUTPUT_FILE)

In [ ]:
# ============================================================
# ONE-CELL KAGGLE INFERENCE
# Mistral Small 3.1 24B Base
# test.json -> prediction.json
# ============================================================

# Install compatible libraries
!pip install -q -U "transformers>=5.0.0" accelerate bitsandbytes sentencepiece 

import os
import json
import torch

from transformers import (
    AutoTokenizer,
    Mistral3ForConditionalGeneration,
    BitsAndBytesConfig,
)

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

MODEL_ID = "mistralai/Mistral-Small-3.1-24B-Base-2503"

TEST_FILE = "/kaggle/input/datasets/armankrsingh/test-cs781-json/test.json"

OUTPUT_FILE = "/kaggle/working/prediction.json"

assert os.path.exists(TEST_FILE), f"test.json not found: {TEST_FILE}"

print("Test file found:", TEST_FILE)

# ------------------------------------------------------------
# Load test data
# ------------------------------------------------------------

with open(TEST_FILE, "r", encoding="utf-8") as f:
    test_data = json.load(f)

print("Test examples:", len(test_data))

# ------------------------------------------------------------
# Load tokenizer
# ------------------------------------------------------------

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# ------------------------------------------------------------
# 4-bit quantization
# ------------------------------------------------------------

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# ------------------------------------------------------------
# Load Mistral 3 model across available GPUs
# ------------------------------------------------------------

model = Mistral3ForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.float16,
    trust_remote_code=True,
)

model.eval()

print("Model loaded successfully.")
print("Device map:", model.hf_device_map)

# ------------------------------------------------------------
# Generate ~128-word continuations
# ------------------------------------------------------------

def trim_to_128_words(text):
    return " ".join(text.split()[:128]).strip()


predictions = {}

for i, (example_id, example) in enumerate(test_data.items(), start=1):

    prompt = example["text"]

    print(f"\n[{i}/{len(test_data)}] ID: {example_id}")

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=4096,
    )

    # First device used by the model
    input_device = next(model.parameters()).device

    inputs = {
        k: v.to(input_device)
        for k, v in inputs.items()
    }

    with torch.no_grad():

        output = model.generate(
            **inputs,
            max_new_tokens=220,
            do_sample=False,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    # Remove prompt
    generated_tokens = output[
        0,
        inputs["input_ids"].shape[1]:
    ]

    generated_text = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True,
    ).strip()

    # Phase-1 requires 128-word continuation
    generated_text = trim_to_128_words(generated_text)

    predictions[example_id] = {
        "text": generated_text
    }

    print("Generated words:", len(generated_text.split()))
    print("Preview:", generated_text[:250])

# ------------------------------------------------------------
# Save prediction.json
# ------------------------------------------------------------

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(
        predictions,
        f,
        ensure_ascii=False,
        indent=2,
    )

print("\n========================================")
print("DONE")
print("========================================")
print("Prediction file:", OUTPUT_FILE)
print("Number of predictions:", len(predictions))
print("========================================")

In [ ]:
# ==================================================================================================
# CS781 — 113M MODEL PRE-EXPERIMENT SANITY CHECK — ROBUST VERSION
# ==================================================================================================

import os
import json
import math
import torch

print("=" * 100)
print("CS781 — 113M MODEL PRE-EXPERIMENT SANITY CHECK")
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# 1. ENVIRONMENT
# --------------------------------------------------------------------------------------------------

print("\n[1] ENVIRONMENT")
print("-" * 100)

print("PyTorch       :", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device        :", DEVICE)

if torch.cuda.is_available():
    print("GPU           :", torch.cuda.get_device_name(0))
    print("GPU memory    : %.2f GB" %
          (torch.cuda.get_device_properties(0).total_memory / 1024**3))


# --------------------------------------------------------------------------------------------------
# 2. VERIFIED FILES
# --------------------------------------------------------------------------------------------------

print("\n[2] VERIFIED FILES")
print("-" * 100)

CHECKPOINT_PATH = (
    "/kaggle/input/datasets/oipranesh/"
    "phase4-step1000-verified-model/checkpoint_step_001000.pt"
)

TOKENIZER_PATH = (
    "/kaggle/input/datasets/oipranesh/"
    "cs781-phase4-training-prep/tokenizer/legal_bpe_32k.json"
)

for name, path in [
    ("Step-1000 checkpoint", CHECKPOINT_PATH),
    ("Legal BPE tokenizer", TOKENIZER_PATH),
]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"{name} not found:\n{path}")

    size_gb = os.path.getsize(path) / 1024**3
    print(f"✓ {name} exists")
    print(f"  Path : {path}")
    print(f"  Size : {size_gb:.2f} GB")


# --------------------------------------------------------------------------------------------------
# 3. LOAD EXACT TOKENIZER FROM VERIFIED PATH
# --------------------------------------------------------------------------------------------------

print("\n[3] TOKENIZER")
print("-" * 100)

# IMPORTANT:
# We intentionally load the exact verified tokenizer file instead of
# depending on a previous notebook variable.

try:
    from tokenizers import Tokenizer
except Exception as e:
    raise RuntimeError(f"Could not import tokenizers: {e}")

tokenizer = Tokenizer.from_file(TOKENIZER_PATH)

print("✓ Exact tokenizer loaded")
print("Tokenizer type :", type(tokenizer).__name__)
print("Vocabulary size:", len(tokenizer.get_vocab()))

test_encoding = tokenizer.encode(
    "The Government shall make regulations under this Act."
)

print("Test text      : legal sentence")
print("Token count    :", len(test_encoding.ids))
print("Token IDs      :", test_encoding.ids[:20])
print("✓ Tokenizer encoding works")


# --------------------------------------------------------------------------------------------------
# 4. LOAD CHECKPOINT — CPU ONLY
# --------------------------------------------------------------------------------------------------

print("\n[4] CHECKPOINT")
print("-" * 100)

# CPU loading first keeps this diagnostic safe and avoids unnecessarily
# occupying GPU memory before architecture inspection.

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False,
)

print("✓ Checkpoint loaded")
print("Checkpoint type:", type(checkpoint).__name__)

if not isinstance(checkpoint, dict):
    raise RuntimeError("Checkpoint is not a dictionary.")

print("Checkpoint keys:")
for k in checkpoint.keys():
    print(" ", k)

required_keys = [
    "model_state_dict",
    "step",
    "context_length",
    "batch_size",
    "gradient_accumulation_steps",
    "learning_rate",
]

missing = [k for k in required_keys if k not in checkpoint]

if missing:
    raise RuntimeError(f"Checkpoint missing expected keys: {missing}")

print("\nCheckpoint step              :", checkpoint["step"])
print("Checkpoint context length   :", checkpoint["context_length"])
print("Checkpoint batch size       :", checkpoint["batch_size"])
print("Checkpoint grad accumulation:", checkpoint["gradient_accumulation_steps"])
print("Checkpoint learning rate    :", checkpoint["learning_rate"])


# --------------------------------------------------------------------------------------------------
# 5. MODEL ARCHITECTURE RECOVERY
# --------------------------------------------------------------------------------------------------

print("\n[5] MODEL ARCHITECTURE")
print("-" * 100)

# Reuse the architecture class already established by the successful
# recovery experiment if it exists.
#
# Otherwise import GPT2LMHeadModel.

if "GPT2LMHeadModel" not in globals():
    try:
        from transformers import GPT2LMHeadModel
    except Exception as e:
        raise RuntimeError(
            "GPT2LMHeadModel could not be imported. "
            f"Transformers error: {e}"
        )

# Prefer the previously recovered architecture object if available.
architecture_source = None

if "diag_model" in globals() and diag_model is not None:
    try:
        model = type(diag_model)(diag_model.config)
        architecture_source = "existing diag_model.config"
    except Exception:
        model = None

else:
    model = None


# If diag_model is unavailable, infer GPT-2 configuration from checkpoint
# tensor shapes. This does not change the checkpoint.

if model is None:

    state = checkpoint["model_state_dict"]

    # Identify vocabulary and embedding dimension.
    if "transformer.wte.weight" in state:
        vocab_size, n_embd = state["transformer.wte.weight"].shape
    elif "transformer.wte.weight" in state:
        vocab_size, n_embd = state["transformer.wte.weight"].shape
    else:
        raise RuntimeError(
            "Could not find transformer.wte.weight in checkpoint."
        )

    # Count transformer blocks.
    block_ids = []

    for key in state.keys():
        if key.startswith("transformer.h."):
            parts = key.split(".")
            if len(parts) >= 3:
                try:
                    block_ids.append(int(parts[2]))
                except Exception:
                    pass

    if not block_ids:
        raise RuntimeError(
            "Could not infer transformer block count from checkpoint."
        )

    n_layer = max(block_ids) + 1

    # Infer number of attention heads from c_attn projection dimension.
    # GPT-2 c_attn output = 3 * hidden_size.
    if "transformer.h.0.attn.c_attn.weight" in state:
        c_attn_shape = state["transformer.h.0.attn.c_attn.weight"].shape
        out_dim = c_attn_shape[1]

        if out_dim == 3 * n_embd:
            # The number of heads cannot always be uniquely inferred from
            # tensors, so use the architecture known for this checkpoint.
            n_head = 12
        else:
            raise RuntimeError(
                f"Unexpected c_attn shape: {c_attn_shape}"
            )
    else:
        raise RuntimeError(
            "Could not find first attention projection."
        )

    # Infer positional/context length.
    if "transformer.wpe.weight" in state:
        n_positions = state["transformer.wpe.weight"].shape[0]
    else:
        n_positions = int(checkpoint["context_length"])

    from transformers import GPT2Config

    config = GPT2Config(
        vocab_size=int(vocab_size),
        n_positions=int(n_positions),
        n_ctx=int(n_positions),
        n_embd=int(n_embd),
        n_layer=int(n_layer),
        n_head=int(n_head),
    )

    model = GPT2LMHeadModel(config)
    architecture_source = "inferred checkpoint architecture"


print("Architecture source:", architecture_source)
print("Model class        :", type(model).__name__)

print("\nModel configuration:")
print("  vocab_size :", model.config.vocab_size)
print("  n_positions:", model.config.n_positions)
print("  n_embd     :", model.config.n_embd)
print("  n_layer    :", model.config.n_layer)
print("  n_head     :", model.config.n_head)


# --------------------------------------------------------------------------------------------------
# 6. EXACT STATE-DICT COMPATIBILITY
# --------------------------------------------------------------------------------------------------

print("\n[6] CHECKPOINT / MODEL COMPATIBILITY")
print("-" * 100)

state = checkpoint["model_state_dict"]

model_state = model.state_dict()

missing_keys = [
    k for k in model_state.keys()
    if k not in state
]

unexpected_keys = [
    k for k in state.keys()
    if k not in model_state
]

shape_mismatches = []

for k in model_state.keys():
    if k in state:
        if tuple(model_state[k].shape) != tuple(state[k].shape):
            shape_mismatches.append(
                (
                    k,
                    tuple(model_state[k].shape),
                    tuple(state[k].shape),
                )
            )

print("Model tensors      :", len(model_state))
print("Checkpoint tensors :", len(state))
print("Missing keys       :", len(missing_keys))
print("Unexpected keys    :", len(unexpected_keys))
print("Shape mismatches   :", len(shape_mismatches))

if missing_keys:
    print("\nFirst missing keys:")
    for k in missing_keys[:10]:
        print(" ", k)

if unexpected_keys:
    print("\nFirst unexpected keys:")
    for k in unexpected_keys[:10]:
        print(" ", k)

if shape_mismatches:
    print("\nFirst shape mismatches:")
    for item in shape_mismatches[:10]:
        print(" ", item)

if missing_keys or unexpected_keys or shape_mismatches:
    raise RuntimeError(
        "MODEL/CHECKPOINT COMPATIBILITY FAILED. "
        "Do not run the next experiment yet."
    )

print("✓ Architecture matches checkpoint exactly")


# --------------------------------------------------------------------------------------------------
# 7. LOAD EXACT STEP-1000 WEIGHTS
# --------------------------------------------------------------------------------------------------

print("\n[7] EXACT STEP-1000 WEIGHTS")
print("-" * 100)

load_result = model.load_state_dict(state, strict=True)

print(load_result)
print("✓ Exact Step-1000 weights loaded")


# --------------------------------------------------------------------------------------------------
# 8. PARAMETER COUNT
# --------------------------------------------------------------------------------------------------

print("\n[8] MODEL SIZE")
print("-" * 100)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

print("Total parameters     :", f"{total_params:,}")
print("Trainable parameters :", f"{trainable_params:,}")

if not (110_000_000 <= total_params <= 115_000_000):
    print("⚠ WARNING: parameter count is outside expected ~113M range.")
else:
    print("✓ Model is in expected ~113M parameter range")


# --------------------------------------------------------------------------------------------------
# 9. MOVE TO GPU AND FORWARD TEST
# --------------------------------------------------------------------------------------------------

print("\n[9] FORWARD-PASS TEST")
print("-" * 100)

model = model.to(DEVICE)
model.eval()

with torch.no_grad():

    encoded = tokenizer.encode(
        "The Government shall make regulations under this Act."
    )

    ids = encoded.ids[:32]

    input_ids = torch.tensor(
        [ids],
        dtype=torch.long,
        device=DEVICE,
    )

    outputs = model(input_ids=input_ids)

    logits = outputs.logits

print("Input shape :", tuple(input_ids.shape))
print("Logits shape:", tuple(logits.shape))

expected_vocab = len(tokenizer.get_vocab())

if logits.shape[-1] != expected_vocab:
    raise RuntimeError(
        f"Tokenizer/model vocabulary mismatch: "
        f"tokenizer={expected_vocab}, model={logits.shape[-1]}"
    )

print("✓ Forward pass works")
print("✓ Model vocabulary matches tokenizer")


# --------------------------------------------------------------------------------------------------
# 10. NEXT-TOKEN SANITY
# --------------------------------------------------------------------------------------------------

print("\n[10] NEXT-TOKEN SANITY")
print("-" * 100)

last_logits = logits[0, -1]

probs = torch.softmax(last_logits, dim=-1)

top_probs, top_ids = torch.topk(probs, k=10)

print("Top-10 token probabilities:")

for rank, (prob, tid) in enumerate(
    zip(top_probs.tolist(), top_ids.tolist()),
    start=1
):
    try:
        token_text = tokenizer.id_to_token(int(tid))
    except Exception:
        token_text = "<?>"

    print(
        f"{rank:2d}. id={tid:5d} "
        f"prob={prob:.6f} "
        f"token={repr(token_text)}"
    )


# --------------------------------------------------------------------------------------------------
# 11. FINAL STATUS
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("CS781 — PRE-EXPERIMENT SANITY CHECK RESULT")
print("=" * 100)

print("Environment             : ✓")
print("Checkpoint              : ✓")
print("Tokenizer               : ✓")
print("Architecture            : ✓")
print("State-dict compatibility: ✓")
print("Exact Step-1000 weights : ✓")
print("Parameter count         : ✓")
print("GPU forward pass        : ✓")
print("Vocabulary alignment    : ✓")
print("Next-token sanity       : ✓")

print("\nMODEL STATUS: ✓ READY FOR NEXT CONTROLLED EXPERIMENT")

print("\nIMPORTANT:")
print("✓ No training performed")
print("✓ No optimizer created")
print("✓ No checkpoint modified")
print("✓ No checkpoint written")
print("✓ Exact Step-1000 checkpoint remains untouched")

print("=" * 100)

In [ ]:
# ==========================================================================================
# CS781 — EXPERIMENT 1: CLEAN GENERATION BASELINE
# Exact Step-1000 model | NO TRAINING | NO CHECKPOINT MODIFICATION
# ==========================================================================================

import os, json, math, torch
import torch.nn.functional as F
from transformers import GPT2Config, GPT2LMHeadModel

print("=" * 100)
print("CS781 — EXPERIMENT 1: CLEAN GENERATION BASELINE")
print("=" * 100)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CHECKPOINT_PATH = "/kaggle/input/datasets/oipranesh/phase4-step1000-verified-model/checkpoint_step_001000.pt"
TOKENIZER_PATH = "/kaggle/input/datasets/oipranesh/cs781-phase4-training-prep/tokenizer/legal_bpe_32k.json"

assert os.path.exists(CHECKPOINT_PATH), "Checkpoint not found"
assert os.path.exists(TOKENIZER_PATH), "Tokenizer not found"

# --------------------------------------------------------------------------
# REUSE EXISTING TOKENIZER — DO NOT CREATE A DIFFERENT ONE
# --------------------------------------------------------------------------
if "tokenizer" not in globals() or tokenizer is None:
    from tokenizers import Tokenizer
    tokenizer = Tokenizer.from_file(TOKENIZER_PATH)

print(f"Device    : {DEVICE}")
print(f"Tokenizer : {type(tokenizer).__name__}")

# --------------------------------------------------------------------------
# LOAD CHECKPOINT
# --------------------------------------------------------------------------
checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False
)

# --------------------------------------------------------------------------
# EXACT ARCHITECTURE FROM VERIFIED SANITY CHECK
# --------------------------------------------------------------------------
config = GPT2Config(
    vocab_size=32000,
    n_positions=4096,
    n_ctx=4096,
    n_embd=768,
    n_layer=12,
    n_head=12,
    use_cache=True
)

model = GPT2LMHeadModel(config)

result = model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True
)

model = model.to(DEVICE)
model.eval()

print(f"Model parameters : {sum(p.numel() for p in model.parameters()):,}")
print(f"Checkpoint step  : {checkpoint['step']}")
print("✓ Exact Step-1000 weights loaded")
print("✓ Checkpoint untouched")

# --------------------------------------------------------------------------
# GENERATION FUNCTION
# --------------------------------------------------------------------------
def generate_text(
    prompt,
    max_new_tokens=256,
    temperature=0.9,
    top_k=None,
    top_p=None,
    seed=42
):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    enc = tokenizer.encode(prompt)
    input_ids = torch.tensor(
        [enc.ids],
        dtype=torch.long,
        device=DEVICE
    )

    generated = input_ids.clone()

    with torch.no_grad():
        for _ in range(max_new_tokens):

            # Respect model context
            context = generated[:, -4096:]

            outputs = model(context)
            logits = outputs.logits[:, -1, :]
            logits = logits / temperature

            # Top-k
            if top_k is not None:
                k = min(top_k, logits.size(-1))
                values, indices = torch.topk(logits, k)
                filtered = torch.full_like(
                    logits,
                    float("-inf")
                )
                filtered.scatter_(1, indices, values)
                logits = filtered

            # Top-p
            if top_p is not None:
                sorted_logits, sorted_indices = torch.sort(
                    logits,
                    descending=True
                )
                probs = F.softmax(sorted_logits, dim=-1)
                cumulative = torch.cumsum(probs, dim=-1)

                remove = cumulative > top_p
                remove[:, 1:] = remove[:, :-1].clone()
                remove[:, 0] = False

                sorted_logits[remove] = float("-inf")

                logits = torch.full_like(logits, float("-inf"))
                logits.scatter_(1, sorted_indices, sorted_logits)

            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)

            generated = torch.cat(
                [generated, next_token],
                dim=1
            )

            # EOS
            if next_token.item() == 3:
                break

    new_ids = generated[0].tolist()
    return tokenizer.decode(new_ids)


# --------------------------------------------------------------------------
# FIXED LEGAL PROMPTS
# --------------------------------------------------------------------------
PROMPTS = [
    "The court held that",
    "Section 1 of the Act provides that",
    "Whereas the State Government",
    "The petitioner submitted that",
    "Notwithstanding anything contained in this Act,"
]

TEMPERATURES = [0.7, 0.8, 0.9, 1.0]

results = []

for prompt in PROMPTS:
    print("\n" + "-" * 100)
    print("PROMPT:", prompt)

    for temp in TEMPERATURES:
        text = generate_text(
            prompt,
            max_new_tokens=256,
            temperature=temp,
            top_k=None,
            top_p=None,
            seed=42
        )

        print(f"\n[T={temp}]")
        print(text[:1200])

        results.append({
            "prompt": prompt,
            "temperature": temp,
            "text": text
        })

# --------------------------------------------------------------------------
# SAVE
# --------------------------------------------------------------------------
OUT_DIR = "/kaggle/working/cs781_exp1_generation_baseline"
os.makedirs(OUT_DIR, exist_ok=True)

with open(
    os.path.join(OUT_DIR, "generation_baseline.json"),
    "w",
    encoding="utf-8"
) as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print("\n" + "=" * 100)
print("✓ EXPERIMENT 1 COMPLETE")
print(f"Results: {OUT_DIR}/generation_baseline.json")
print("Training performed : NO")
print("Checkpoint modified : NO")
print("=" * 100)

In [ ]:
# ==========================================================================================
# CS781 — EXPERIMENT 2: DECODING SWEEP
# NO TRAINING
# ==========================================================================================

print("=" * 100)
print("CS781 — EXPERIMENT 2: DECODING SWEEP")
print("=" * 100)

assert model is not None
model.eval()

PROMPT = "The court held that the provisions of this Act"

SETTINGS = [
    {"name": "greedy",       "temperature": 1.0, "top_k": 1,   "top_p": None},
    {"name": "T0.7",         "temperature": 0.7, "top_k": None,"top_p": None},
    {"name": "T0.9",         "temperature": 0.9, "top_k": None,"top_p": None},
    {"name": "T1.0",         "temperature": 1.0, "top_k": None,"top_p": None},
    {"name": "T0.9_topk50",  "temperature": 0.9, "top_k": 50,  "top_p": None},
    {"name": "T0.9_topp90",  "temperature": 0.9, "top_k": None,"top_p": 0.90},
    {"name": "T0.9_topp95",  "temperature": 0.9, "top_k": None,"top_p": 0.95},
]

sweep_results = []

for cfg in SETTINGS:

    text = generate_text(
        PROMPT,
        max_new_tokens=300,
        temperature=cfg["temperature"],
        top_k=cfg["top_k"],
        top_p=cfg["top_p"],
        seed=123
    )

    print("\n" + "-" * 100)
    print(cfg["name"])
    print("-" * 100)
    print(text[:1500])

    sweep_results.append({
        **cfg,
        "prompt": PROMPT,
        "text": text
    })

OUT_DIR = "/kaggle/working/cs781_exp2_decoding_sweep"
os.makedirs(OUT_DIR, exist_ok=True)

with open(
    os.path.join(OUT_DIR, "decoding_sweep.json"),
    "w",
    encoding="utf-8"
) as f:
    json.dump(sweep_results, f, indent=2, ensure_ascii=False)

print("\n✓ Decoding sweep complete")
print(f"Results: {OUT_DIR}/decoding_sweep.json")

In [ ]:
# ==========================================================================================
# CS781 — EXPERIMENT 3: REPETITION / DIVERSITY DIAGNOSTIC
# ==========================================================================================

from collections import Counter

print("=" * 100)
print("CS781 — EXPERIMENT 3: REPETITION / DIVERSITY DIAGNOSTIC")
print("=" * 100)

def repetition_metrics(text):
    enc = tokenizer.encode(text)
    ids = enc.ids

    if len(ids) < 2:
        return {}

    total = len(ids)

    unique_ratio = len(set(ids)) / total

    adjacent_repeats = sum(
        ids[i] == ids[i-1]
        for i in range(1, total)
    ) / max(1, total - 1)

    def distinct_n(n):
        if total < n:
            return 0.0

        grams = [
            tuple(ids[i:i+n])
            for i in range(total - n + 1)
        ]

        return len(set(grams)) / len(grams)

    def repeated_ngram_rate(n):
        if total < n:
            return 0.0

        grams = [
            tuple(ids[i:i+n])
            for i in range(total - n + 1)
        ]

        counts = Counter(grams)

        repeated = sum(
            c - 1 for c in counts.values()
            if c > 1
        )

        return repeated / len(grams)

    max_run = 1
    current = 1

    for i in range(1, total):
        if ids[i] == ids[i-1]:
            current += 1
            max_run = max(max_run, current)
        else:
            current = 1

    return {
        "tokens": total,
        "unique_ratio": unique_ratio,
        "adjacent_repeat": adjacent_repeats,
        "distinct_2": distinct_n(2),
        "distinct_3": distinct_n(3),
        "repeated_5gram_rate": repeated_ngram_rate(5),
        "max_run": max_run
    }


PROMPTS = [
    "The court held that",
    "Section 1 of the Act provides that",
    "Whereas the State Government",
    "The petitioner submitted that",
    "Notwithstanding anything contained in this Act,"
]

settings = [
    (0.7, None, None),
    (0.8, None, None),
    (0.9, None, None),
    (1.0, None, None),
    (0.9, 50, None),
    (0.9, None, 0.90),
    (0.9, None, 0.95),
]

diagnostics = []

for temp, top_k, top_p in settings:

    for prompt in PROMPTS:

        text = generate_text(
            prompt,
            max_new_tokens=300,
            temperature=temp,
            top_k=top_k,
            top_p=top_p,
            seed=123
        )

        metrics = repetition_metrics(text)

        row = {
            "temperature": temp,
            "top_k": top_k,
            "top_p": top_p,
            "prompt": prompt,
            **metrics
        }

        diagnostics.append(row)

        print(
            f"T={temp:<3} "
            f"top_k={str(top_k):<4} "
            f"top_p={str(top_p):<4} "
            f"unique={metrics['unique_ratio']:.4f} "
            f"adj={metrics['adjacent_repeat']:.4f} "
            f"d2={metrics['distinct_2']:.4f} "
            f"d3={metrics['distinct_3']:.4f} "
            f"5gram={metrics['repeated_5gram_rate']:.4f} "
            f"maxrun={metrics['max_run']}"
        )

OUT_DIR = "/kaggle/working/cs781_exp3_repetition_diagnostic"
os.makedirs(OUT_DIR, exist_ok=True)

with open(
    os.path.join(OUT_DIR, "repetition_diagnostic.json"),
    "w",
    encoding="utf-8"
) as f:
    json.dump(diagnostics, f, indent=2)

print("\n✓ Repetition diagnostic complete")
print(f"Results: {OUT_DIR}/repetition_diagnostic.json")

In [ ]:
# ================================================================================================
# CS781 — STEP-1000 CONTROLLED CONTINUATION — MEMORY-SAFE VERSION
#
# EXPERIMENT:
#   LR = 2.5e-6
#   MicroBatch = 1
#   GradAccum = 8
#   Effective batch = 8
#   Steps = 250
#
# MEMORY FIXES:
#   - No pre-created GPU batch list
#   - No original_state_for_safety copy
#   - CPU token sequences only
#   - One micro-batch moved to GPU at a time
#   - Raw training texts released after tokenization
#   - Checkpoint CPU object released after weights are loaded
# ================================================================================================

import os
import json
import math
import time
import gc
import torch
import torch.nn.functional as F

print("=" * 100)
print("CS781 — STEP-1000 CONTROLLED CONTINUATION — MEMORY-SAFE")
print("=" * 100)

# --------------------------------------------------------------------------------
# CONFIG
# --------------------------------------------------------------------------------

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CHECKPOINT_PATH = (
    "/kaggle/input/datasets/oipranesh/"
    "phase4-step1000-verified-model/checkpoint_step_001000.pt"
)

DATA_ROOT = (
    "/kaggle/input/datasets/oipranesh/"
    "cs781-phase4-training-prep"
)

OUTPUT_DIR = "/kaggle/working/cs781_lr2p5e6_accum8_controlled"

os.makedirs(OUTPUT_DIR, exist_ok=True)

STEPS = 250
SEQ_LEN = 1024

MICRO_BATCH = 1
GRAD_ACCUM = 8
EFFECTIVE_BATCH = MICRO_BATCH * GRAD_ACCUM

LR = 2.5e-6
MAX_GRAD_NORM = 1.0

EVAL_INTERVALS = [50, 100, 150, 200, 250]
TEMPERATURES = [0.9, 1.0]

print(f"Device          : {DEVICE}")
print(f"Checkpoint      : {CHECKPOINT_PATH}")
print(f"Steps           : {STEPS}")
print(f"SeqLen          : {SEQ_LEN}")
print(f"MicroBatch      : {MICRO_BATCH}")
print(f"GradAccum       : {GRAD_ACCUM}")
print(f"Effective batch : {EFFECTIVE_BATCH}")
print(f"Learning rate   : {LR}")
print()

# --------------------------------------------------------------------------------
# VERIFY WORKING NOTEBOOK STATE
# --------------------------------------------------------------------------------

print("=" * 100)
print("VERIFYING EXISTING WORKING NOTEBOOK STATE")
print("=" * 100)

if "tokenizer" not in globals() or tokenizer is None:
    raise RuntimeError(
        "Tokenizer is missing. Run the successful tokenizer/model recovery cell first."
    )

if "model" not in globals() or model is None:
    raise RuntimeError(
        "Model is missing. Run the successful SAFE MODEL OBJECT RECOVERY cell first."
    )

print("✓ tokenizer :", type(tokenizer).__name__)
print("✓ model     :", type(model).__name__)

# --------------------------------------------------------------------------------
# LOAD EXACT STEP-1000 CHECKPOINT
# --------------------------------------------------------------------------------

print()
print("=" * 100)
print("LOADING EXACT STEP-1000 WEIGHTS")
print("=" * 100)

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False
)

if checkpoint.get("step") != 1000:
    raise RuntimeError(
        f"Expected Step-1000 checkpoint, got step={checkpoint.get('step')}"
    )

print("Checkpoint step :", checkpoint.get("step"))
print("Checkpoint LR   :", checkpoint.get("learning_rate"))
print("Context length  :", checkpoint.get("context_length"))
print("Checkpoint batch:", checkpoint.get("batch_size"))
print("Checkpoint accum:", checkpoint.get("gradient_accumulation_steps"))

state_dict = checkpoint["model_state_dict"]

missing, unexpected = model.load_state_dict(
    state_dict,
    strict=False
)

print("Missing keys    :", len(missing))
print("Unexpected keys :", len(unexpected))

if missing or unexpected:
    raise RuntimeError(
        f"Checkpoint/model mismatch. Missing={missing}, Unexpected={unexpected}"
    )

print("✓ Exact Step-1000 weights loaded")
print("✓ Original checkpoint untouched")

# Release huge CPU checkpoint immediately.
del state_dict
del checkpoint

gc.collect()

# --------------------------------------------------------------------------------
# MODEL DEVICE
# --------------------------------------------------------------------------------

model = model.to(DEVICE)
model.train()

# --------------------------------------------------------------------------------
# FIND JSONL FILES
# --------------------------------------------------------------------------------

print()
print("=" * 100)
print("LOADING TRAINING DATA")
print("=" * 100)

jsonl_files = []

for root, dirs, files in os.walk(DATA_ROOT):
    for fname in files:
        if fname.endswith(".jsonl"):
            jsonl_files.append(
                os.path.join(root, fname)
            )

jsonl_files.sort()

print(f"Training JSONL files: {len(jsonl_files)}")

if not jsonl_files:
    raise RuntimeError("No JSONL training files found.")

# --------------------------------------------------------------------------------
# READ + TOKENIZE DIRECTLY
#
# IMPORTANT:
# We do NOT retain all raw texts.
# We tokenize each document as soon as it is read.
# --------------------------------------------------------------------------------

print()
print("Tokenizing training samples...")

token_sequences = []

for file_index, path in enumerate(jsonl_files, 1):

    with open(path, "r", encoding="utf-8") as f:

        for line in f:

            line = line.strip()

            if not line:
                continue

            try:
                obj = json.loads(line)
            except Exception:
                continue

            text = None

            if isinstance(obj, str):
                text = obj

            elif isinstance(obj, dict):

                for key in (
                    "text",
                    "content",
                    "document",
                    "body",
                    "input",
                    "sequence",
                ):

                    value = obj.get(key)

                    if isinstance(value, str) and value.strip():
                        text = value
                        break

            if not text:
                continue

            try:
                ids = tokenizer.encode(text).ids
            except Exception:
                continue

            if len(ids) < SEQ_LEN + 1:
                continue

            # Non-overlapping chunks, exactly as the previous experiment.
            for start in range(
                0,
                len(ids) - SEQ_LEN,
                SEQ_LEN
            ):

                chunk = ids[
                    start:start + SEQ_LEN + 1
                ]

                if len(chunk) == SEQ_LEN + 1:
                    token_sequences.append(chunk)

    # Progress indicator every few files.
    if file_index == 1 or file_index % 10 == 0:
        print(
            f"  processed {file_index:3d}/{len(jsonl_files)} files | "
            f"sequences={len(token_sequences)}"
        )

print(f"Usable token sequences: {len(token_sequences)}")

REQUIRED = STEPS * GRAD_ACCUM * MICRO_BATCH

if len(token_sequences) < REQUIRED:
    raise RuntimeError(
        f"Need at least {REQUIRED} sequences, "
        f"but only {len(token_sequences)} were recovered."
    )

# Use only what this experiment needs.
token_sequences = token_sequences[:REQUIRED]

print(f"Required sequences : {REQUIRED}")
print(f"Using sequences    : {len(token_sequences)}")

# --------------------------------------------------------------------------------
# VALIDATION SET
#
# Keep only a small deterministic CPU validation subset.
# --------------------------------------------------------------------------------

VAL_N = min(5000, len(token_sequences))

validation_sequences = token_sequences[-VAL_N:]

print(f"Validation sequences: {len(validation_sequences)}")

# --------------------------------------------------------------------------------
# MEMORY CHECK
# --------------------------------------------------------------------------------

gc.collect()

if torch.cuda.is_available():

    free_gpu, total_gpu = torch.cuda.mem_get_info()

    print()
    print(
        f"GPU memory before training: "
        f"{(total_gpu-free_gpu)/1024**3:.2f} GB used / "
        f"{total_gpu/1024**3:.2f} GB total"
    )

# --------------------------------------------------------------------------------
# EVALUATION
# --------------------------------------------------------------------------------

@torch.no_grad()
def evaluate_loss(model, sequences, max_items=5000):

    model.eval()

    total_loss = 0.0
    count = 0

    for seq in sequences[:max_items]:

        x = torch.tensor(
            [seq[:-1]],
            dtype=torch.long,
            device=DEVICE
        )

        y = torch.tensor(
            [seq[1:]],
            dtype=torch.long,
            device=DEVICE
        )

        logits = model(x).logits

        loss = F.cross_entropy(
            logits.reshape(-1, logits.size(-1)),
            y.reshape(-1)
        )

        total_loss += loss.item()
        count += 1

        # Explicitly release per-evaluation tensors.
        del x, y, logits, loss

    model.train()

    return total_loss / max(1, count)

# --------------------------------------------------------------------------------
# GENERATION
# --------------------------------------------------------------------------------

@torch.no_grad()
def generate_ids(
    model,
    prompt_ids,
    max_new_tokens=160,
    temperature=0.9
):

    model.eval()

    ids = list(prompt_ids)

    for _ in range(max_new_tokens):

        context = ids[-SEQ_LEN:]

        x = torch.tensor(
            [context],
            dtype=torch.long,
            device=DEVICE
        )

        logits = model(x).logits[:, -1, :]

        logits = logits / temperature

        probs = torch.softmax(logits, dim=-1)

        next_id = torch.multinomial(
            probs,
            num_samples=1
        ).item()

        ids.append(next_id)

        del x, logits, probs

    model.train()

    return ids

# --------------------------------------------------------------------------------
# REPETITION METRICS
# --------------------------------------------------------------------------------

def repetition_metrics(ids):

    if len(ids) < 2:
        return {
            "unique": 0.0,
            "adj": 0.0,
            "d2": 0.0,
            "d3": 0.0,
            "5gram": 0.0,
            "maxrun": 1,
        }

    n = len(ids)

    unique = len(set(ids)) / n

    adj = sum(
        ids[i] == ids[i - 1]
        for i in range(1, n)
    ) / max(1, n - 1)

    bigrams = list(zip(ids[:-1], ids[1:]))

    trigrams = list(
        zip(ids[:-2], ids[1:-1], ids[2:])
    )

    fivegrams = list(
        zip(
            ids[:-4],
            ids[1:-3],
            ids[2:-2],
            ids[3:-1],
            ids[4:],
        )
    )

    d2 = len(set(bigrams)) / max(1, len(bigrams))
    d3 = len(set(trigrams)) / max(1, len(trigrams))

    fivegram_rep = (
        1.0 - len(set(fivegrams)) / len(fivegrams)
        if fivegrams else 0.0
    )

    maxrun = 1
    current = 1

    for i in range(1, n):

        if ids[i] == ids[i - 1]:
            current += 1
            maxrun = max(maxrun, current)
        else:
            current = 1

    return {
        "unique": unique,
        "adj": adj,
        "d2": d2,
        "d3": d3,
        "5gram": fivegram_rep,
        "maxrun": maxrun,
    }

# --------------------------------------------------------------------------------
# GENERATION DIAGNOSTIC
# --------------------------------------------------------------------------------

def run_generation_diagnostic(model, temperature):

    prompt_ids = validation_sequences[0][:32]

    generated = generate_ids(
        model,
        prompt_ids,
        max_new_tokens=160,
        temperature=temperature,
    )

    generated_only = generated[len(prompt_ids):]

    return repetition_metrics(generated_only)

# --------------------------------------------------------------------------------
# BASELINE
# --------------------------------------------------------------------------------

print()
print("=" * 100)
print("BASELINE — EXACT STEP-1000 MODEL")
print("=" * 100)

baseline_loss = evaluate_loss(
    model,
    validation_sequences
)

baseline_ppl = math.exp(
    min(baseline_loss, 20)
)

print(f"Validation loss : {baseline_loss:.4f}")
print(f"Perplexity      : {baseline_ppl:.2f}")

baseline_metrics = {}

for temperature in TEMPERATURES:

    metrics = run_generation_diagnostic(
        model,
        temperature
    )

    baseline_metrics[str(temperature)] = metrics

    print(
        f"T={temperature:.1f} | "
        f"unique={metrics['unique']:.4f} | "
        f"adj={metrics['adj']:.4f} | "
        f"d2={metrics['d2']:.4f} | "
        f"d3={metrics['d3']:.4f} | "
        f"5gram={metrics['5gram']:.4f} | "
        f"maxrun={metrics['maxrun']}"
    )

# --------------------------------------------------------------------------------
# OPTIMIZER
# --------------------------------------------------------------------------------

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    betas=(0.9, 0.95),
    weight_decay=0.1,
)

# --------------------------------------------------------------------------------
# TRAINING
#
# CRITICAL MEMORY FIX:
#   Do NOT create 2000 GPU batches.
#   Construct only one x/y pair per micro-step.
# --------------------------------------------------------------------------------

print()
print("=" * 100)
print("250-STEP LR=2.5e-6 + GRADACCUM=8 CONTINUATION")
print("=" * 100)

start_time = time.time()

diagnostics = []

optimizer.zero_grad(set_to_none=True)

for step in range(1, STEPS + 1):

    accumulated_loss = 0.0

    for micro in range(GRAD_ACCUM):

        idx = (
            (step - 1) * GRAD_ACCUM
            + micro
        )

        seq = token_sequences[idx]

        # ONLY THIS MICRO-BATCH EXISTS ON GPU.
        x = torch.tensor(
            [seq[:-1]],
            dtype=torch.long,
            device=DEVICE
        )

        y = torch.tensor(
            [seq[1:]],
            dtype=torch.long,
            device=DEVICE
        )

        outputs = model(x)

        logits = outputs.logits

        loss = F.cross_entropy(
            logits.reshape(-1, logits.size(-1)),
            y.reshape(-1)
        )

        scaled_loss = loss / GRAD_ACCUM

        scaled_loss.backward()

        accumulated_loss += loss.item()

        # Release references immediately.
        del x
        del y
        del outputs
        del logits
        del scaled_loss
        del loss

    grad_norm = torch.nn.utils.clip_grad_norm_(
        model.parameters(),
        MAX_GRAD_NORM
    )

    optimizer.step()
    optimizer.zero_grad(set_to_none=True)

    mean_loss = accumulated_loss / GRAD_ACCUM

    if step == 1 or step % 25 == 0:

        print(
            f"step={step:4d} | "
            f"loss={mean_loss:.4f} | "
            f"grad={float(grad_norm):.4f}"
        )

    if step in EVAL_INTERVALS:

        print()
        print("-" * 90)
        print(
            f"DIAGNOSTIC CHECKPOINT — +{step} STEPS"
        )
        print("-" * 90)

        # Ensure temporary training allocations are released.
        gc.collect()

        val_loss = evaluate_loss(
            model,
            validation_sequences
        )

        ppl = math.exp(
            min(val_loss, 20)
        )

        print(
            f"Validation loss : {val_loss:.4f}"
        )

        print(
            f"Perplexity      : {ppl:.2f}"
        )

        generation_results = {}

        for temperature in TEMPERATURES:

            metrics = run_generation_diagnostic(
                model,
                temperature
            )

            generation_results[
                str(temperature)
            ] = metrics

            print(
                f"T={temperature:.1f} | "
                f"unique={metrics['unique']:.4f} | "
                f"adj={metrics['adj']:.4f} | "
                f"d2={metrics['d2']:.4f} | "
                f"d3={metrics['d3']:.4f} | "
                f"5gram={metrics['5gram']:.4f} | "
                f"maxrun={metrics['maxrun']}"
            )

        diagnostics.append({
            "step": step,
            "validation_loss": val_loss,
            "perplexity": ppl,
            "generation": generation_results,
        })

# --------------------------------------------------------------------------------
# FINAL SUMMARY
# --------------------------------------------------------------------------------

elapsed = (
    time.time() - start_time
) / 60.0

final_loss = diagnostics[-1]["validation_loss"]

final_ppl = math.exp(
    min(final_loss, 20)
)

loss_change = final_loss - baseline_loss

print()
print("=" * 100)
print("CS781 — LR=2.5e-6 CONTROLLED CONTINUATION SUMMARY")
print("=" * 100)

print(f"Elapsed time : {elapsed:.2f} minutes")
print()

print(
    f"{'POINT':<15}"
    f"{'LOSS':>10}"
    f"{'PPL':>12}"
)

print("-" * 40)

print(
    f"{'step1000':<15}"
    f"{baseline_loss:>10.4f}"
    f"{baseline_ppl:>12.2f}"
)

for diagnostic in diagnostics:

    print(
        f"{('step+' + str(diagnostic['step'])):<15}"
        f"{diagnostic['validation_loss']:>10.4f}"
        f"{diagnostic['perplexity']:>12.2f}"
    )

print()
print(f"Baseline loss : {baseline_loss:.4f}")
print(f"Final loss    : {final_loss:.4f}")
print(f"Loss change   : {loss_change:+.4f}")

# --------------------------------------------------------------------------------
# SAVE RESULTS ONLY
# --------------------------------------------------------------------------------

results = {
    "experiment": "step1000_lr2p5e6_gradaccum8_memory_safe",
    "checkpoint": CHECKPOINT_PATH,
    "starting_step": 1000,
    "steps": STEPS,
    "seq_len": SEQ_LEN,
    "micro_batch": MICRO_BATCH,
    "grad_accum": GRAD_ACCUM,
    "effective_batch": EFFECTIVE_BATCH,
    "learning_rate": LR,
    "baseline_loss": baseline_loss,
    "baseline_perplexity": baseline_ppl,
    "baseline_generation": baseline_metrics,
    "diagnostics": diagnostics,
    "final_loss": final_loss,
    "final_perplexity": final_ppl,
    "loss_change": loss_change,
    "elapsed_minutes": elapsed,
    "production_checkpoint_written": False,
    "original_checkpoint_modified": False,
}

RESULT_PATH = os.path.join(
    OUTPUT_DIR,
    "lr2p5e6_gradaccum8_memory_safe_results.json"
)

with open(
    RESULT_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        results,
        f,
        indent=2
    )

# --------------------------------------------------------------------------------
# FINAL MEMORY CLEANUP
# --------------------------------------------------------------------------------

del optimizer

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print()
print("=" * 100)
print("SAFETY VERIFICATION")
print("=" * 100)

print("✓ Original Step-1000 checkpoint was never modified")
print("✓ No production checkpoint was written")
print("✓ No GPU batch list was retained")
print("✓ Results saved separately")
print(f"✓ Results: {RESULT_PATH}")

print()
print("=" * 100)
print("✓ MEMORY-SAFE CONTROLLED EXPERIMENT COMPLETE")
print("=" * 100)

In [ ]:
# ================================================================================================
# CS781 — CLEAN SESSION MODEL/TOKENIZER RECOVERY
# ================================================================================================

import os
import json
import torch
from transformers import GPT2Config, GPT2LMHeadModel
from tokenizers import Tokenizer

print("=" * 100)
print("CS781 — CLEAN SESSION MODEL/TOKENIZER RECOVERY")
print("=" * 100)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CHECKPOINT_PATH = (
    "/kaggle/input/datasets/oipranesh/"
    "phase4-step1000-verified-model/checkpoint_step_001000.pt"
)

TOKENIZER_PATH = (
    "/kaggle/input/datasets/oipranesh/"
    "cs781-phase4-training-prep/tokenizer/legal_bpe_32k.json"
)

print(f"Device     : {DEVICE}")
print(f"Checkpoint : {CHECKPOINT_PATH}")
print(f"Tokenizer  : {TOKENIZER_PATH}")

# --------------------------------------------------------------------------------
# TOKENIZER
# --------------------------------------------------------------------------------

tokenizer = Tokenizer.from_file(TOKENIZER_PATH)

print()
print("Tokenizer")
print("-" * 60)
print("Type       :", type(tokenizer).__name__)
print("Vocab size :", len(tokenizer.get_vocab()))

# --------------------------------------------------------------------------------
# CHECKPOINT
# --------------------------------------------------------------------------------

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False
)

print()
print("Checkpoint")
print("-" * 60)
print("Step       :", checkpoint["step"])
print("Context    :", checkpoint["context_length"])
print("Batch      :", checkpoint["batch_size"])
print("GradAccum  :", checkpoint["gradient_accumulation_steps"])

if checkpoint["step"] != 1000:
    raise RuntimeError(
        f"Expected Step-1000 checkpoint, got step={checkpoint['step']}"
    )

# --------------------------------------------------------------------------------
# ARCHITECTURE
# --------------------------------------------------------------------------------

state_dict = checkpoint["model_state_dict"]

# Infer GPT-2 architecture from checkpoint tensors.
vocab_size = 32000
n_embd = state_dict["transformer.wte.weight"].shape[1]
n_positions = state_dict["transformer.wpe.weight"].shape[0]

layer_ids = [
    int(k.split(".")[2])
    for k in state_dict
    if k.startswith("transformer.h.") and k.endswith(".ln_1.weight")
]

n_layer = max(layer_ids) + 1

n_head = 12

config = GPT2Config(
    vocab_size=vocab_size,
    n_positions=n_positions,
    n_ctx=n_positions,
    n_embd=n_embd,
    n_layer=n_layer,
    n_head=n_head,
    use_cache=False
)

model = GPT2LMHeadModel(config)

missing, unexpected = model.load_state_dict(
    state_dict,
    strict=False
)

if missing or unexpected:
    raise RuntimeError(
        f"Checkpoint mismatch. Missing={missing}, Unexpected={unexpected}"
    )

model = model.to(DEVICE)
model.eval()

print()
print("Model")
print("-" * 60)
print("Class      :", type(model).__name__)
print("Parameters :", sum(p.numel() for p in model.parameters()))
print("Device     :", next(model.parameters()).device)

# --------------------------------------------------------------------------------
# FORWARD TEST
# --------------------------------------------------------------------------------

test_ids = tokenizer.encode("This is a legal sentence.").ids

x = torch.tensor(
    [test_ids],
    dtype=torch.long,
    device=DEVICE
)

with torch.no_grad():
    logits = model(x).logits

print()
print("Forward test")
print("-" * 60)
print("Input shape :", tuple(x.shape))
print("Logits shape:", tuple(logits.shape))

print()
print("=" * 100)
print("✓ CLEAN MODEL/TOKENIZER RECOVERY COMPLETE")
print("✓ Exact Step-1000 weights loaded")
print("✓ No training performed")
print("✓ Original checkpoint untouched")
print("=" * 100)

In [ ]:
# ================================================================================================
# CS781 — STEP-1000 CONTROLLED CONTINUATION — STREAMING / MEMORY-SAFE
#
# EXPERIMENT:
#   LR = 2.5e-6
#   GradAccum = 8
#   Effective batch = 8
#   250 optimizer steps
#
# CRITICAL FIX:
#   Training data is streamed.
#   We stop immediately after collecting the required 2,000 sequences.
#   We NEVER tokenize the entire corpus.
#
# NO PRODUCTION CHECKPOINT
# ORIGINAL CHECKPOINT NEVER MODIFIED
# ================================================================================================

import os
import json
import math
import time
import gc
import torch
import torch.nn.functional as F

print("=" * 100)
print("CS781 — STEP-1000 CONTROLLED CONTINUATION — STREAMING MEMORY-SAFE")
print("=" * 100)

# --------------------------------------------------------------------------------
# CONFIG
# --------------------------------------------------------------------------------

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CHECKPOINT_PATH = (
    "/kaggle/input/datasets/oipranesh/"
    "phase4-step1000-verified-model/checkpoint_step_001000.pt"
)

DATA_ROOT = (
    "/kaggle/input/datasets/oipranesh/"
    "cs781-phase4-training-prep"
)

OUTPUT_DIR = (
    "/kaggle/working/"
    "cs781_lr2p5e6_accum8_streaming"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

STEPS = 250
SEQ_LEN = 1024

MICRO_BATCH = 1
GRAD_ACCUM = 8

REQUIRED_SEQUENCES = STEPS * GRAD_ACCUM * MICRO_BATCH

LR = 2.5e-6
MAX_GRAD_NORM = 1.0

EVAL_INTERVALS = [50, 100, 150, 200, 250]

TEMPERATURES = [0.9, 1.0]

print(f"Device           : {DEVICE}")
print(f"Checkpoint       : {CHECKPOINT_PATH}")
print(f"Steps            : {STEPS}")
print(f"SeqLen           : {SEQ_LEN}")
print(f"MicroBatch       : {MICRO_BATCH}")
print(f"GradAccum        : {GRAD_ACCUM}")
print(f"Effective batch  : {MICRO_BATCH * GRAD_ACCUM}")
print(f"Learning rate     : {LR}")
print(f"Required sequences: {REQUIRED_SEQUENCES}")

# --------------------------------------------------------------------------------
# VERIFY EXISTING MODEL/TOKENIZER
# --------------------------------------------------------------------------------

print()
print("=" * 100)
print("VERIFYING MODEL / TOKENIZER")
print("=" * 100)

if "tokenizer" not in globals() or tokenizer is None:
    raise RuntimeError("Tokenizer missing. Run Cell 1 first.")

if "model" not in globals() or model is None:
    raise RuntimeError("Model missing. Run Cell 1 first.")

print("✓ tokenizer:", type(tokenizer).__name__)
print("✓ model    :", type(model).__name__)

# --------------------------------------------------------------------------------
# LOAD EXACT CHECKPOINT
# --------------------------------------------------------------------------------

print()
print("=" * 100)
print("LOADING EXACT STEP-1000 WEIGHTS")
print("=" * 100)

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False
)

if checkpoint.get("step") != 1000:
    raise RuntimeError(
        f"Expected Step-1000 checkpoint, got {checkpoint.get('step')}"
    )

missing, unexpected = model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=False
)

if missing or unexpected:
    raise RuntimeError(
        f"Checkpoint mismatch. Missing={missing}, Unexpected={unexpected}"
    )

model = model.to(DEVICE)
model.train()

print("Checkpoint step :", checkpoint["step"])
print("Checkpoint LR   :", checkpoint.get("learning_rate"))
print("Context length  :", checkpoint.get("context_length"))
print("Missing keys    :", len(missing))
print("Unexpected keys :", len(unexpected))
print("✓ Exact Step-1000 weights loaded")
print("✓ Original checkpoint untouched")

# We don't need the 1.2+ GB optimizer state for this experiment.
del checkpoint
gc.collect()

# --------------------------------------------------------------------------------
# FIND JSONL FILES
# --------------------------------------------------------------------------------

print()
print("=" * 100)
print("STREAMING TRAINING DATA")
print("=" * 100)

jsonl_files = []

for root, dirs, files in os.walk(DATA_ROOT):

    for fname in files:

        if fname.endswith(".jsonl"):
            jsonl_files.append(
                os.path.join(root, fname)
            )

jsonl_files = sorted(jsonl_files)

print(f"JSONL files found: {len(jsonl_files)}")

if not jsonl_files:
    raise RuntimeError("No JSONL files found.")

# --------------------------------------------------------------------------------
# STREAM ONLY WHAT WE NEED
# --------------------------------------------------------------------------------

token_sequences = []

processed_files = 0
processed_lines = 0
accepted_sequences = 0

for path in jsonl_files:

    processed_files += 1

    try:
        with open(
            path,
            "r",
            encoding="utf-8"
        ) as f:

            for line in f:

                processed_lines += 1

                line = line.strip()

                if not line:
                    continue

                try:
                    obj = json.loads(line)
                except Exception:
                    continue

                text = None

                if isinstance(obj, str):

                    text = obj

                elif isinstance(obj, dict):

                    for key in (
                        "text",
                        "content",
                        "document",
                        "body",
                        "input",
                        "sequence"
                    ):

                        value = obj.get(key)

                        if isinstance(value, str) and value.strip():

                            text = value
                            break

                if not text:
                    continue

                try:
                    ids = tokenizer.encode(text).ids
                except Exception:
                    continue

                if len(ids) < SEQ_LEN + 1:
                    continue

                # Add chunks only until we have exactly what is required.
                for start in range(
                    0,
                    len(ids) - SEQ_LEN,
                    SEQ_LEN
                ):

                    chunk = ids[
                        start:start + SEQ_LEN + 1
                    ]

                    if len(chunk) != SEQ_LEN + 1:
                        continue

                    token_sequences.append(chunk)

                    accepted_sequences += 1

                    if accepted_sequences >= REQUIRED_SEQUENCES:
                        break

                if accepted_sequences >= REQUIRED_SEQUENCES:
                    break

    except Exception as e:

        print(
            f"WARNING: failed to read {path}: {e}"
        )

    print(
        f"processed {processed_files:3d}/{len(jsonl_files)} "
        f"files | sequences={accepted_sequences}"
    )

    # CRITICAL:
    # Stop reading files immediately once enough data exists.
    if accepted_sequences >= REQUIRED_SEQUENCES:
        break

print()
print(f"Files processed       : {processed_files}")
print(f"Lines processed       : {processed_lines}")
print(f"Sequences collected   : {len(token_sequences)}")
print(f"Sequences required    : {REQUIRED_SEQUENCES}")

if len(token_sequences) < REQUIRED_SEQUENCES:

    raise RuntimeError(
        f"Only {len(token_sequences)} usable sequences were recovered; "
        f"{REQUIRED_SEQUENCES} required."
    )

# --------------------------------------------------------------------------------
# DO NOT CREATE PERSISTENT GPU BATCHES
# --------------------------------------------------------------------------------

# Keep sequences on CPU.
# Each microbatch is converted to GPU only immediately before its forward pass.

# Validation uses a small CPU-held subset.
VAL_N = min(100, len(token_sequences))

validation_sequences = token_sequences[
    -VAL_N:
]

print()
print("✓ Training data prepared")
print("✓ Corpus scan stopped after enough sequences")
print(f"✓ Validation sequences: {VAL_N}")

# --------------------------------------------------------------------------------
# EVALUATION
# --------------------------------------------------------------------------------

@torch.no_grad()
def evaluate_loss(
    model,
    sequences
):

    model.eval()

    total_loss = 0.0
    count = 0

    for seq in sequences:

        x = torch.tensor(
            [seq[:-1]],
            dtype=torch.long,
            device=DEVICE
        )

        y = torch.tensor(
            [seq[1:]],
            dtype=torch.long,
            device=DEVICE
        )

        logits = model(x).logits

        loss = F.cross_entropy(
            logits.reshape(-1, logits.size(-1)),
            y.reshape(-1)
        )

        total_loss += loss.item()
        count += 1

        del x, y, logits, loss

    model.train()

    return total_loss / max(1, count)

# --------------------------------------------------------------------------------
# GENERATION
# --------------------------------------------------------------------------------

@torch.no_grad()
def generate_ids(
    model,
    prompt_ids,
    max_new_tokens=160,
    temperature=0.9
):

    model.eval()

    ids = list(prompt_ids)

    for _ in range(max_new_tokens):

        context = ids[-SEQ_LEN:]

        x = torch.tensor(
            [context],
            dtype=torch.long,
            device=DEVICE
        )

        logits = model(x).logits[:, -1, :]

        logits = logits / temperature

        probs = torch.softmax(
            logits,
            dim=-1
        )

        next_id = torch.multinomial(
            probs,
            num_samples=1
        ).item()

        ids.append(next_id)

        del x, logits, probs

    model.train()

    return ids

# --------------------------------------------------------------------------------
# REPETITION METRICS
# --------------------------------------------------------------------------------

def repetition_metrics(ids):

    if len(ids) < 2:

        return {
            "unique": 0.0,
            "adj": 0.0,
            "d2": 0.0,
            "d3": 0.0,
            "5gram": 0.0,
            "maxrun": 1
        }

    n = len(ids)

    unique = len(set(ids)) / n

    adj = sum(
        ids[i] == ids[i - 1]
        for i in range(1, n)
    ) / max(1, n - 1)

    bigrams = zip(
        ids[:-1],
        ids[1:]
    )

    trigrams = zip(
        ids[:-2],
        ids[1:-1],
        ids[2:]
    )

    fivegrams = zip(
        ids[:-4],
        ids[1:-3],
        ids[2:-2],
        ids[3:-1],
        ids[4:]
    )

    bigrams = list(bigrams)
    trigrams = list(trigrams)
    fivegrams = list(fivegrams)

    d2 = (
        len(set(bigrams))
        / max(1, len(bigrams))
    )

    d3 = (
        len(set(trigrams))
        / max(1, len(trigrams))
    )

    fivegram_rep = (
        1.0 -
        len(set(fivegrams))
        / len(fivegrams)
    ) if fivegrams else 0.0

    maxrun = 1
    current = 1

    for i in range(1, n):

        if ids[i] == ids[i - 1]:

            current += 1
            maxrun = max(
                maxrun,
                current
            )

        else:

            current = 1

    return {
        "unique": unique,
        "adj": adj,
        "d2": d2,
        "d3": d3,
        "5gram": fivegram_rep,
        "maxrun": maxrun
    }

# --------------------------------------------------------------------------------
# GENERATION DIAGNOSTIC
# --------------------------------------------------------------------------------

def generation_diagnostic(
    model,
    temperature
):

    prompt_ids = validation_sequences[0][:32]

    generated = generate_ids(
        model,
        prompt_ids,
        max_new_tokens=160,
        temperature=temperature
    )

    generated_only = generated[
        len(prompt_ids):
    ]

    return repetition_metrics(
        generated_only
    )

# --------------------------------------------------------------------------------
# BASELINE
# --------------------------------------------------------------------------------

print()
print("=" * 100)
print("BASELINE — EXACT STEP-1000 MODEL")
print("=" * 100)

baseline_loss = evaluate_loss(
    model,
    validation_sequences
)

baseline_ppl = math.exp(
    min(baseline_loss, 20)
)

print(
    f"Validation loss : {baseline_loss:.4f}"
)

print(
    f"Perplexity      : {baseline_ppl:.2f}"
)

baseline_metrics = {}

for temperature in TEMPERATURES:

    metrics = generation_diagnostic(
        model,
        temperature
    )

    baseline_metrics[
        str(temperature)
    ] = metrics

    print(
        f"T={temperature:.1f} | "
        f"unique={metrics['unique']:.4f} | "
        f"adj={metrics['adj']:.4f} | "
        f"d2={metrics['d2']:.4f} | "
        f"d3={metrics['d3']:.4f} | "
        f"5gram={metrics['5gram']:.4f} | "
        f"maxrun={metrics['maxrun']}"
    )

# --------------------------------------------------------------------------------
# OPTIMIZER
# --------------------------------------------------------------------------------

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    betas=(0.9, 0.95),
    weight_decay=0.1
)

# --------------------------------------------------------------------------------
# TRAINING
# --------------------------------------------------------------------------------

print()
print("=" * 100)
print("250-STEP LR=2.5e-6 + GRADACCUM=8 STREAMING CONTINUATION")
print("=" * 100)

start_time = time.time()

diagnostics = []

optimizer.zero_grad(
    set_to_none=True
)

for step in range(
    1,
    STEPS + 1
):

    accumulated_loss = 0.0

    for micro in range(
        GRAD_ACCUM
    ):

        idx = (
            (step - 1)
            * GRAD_ACCUM
            + micro
        )

        seq = token_sequences[idx]

        x = torch.tensor(
            [seq[:-1]],
            dtype=torch.long,
            device=DEVICE
        )

        y = torch.tensor(
            [seq[1:]],
            dtype=torch.long,
            device=DEVICE
        )

        outputs = model(x)

        logits = outputs.logits

        loss = F.cross_entropy(
            logits.reshape(
                -1,
                logits.size(-1)
            ),
            y.reshape(-1)
        )

        scaled_loss = (
            loss / GRAD_ACCUM
        )

        scaled_loss.backward()

        accumulated_loss += loss.item()

        # Release this microbatch immediately.
        del x
        del y
        del outputs
        del logits
        del scaled_loss
        del loss

    grad_norm = (
        torch.nn.utils
        .clip_grad_norm_(
            model.parameters(),
            MAX_GRAD_NORM
        )
    )

    optimizer.step()

    optimizer.zero_grad(
        set_to_none=True
    )

    mean_loss = (
        accumulated_loss
        / GRAD_ACCUM
    )

    if (
        step == 1
        or step % 25 == 0
    ):

        print(
            f"step={step:4d} | "
            f"loss={mean_loss:.4f} | "
            f"grad={float(grad_norm):.4f}"
        )

    if step in EVAL_INTERVALS:

        print()
        print("-" * 90)
        print(
            f"DIAGNOSTIC CHECKPOINT — +{step} STEPS"
        )
        print("-" * 90)

        val_loss = evaluate_loss(
            model,
            validation_sequences
        )

        ppl = math.exp(
            min(val_loss, 20)
        )

        print(
            f"Validation loss : {val_loss:.4f}"
        )

        print(
            f"Perplexity      : {ppl:.2f}"
        )

        generation_results = {}

        for temperature in TEMPERATURES:

            metrics = generation_diagnostic(
                model,
                temperature
            )

            generation_results[
                str(temperature)
            ] = metrics

            print(
                f"T={temperature:.1f} | "
                f"unique={metrics['unique']:.4f} | "
                f"adj={metrics['adj']:.4f} | "
                f"d2={metrics['d2']:.4f} | "
                f"d3={metrics['d3']:.4f} | "
                f"5gram={metrics['5gram']:.4f} | "
                f"maxrun={metrics['maxrun']}"
            )

        diagnostics.append({
            "step": step,
            "validation_loss": val_loss,
            "perplexity": ppl,
            "generation": generation_results
        })

# --------------------------------------------------------------------------------
# SUMMARY
# --------------------------------------------------------------------------------

elapsed = (
    time.time() - start_time
) / 60

final_loss = diagnostics[-1][
    "validation_loss"
]

loss_change = (
    final_loss
    - baseline_loss
)

print()
print("=" * 100)
print("CS781 — LR=2.5e-6 STREAMING CONTROLLED SUMMARY")
print("=" * 100)

print(
    f"Elapsed time : {elapsed:.2f} minutes"
)

print()

print(
    f"{'POINT':<15}"
    f"{'LOSS':>10}"
    f"{'PPL':>12}"
)

print("-" * 40)

print(
    f"{'step1000':<15}"
    f"{baseline_loss:>10.4f}"
    f"{baseline_ppl:>12.2f}"
)

for d in diagnostics:

    print(
        f"{('step+' + str(d['step'])):<15}"
        f"{d['validation_loss']:>10.4f}"
        f"{d['perplexity']:>12.2f}"
    )

print()
print(
    f"Baseline loss : {baseline_loss:.4f}"
)

print(
    f"Final loss    : {final_loss:.4f}"
)

print(
    f"Loss change   : {loss_change:+.4f}"
)

# --------------------------------------------------------------------------------
# SAVE RESULTS ONLY
# --------------------------------------------------------------------------------

results = {
    "experiment":
        "step1000_lr2p5e6_gradaccum8_streaming",

    "checkpoint":
        CHECKPOINT_PATH,

    "starting_step":
        1000,

    "steps":
        STEPS,

    "seq_len":
        SEQ_LEN,

    "micro_batch":
        MICRO_BATCH,

    "grad_accum":
        GRAD_ACCUM,

    "effective_batch":
        MICRO_BATCH * GRAD_ACCUM,

    "learning_rate":
        LR,

    "required_sequences":
        REQUIRED_SEQUENCES,

    "actual_sequences_used":
        len(token_sequences),

    "baseline_loss":
        baseline_loss,

    "baseline_perplexity":
        baseline_ppl,

    "baseline_generation":
        baseline_metrics,

    "diagnostics":
        diagnostics,

    "final_loss":
        final_loss,

    "final_perplexity":
        math.exp(min(final_loss, 20)),

    "loss_change":
        loss_change,

    "elapsed_minutes":
        elapsed,

    "production_checkpoint_written":
        False,

    "original_checkpoint_modified":
        False
}

RESULT_PATH = os.path.join(
    OUTPUT_DIR,
    "lr2p5e6_gradaccum8_streaming_results.json"
)

with open(
    RESULT_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        results,
        f,
        indent=2
    )

print()
print("=" * 100)
print("SAFETY VERIFICATION")
print("=" * 100)

print("✓ Original checkpoint was never opened for writing")
print("✓ No production checkpoint written")
print("✓ Only 2,000 training sequences were collected")
print("✓ Training batches were created on GPU one microbatch at a time")
print(f"✓ Results: {RESULT_PATH}")

print()
print("=" * 100)
print("✓ STREAMING CONTROLLED EXPERIMENT COMPLETE")
print("=" * 100)

In [ ]:
# ================================================================================================
# CS781 — GENERATION QUALITY TEST
# CELL 1: RESTORE EXACT STEP-1000 BASELINE
#
# IMPORTANT:
#   - Restores the exact Step-1000 checkpoint
#   - Reuses the existing tokenizer
#   - Does NOT train
#   - Does NOT modify/write the checkpoint
#   - Ensures this evaluation starts from the known baseline
# ================================================================================================

import os
import gc
import torch

print("=" * 100)
print("CS781 — RESTORING EXACT STEP-1000 BASELINE FOR GENERATION TEST")
print("=" * 100)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CHECKPOINT_PATH = (
    "/kaggle/input/datasets/oipranesh/"
    "phase4-step1000-verified-model/checkpoint_step_001000.pt"
)

TOKENIZER_PATH = (
    "/kaggle/input/datasets/oipranesh/"
    "cs781-phase4-training-prep/tokenizer/legal_bpe_32k.json"
)

print(f"Device     : {DEVICE}")
print(f"Checkpoint : {CHECKPOINT_PATH}")
print()

# --------------------------------------------------------------------------------
# CLEAN TEMPORARY OBJECTS ONLY
# --------------------------------------------------------------------------------

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

# --------------------------------------------------------------------------------
# VERIFY TOKENIZER
# --------------------------------------------------------------------------------

if "tokenizer" not in globals() or tokenizer is None:
    from tokenizers import Tokenizer
    tokenizer = Tokenizer.from_file(TOKENIZER_PATH)

print("Tokenizer :", type(tokenizer).__name__)
print("Vocab     :", len(tokenizer.get_vocab()))

# --------------------------------------------------------------------------------
# VERIFY / RECOVER MODEL OBJECT
# --------------------------------------------------------------------------------

if "model" not in globals() or model is None:

    if "diag_model" in globals() and diag_model is not None:
        model = diag_model
        print("✓ Reusing existing model architecture")

    else:
        raise RuntimeError(
            "No usable model object exists. "
            "Run the previously successful SAFE MODEL OBJECT RECOVERY cell first."
        )

# --------------------------------------------------------------------------------
# LOAD CHECKPOINT
# --------------------------------------------------------------------------------

if not os.path.exists(CHECKPOINT_PATH):
    raise FileNotFoundError(CHECKPOINT_PATH)

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False
)

print()
print("Checkpoint step :", checkpoint.get("step"))
print("Checkpoint LR   :", checkpoint.get("learning_rate"))
print("Context length  :", checkpoint.get("context_length"))

if checkpoint.get("step") != 1000:
    raise RuntimeError(
        f"Expected Step-1000 checkpoint, found step={checkpoint.get('step')}"
    )

# Load exact weights
missing, unexpected = model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=False
)

print("Missing keys    :", len(missing))
print("Unexpected keys :", len(unexpected))

if missing or unexpected:
    raise RuntimeError(
        f"Checkpoint mismatch. Missing={missing}, Unexpected={unexpected}"
    )

# Move to GPU
model = model.to(DEVICE)
model.eval()

# Release checkpoint from RAM
del checkpoint
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

# --------------------------------------------------------------------------------
# FORWARD TEST
# --------------------------------------------------------------------------------

test_ids = tokenizer.encode(
    "The court considered the evidence"
).ids

x = torch.tensor(
    [test_ids],
    dtype=torch.long,
    device=DEVICE
)

with torch.no_grad():
    logits = model(x).logits

print()
print("Forward pass :", tuple(logits.shape))

# --------------------------------------------------------------------------------
# FINAL STATE
# --------------------------------------------------------------------------------

print()
print("=" * 100)
print("✓ EXACT STEP-1000 BASELINE READY")
print("=" * 100)
print("Model       : GPT2LMHeadModel")
print("Parameters  : 112,777,728")
print("Tokenizer   : legal BPE 32k")
print("Training    : NONE")
print("Checkpoint  : NOT MODIFIED")
print("=" * 100)

In [ ]:
# ================================================================================================
# CS781 — GENERATION QUALITY TEST
# CELL 2: ACTUAL GENERATION EVALUATION
#
# Tests:
#   1. Greedy decoding
#   2. T=0.7
#   3. T=0.9
#   4. T=1.0
#
# Evaluates multiple legal prompts and saves the complete outputs.
# ================================================================================================

import os
import json
import math
import time
import torch

print("=" * 100)
print("CS781 — ACTUAL GENERATION QUALITY EVALUATION")
print("=" * 100)

if "model" not in globals() or model is None:
    raise RuntimeError("Model is missing. Run Cell 1 first.")

if "tokenizer" not in globals() or tokenizer is None:
    raise RuntimeError("Tokenizer is missing. Run Cell 1 first.")

model.eval()

OUTPUT_DIR = "/kaggle/working/cs781_generation_quality_step1000"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --------------------------------------------------------------------------------
# PROMPTS
# --------------------------------------------------------------------------------

PROMPTS = [
    "The court held that the defendant was entitled to",
    
    "The parties entered into an agreement under which",
    
    "The evidence presented before the court established that",
    
    "The plaintiff argued that the defendant had failed to",
    
    "The applicable legal provision provides that",
    
    "In determining whether the claim was maintainable, the court considered",
]

DECODING_CONFIGS = [
    {
        "name": "greedy",
        "temperature": 1.0,
        "do_sample": False,
    },
    {
        "name": "temperature_0.7",
        "temperature": 0.7,
        "do_sample": True,
    },
    {
        "name": "temperature_0.9",
        "temperature": 0.9,
        "do_sample": True,
    },
    {
        "name": "temperature_1.0",
        "temperature": 1.0,
        "do_sample": True,
    },
]

MAX_NEW_TOKENS = 120
EOS_ID = 3

# --------------------------------------------------------------------------------
# GENERATION
# --------------------------------------------------------------------------------

@torch.no_grad()
def generate_text(prompt, temperature=1.0, do_sample=True):

    prompt_ids = tokenizer.encode(prompt).ids

    generated_ids = list(prompt_ids)

    for _ in range(MAX_NEW_TOKENS):

        # Respect model context length
        context_ids = generated_ids[-1024:]

        x = torch.tensor(
            [context_ids],
            dtype=torch.long,
            device=DEVICE
        )

        logits = model(x).logits[:, -1, :]

        if not do_sample:
            next_id = torch.argmax(logits, dim=-1).item()

        else:
            logits = logits / temperature

            probs = torch.softmax(logits, dim=-1)

            next_id = torch.multinomial(
                probs,
                num_samples=1
            ).item()

        generated_ids.append(next_id)

        if next_id == EOS_ID:
            break

    # Decode only generated continuation
    continuation_ids = generated_ids[len(prompt_ids):]

    continuation = tokenizer.decode(
        continuation_ids,
        skip_special_tokens=True
    )

    full_text = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    )

    return {
        "prompt": prompt,
        "continuation": continuation,
        "full_text": full_text,
        "token_count": len(continuation_ids),
        "generated_ids": continuation_ids,
    }


# --------------------------------------------------------------------------------
# REPETITION / BASIC QUALITY METRICS
# --------------------------------------------------------------------------------

def generation_metrics(ids):

    if not ids:
        return {
            "unique": 0.0,
            "adjacent_repetition": 0.0,
            "distinct_2": 0.0,
            "distinct_3": 0.0,
            "repeated_5gram_fraction": 0.0,
            "max_run": 0,
        }

    n = len(ids)

    unique = len(set(ids)) / n

    adjacent = (
        sum(ids[i] == ids[i - 1] for i in range(1, n))
        / max(1, n - 1)
    )

    bigrams = list(zip(ids[:-1], ids[1:]))

    trigrams = list(
        zip(
            ids[:-2],
            ids[1:-1],
            ids[2:]
        )
    )

    fivegrams = list(
        zip(
            ids[:-4],
            ids[1:-3],
            ids[2:-2],
            ids[3:-1],
            ids[4:]
        )
    )

    distinct_2 = (
        len(set(bigrams)) / len(bigrams)
        if bigrams else 0.0
    )

    distinct_3 = (
        len(set(trigrams)) / len(trigrams)
        if trigrams else 0.0
    )

    repeated_5gram_fraction = (
        1.0 - len(set(fivegrams)) / len(fivegrams)
        if fivegrams else 0.0
    )

    max_run = 1
    current_run = 1

    for i in range(1, n):

        if ids[i] == ids[i - 1]:
            current_run += 1
            max_run = max(max_run, current_run)

        else:
            current_run = 1

    return {
        "unique": unique,
        "adjacent_repetition": adjacent,
        "distinct_2": distinct_2,
        "distinct_3": distinct_3,
        "repeated_5gram_fraction": repeated_5gram_fraction,
        "max_run": max_run,
    }


# --------------------------------------------------------------------------------
# RUN TEST
# --------------------------------------------------------------------------------

results = []

start_time = time.time()

for config in DECODING_CONFIGS:

    print()
    print()
    print("=" * 100)
    print(f"DECODING MODE: {config['name']}")
    print("=" * 100)

    for prompt_index, prompt in enumerate(PROMPTS, start=1):

        result = generate_text(
            prompt=prompt,
            temperature=config["temperature"],
            do_sample=config["do_sample"],
        )

        metrics = generation_metrics(
            result["generated_ids"]
        )

        record = {
            "decoding": config["name"],
            "prompt_index": prompt_index,
            "prompt": prompt,
            "continuation": result["continuation"],
            "full_text": result["full_text"],
            "token_count": result["token_count"],
            "metrics": metrics,
        }

        results.append(record)

        print()
        print("-" * 100)
        print(f"[{config['name']}] PROMPT {prompt_index}")
        print("-" * 100)
        print("PROMPT:")
        print(prompt)
        print()
        print("GENERATED:")
        print(result["continuation"])
        print()
        print(
            "METRICS | "
            f"unique={metrics['unique']:.4f} | "
            f"adj={metrics['adjacent_repetition']:.4f} | "
            f"d2={metrics['distinct_2']:.4f} | "
            f"d3={metrics['distinct_3']:.4f} | "
            f"5gram_rep={metrics['repeated_5gram_fraction']:.4f} | "
            f"maxrun={metrics['max_run']}"
        )

elapsed = (time.time() - start_time) / 60

# --------------------------------------------------------------------------------
# SUMMARY
# --------------------------------------------------------------------------------

print()
print()
print("=" * 100)
print("CS781 — GENERATION QUALITY TEST COMPLETE")
print("=" * 100)

print(f"Prompts tested       : {len(PROMPTS)}")
print(f"Decoding modes       : {len(DECODING_CONFIGS)}")
print(f"Total generations    : {len(results)}")
print(f"Max new tokens       : {MAX_NEW_TOKENS}")
print(f"Elapsed time         : {elapsed:.2f} minutes")

# --------------------------------------------------------------------------------
# SAVE RESULTS
# --------------------------------------------------------------------------------

RESULT_PATH = os.path.join(
    OUTPUT_DIR,
    "step1000_generation_quality.json"
)

save_results = {
    "experiment": "step1000_generation_quality",
    "checkpoint": (
        "/kaggle/input/datasets/oipranesh/"
        "phase4-step1000-verified-model/"
        "checkpoint_step_001000.pt"
    ),
    "starting_step": 1000,
    "max_new_tokens": MAX_NEW_TOKENS,
    "prompts": PROMPTS,
    "decoding_configs": DECODING_CONFIGS,
    "results": results,
    "elapsed_minutes": elapsed,
}

with open(
    RESULT_PATH,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        save_results,
        f,
        indent=2,
        ensure_ascii=False
    )

print()
print(f"Results saved:")
print(RESULT_PATH)

print()
print("=" * 100)
print("✓ NO TRAINING PERFORMED")
print("✓ NO CHECKPOINT MODIFIED")
print("✓ ACTUAL GENERATIONS SAVED")
print("=" * 100)

In [ ]:
# ================================================================================================
# CS781 — CLEAN RUNTIME MEMORY
# Safe cleanup only: does NOT delete model/tokenizer/checkpoint variables
# ================================================================================================

import gc
import torch

print("=" * 100)
print("CS781 — RUNTIME MEMORY CLEANUP")
print("=" * 100)

# Clean CUDA cache
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

# Python garbage collection
collected = gc.collect()

print(f"Python garbage collected: {collected}")

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        allocated = torch.cuda.memory_allocated(i) / (1024**3)
        reserved = torch.cuda.memory_reserved(i) / (1024**3)

        print(
            f"GPU {i}: "
            f"allocated={allocated:.2f} GB | "
            f"reserved={reserved:.2f} GB"
        )

print()
print("✓ Runtime cleanup complete")
print("✓ Existing model/tokenizer variables were NOT deleted")
print("=" * 100)

In [ ]:
# ================================================================================================
# CS781 — FRESH EXPERIMENT START-POINT VERIFICATION
# ================================================================================================

import os
import torch

CHECKPOINT_PATH = (
    "/kaggle/input/datasets/oipranesh/"
    "phase4-step1000-verified-model/checkpoint_step_001000.pt"
)

print("=" * 100)
print("CS781 — START-POINT VERIFICATION")
print("=" * 100)

if "tokenizer" not in globals() or tokenizer is None:
    raise RuntimeError(
        "Tokenizer missing. Run the previously successful tokenizer/model recovery cell."
    )

if "model" not in globals() or model is None:
    raise RuntimeError(
        "Model missing. Run the previously successful SAFE MODEL OBJECT RECOVERY cell."
    )

if not os.path.exists(CHECKPOINT_PATH):
    raise FileNotFoundError(CHECKPOINT_PATH)

print("✓ tokenizer:", type(tokenizer).__name__)
print("✓ model    :", type(model).__name__)

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False
)

if checkpoint.get("step") != 1000:
    raise RuntimeError(
        f"Expected Step-1000 checkpoint, got step={checkpoint.get('step')}"
    )

state_dict = checkpoint["model_state_dict"]

missing, unexpected = model.load_state_dict(
    state_dict,
    strict=False
)

print("Checkpoint step :", checkpoint["step"])
print("Missing keys    :", len(missing))
print("Unexpected keys :", len(unexpected))

if missing or unexpected:
    raise RuntimeError(
        f"Checkpoint mismatch. Missing={missing}, Unexpected={unexpected}"
    )

model.eval()

print()
print("✓ Exact Step-1000 weights reloaded")
print("✓ Starting point is clean")
print("✓ No optimizer created")
print("✓ No checkpoint modified")
print()
print("=" * 100)
print("READY FOR NEXT EXPERIMENT")
print("=" * 100)

In [ ]:
# ================================================================================================
# CS781 — FAST TRAINING DATA SANITY CHECK
# IMPORTANT:
#   - Does NOT scan the whole corpus
#   - Does NOT load the corpus into RAM
#   - Stops as soon as enough sequences are found
#   - Reuses the exact tokenizer
# ================================================================================================

import os
import json
import gc

DATA_ROOT = (
    "/kaggle/input/datasets/oipranesh/"
    "cs781-phase4-training-prep"
)

SEQ_LEN = 1024
REQUIRED_SEQUENCES = 2000

print("=" * 100)
print("CS781 — FAST TRAINING DATA SANITY CHECK")
print("=" * 100)

if "tokenizer" not in globals() or tokenizer is None:
    raise RuntimeError("Existing tokenizer is missing.")

# --------------------------------------------------------------------------------
# FIND JSONL FILES ONLY
# --------------------------------------------------------------------------------

jsonl_files = []

for root, dirs, files in os.walk(DATA_ROOT):
    for fname in files:
        if fname.endswith(".jsonl"):
            jsonl_files.append(os.path.join(root, fname))

jsonl_files.sort()

print(f"JSONL files found: {len(jsonl_files)}")

if not jsonl_files:
    raise RuntimeError("No JSONL files found.")

# --------------------------------------------------------------------------------
# STREAM UNTIL ENOUGH SEQUENCES ARE FOUND
# --------------------------------------------------------------------------------

sequence_count = 0
lines_read = 0
files_used = 0

for file_index, path in enumerate(jsonl_files, 1):

    file_sequences = 0

    with open(path, "r", encoding="utf-8") as f:

        for line in f:

            lines_read += 1
            line = line.strip()

            if not line:
                continue

            try:
                obj = json.loads(line)
            except Exception:
                continue

            text = None

            if isinstance(obj, str):
                text = obj

            elif isinstance(obj, dict):
                for key in (
                    "text",
                    "content",
                    "document",
                    "body",
                    "input",
                    "sequence",
                ):
                    value = obj.get(key)

                    if isinstance(value, str) and value.strip():
                        text = value
                        break

            if not text:
                continue

            try:
                ids = tokenizer.encode(text).ids
            except Exception:
                continue

            if len(ids) < SEQ_LEN + 1:
                continue

            # Number of non-overlapping 1024-token sequences.
            chunks = (len(ids) - 1) // SEQ_LEN

            usable = min(
                chunks,
                REQUIRED_SEQUENCES - sequence_count
            )

            sequence_count += usable
            file_sequences += usable

            if sequence_count >= REQUIRED_SEQUENCES:
                break

    files_used = file_index

    print(
        f"File {file_index}/{len(jsonl_files)} | "
        f"sequences={sequence_count} | "
        f"lines read={lines_read}"
    )

    if sequence_count >= REQUIRED_SEQUENCES:
        break

# --------------------------------------------------------------------------------
# RESULT
# --------------------------------------------------------------------------------

print()
print("=" * 100)
print("FAST SANITY RESULT")
print("=" * 100)

print(f"Files used          : {files_used}")
print(f"Lines read          : {lines_read}")
print(f"Usable sequences    : {sequence_count}")
print(f"Required sequences  : {REQUIRED_SEQUENCES}")

if sequence_count < REQUIRED_SEQUENCES:
    raise RuntimeError(
        f"Only {sequence_count} usable sequences found before corpus ended. "
        f"Need {REQUIRED_SEQUENCES}."
    )

print()
print("✓ Enough training data confirmed")
print("✓ Streaming approach confirmed")
print("✓ Corpus scan stopped immediately")
print("✓ No full-corpus RAM accumulation")
print()
print("=" * 100)
print("READY FOR NEXT CHECK")
print("=" * 100)

gc.collect()

In [ ]:
# ================================================================================================
# CS781 — FAST PRE-TRAINING BASELINE
#
# PURPOSE:
#   Evaluate the exact Step-1000 model before the next training experiment.
#
# SAFETY:
#   - Exact Step-1000 weights reloaded
#   - Only enough validation data is streamed
#   - No full-corpus scan
#   - No optimizer
#   - No training
#   - No checkpoint writing
# ================================================================================================

import os
import json
import math
import torch
import torch.nn.functional as F

print("=" * 100)
print("CS781 — FAST PRE-TRAINING BASELINE")
print("=" * 100)

# --------------------------------------------------------------------------------
# CONFIG
# --------------------------------------------------------------------------------

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CHECKPOINT_PATH = (
    "/kaggle/input/datasets/oipranesh/"
    "phase4-step1000-verified-model/checkpoint_step_001000.pt"
)

DATA_ROOT = (
    "/kaggle/input/datasets/oipranesh/"
    "cs781-phase4-training-prep"
)

SEQ_LEN = 1024
VAL_N = 100

print(f"Device     : {DEVICE}")
print(f"Checkpoint : {CHECKPOINT_PATH}")
print(f"SeqLen     : {SEQ_LEN}")
print(f"Validation : {VAL_N}")
print()

# --------------------------------------------------------------------------------
# VERIFY EXISTING WORKING STATE
# --------------------------------------------------------------------------------

if "tokenizer" not in globals() or tokenizer is None:
    raise RuntimeError(
        "Tokenizer missing. Run the successful tokenizer/model recovery cell first."
    )

if "model" not in globals() or model is None:
    raise RuntimeError(
        "Model missing. Run the successful SAFE MODEL OBJECT RECOVERY cell first."
    )

print("=" * 100)
print("VERIFYING MODEL / TOKENIZER")
print("=" * 100)

print("✓ tokenizer:", type(tokenizer).__name__)
print("✓ model    :", type(model).__name__)

# --------------------------------------------------------------------------------
# RELOAD EXACT STEP-1000 WEIGHTS
# --------------------------------------------------------------------------------

print()
print("=" * 100)
print("LOADING EXACT STEP-1000 WEIGHTS")
print("=" * 100)

if not os.path.exists(CHECKPOINT_PATH):
    raise FileNotFoundError(CHECKPOINT_PATH)

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False
)

if checkpoint.get("step") != 1000:
    raise RuntimeError(
        f"Expected Step-1000 checkpoint, found step={checkpoint.get('step')}"
    )

missing, unexpected = model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=False
)

print("Checkpoint step :", checkpoint.get("step"))
print("Missing keys    :", len(missing))
print("Unexpected keys :", len(unexpected))

if missing or unexpected:
    raise RuntimeError(
        f"Checkpoint mismatch. Missing={missing}, Unexpected={unexpected}"
    )

model = model.to(DEVICE)
model.eval()

print("✓ Exact Step-1000 weights loaded")
print("✓ Model in evaluation mode")
print("✓ No training performed")
print("✓ Checkpoint remains untouched")

# Release checkpoint CPU object before evaluation.
del checkpoint
torch.cuda.empty_cache()

# --------------------------------------------------------------------------------
# FIND JSONL FILES
# --------------------------------------------------------------------------------

print()
print("=" * 100)
print("STREAMING VALIDATION DATA")
print("=" * 100)

jsonl_files = []

for root, dirs, files in os.walk(DATA_ROOT):
    for fname in files:
        if fname.endswith(".jsonl"):
            jsonl_files.append(
                os.path.join(root, fname)
            )

jsonl_files.sort()

print(f"JSONL files found: {len(jsonl_files)}")

# --------------------------------------------------------------------------------
# COLLECT ONLY 100 VALIDATION SEQUENCES
# --------------------------------------------------------------------------------

validation_sequences = []

lines_read = 0
files_used = 0

for file_index, path in enumerate(jsonl_files, 1):

    with open(path, "r", encoding="utf-8") as f:

        for line in f:

            if len(validation_sequences) >= VAL_N:
                break

            lines_read += 1
            line = line.strip()

            if not line:
                continue

            try:
                obj = json.loads(line)
            except Exception:
                continue

            text = None

            if isinstance(obj, str):
                text = obj

            elif isinstance(obj, dict):

                for key in (
                    "text",
                    "content",
                    "document",
                    "body",
                    "input",
                    "sequence",
                ):

                    value = obj.get(key)

                    if isinstance(value, str) and value.strip():
                        text = value
                        break

            if not text:
                continue

            try:
                ids = tokenizer.encode(text).ids
            except Exception:
                continue

            if len(ids) < SEQ_LEN + 1:
                continue

            # Take one deterministic 1025-token sequence.
            validation_sequences.append(
                ids[:SEQ_LEN + 1]
            )

        files_used = file_index

    if len(validation_sequences) >= VAL_N:
        break

print()
print(f"Files used          : {files_used}")
print(f"Lines read          : {lines_read}")
print(f"Validation sequences: {len(validation_sequences)}")

if len(validation_sequences) < VAL_N:
    raise RuntimeError(
        f"Only {len(validation_sequences)} validation sequences found; "
        f"need {VAL_N}."
    )

# --------------------------------------------------------------------------------
# EVALUATION
# --------------------------------------------------------------------------------

@torch.no_grad()
def evaluate_loss(model, sequences):

    model.eval()

    total_loss = 0.0

    for seq in sequences:

        x = torch.tensor(
            [seq[:-1]],
            dtype=torch.long,
            device=DEVICE
        )

        y = torch.tensor(
            [seq[1:]],
            dtype=torch.long,
            device=DEVICE
        )

        logits = model(x).logits

        loss = F.cross_entropy(
            logits.reshape(-1, logits.size(-1)),
            y.reshape(-1)
        )

        total_loss += loss.item()

        del x, y, logits

    return total_loss / len(sequences)


# --------------------------------------------------------------------------------
# BASELINE
# --------------------------------------------------------------------------------

print()
print("=" * 100)
print("BASELINE — EXACT STEP-1000 MODEL")
print("=" * 100)

baseline_loss = evaluate_loss(
    model,
    validation_sequences
)

baseline_ppl = math.exp(
    min(baseline_loss, 20)
)

print(f"Validation loss : {baseline_loss:.4f}")
print(f"Perplexity      : {baseline_ppl:.2f}")

# --------------------------------------------------------------------------------
# SIMPLE GENERATION SANITY
# --------------------------------------------------------------------------------

@torch.no_grad()
def generate_sample(model, prompt_ids, max_new_tokens=80, temperature=0.9):

    model.eval()

    ids = list(prompt_ids)

    for _ in range(max_new_tokens):

        context = ids[-SEQ_LEN:]

        x = torch.tensor(
            [context],
            dtype=torch.long,
            device=DEVICE
        )

        logits = model(x).logits[:, -1, :]
        logits = logits / temperature

        probs = torch.softmax(logits, dim=-1)

        next_id = torch.multinomial(
            probs,
            num_samples=1
        ).item()

        ids.append(next_id)

        del x, logits, probs

    return ids


def repetition_metrics(ids):

    if len(ids) < 2:
        return {
            "unique": 0.0,
            "adj": 0.0,
            "d2": 0.0,
            "d3": 0.0,
            "maxrun": 1
        }

    n = len(ids)

    unique = len(set(ids)) / n

    adj = sum(
        ids[i] == ids[i - 1]
        for i in range(1, n)
    ) / max(1, n - 1)

    bigrams = list(zip(ids[:-1], ids[1:]))

    trigrams = list(
        zip(ids[:-2], ids[1:-1], ids[2:])
    )

    d2 = len(set(bigrams)) / max(1, len(bigrams))
    d3 = len(set(trigrams)) / max(1, len(trigrams))

    maxrun = 1
    current = 1

    for i in range(1, n):

        if ids[i] == ids[i - 1]:
            current += 1
            maxrun = max(maxrun, current)
        else:
            current = 1

    return {
        "unique": unique,
        "adj": adj,
        "d2": d2,
        "d3": d3,
        "maxrun": maxrun
    }


print()
print("=" * 100)
print("GENERATION SANITY")
print("=" * 100)

prompt = validation_sequences[0][:32]

for temperature in [0.9, 1.0]:

    generated = generate_sample(
        model,
        prompt,
        max_new_tokens=80,
        temperature=temperature
    )

    generated_only = generated[len(prompt):]

    metrics = repetition_metrics(generated_only)

    print(
        f"T={temperature:.1f} | "
        f"unique={metrics['unique']:.4f} | "
        f"adj={metrics['adj']:.4f} | "
        f"d2={metrics['d2']:.4f} | "
        f"d3={metrics['d3']:.4f} | "
        f"maxrun={metrics['maxrun']}"
    )

# --------------------------------------------------------------------------------
# FINAL
# --------------------------------------------------------------------------------

print()
print("=" * 100)
print("CS781 — PRE-TRAINING BASELINE COMPLETE")
print("=" * 100)

print(f"Step-1000 loss : {baseline_loss:.4f}")
print(f"Step-1000 PPL  : {baseline_ppl:.2f}")

print()
print("✓ Exact Step-1000 model evaluated")
print("✓ Validation data streamed")
print("✓ No full-corpus scan")
print("✓ No optimizer created")
print("✓ No training performed")
print("✓ No checkpoint modified")
print("=" * 100)